In [ ]:
from pathlib import Path as _OutputPath

def _prefix_output(path):
    if path is None:
        return None
    p = _OutputPath(path)
    stem = p.stem.removesuffix("_evoked")
    name = stem + p.suffix
    if name.startswith("evoked_"):
        return path
    result = p.with_name("evoked_" + name)
    return result if isinstance(path, _OutputPath) else str(result)


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from collections import defaultdict
from scipy.stats import sem
from scipy.signal import welch
import yaml
from fooof import FOOOF
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
from datetime import datetime
import imageio
import numpy as np
import pandas as pd
from cerebellum.analysis.structure_analysis import StructureReport_v2
import cerebellum
from bsb import from_storage

In [ ]:
Path("./tempPlots").mkdir(parents=True, exist_ok=True)

# kase

In [ ]:
import pandas as pd
from pathlib import Path

base_paths = [
    '20251113214858_kase_stim_vitro_15_0_glutamate_all',
]

resilience_map = {
    "2_0": "Without Resilience",
    "15_0": "With Resilience",

}

all_dfs = []

for base_path in base_paths:
    resilience_label = next((label for key, label in resilience_map.items() if key in base_path), "Unknown")
    sigma_folder = Path(base_path) / "B_collective_results" / "2.functions"
    if sigma_folder.exists():
        for sigma_dir in sigma_folder.iterdir():
            if sigma_dir.is_dir() and sigma_dir.name.startswith("sigma_"):
                sigma_value = int(sigma_dir.name.split("_")[1])
                #if sigma_value % 2 == 0:  # carica solo sigma pari
                df_path = sigma_dir / f"{sigma_value}_df.pkl"
                if df_path.exists():
                    df = pd.read_pickle(df_path)
                    df["sigma"] = sigma_value
                    df["sim_path"] = base_path
                    df["resilience"] = resilience_label
                    all_dfs.append(df)

df_all_1 = pd.concat(all_dfs, ignore_index=True)
df_all_1.keys()


In [ ]:
df

In [ ]:
import pandas as pd
from pathlib import Path

base_paths = [
    '20251113161643_kase_stim_vitro_2_0_glutamate_all'
]

resilience_map = {
    "2_0": "Without Resilience",
    "15_0": "With Resilience",

}

all_dfs = []

for base_path in base_paths:
    resilience_label = next((label for key, label in resilience_map.items() if key in base_path), "Unknown")
    sigma_folder = Path(base_path) / "B_collective_results" / "2.functions"
    if sigma_folder.exists():
        for sigma_dir in sigma_folder.iterdir():
            if sigma_dir.is_dir() and sigma_dir.name.startswith("sigma_"):
                sigma_value = int(sigma_dir.name.split("_")[1])
                #if sigma_value % 2 == 0:  # carica solo sigma pari
                df_path = sigma_dir / f"{sigma_value}_df.pkl"
                if df_path.exists():
                    df = pd.read_pickle(df_path)
                    df["sigma"] = sigma_value
                    df["sim_path"] = base_path
                    df["resilience"] = resilience_label
                    all_dfs.append(df)

df_all_2 = pd.concat(all_dfs, ignore_index=True)
df_all_2.keys()


In [ ]:
df_all = pd.concat([df_all_1, df_all_2], ignore_index=True)


In [ ]:
df_all.keys()

In [ ]:
def plot_epoched_firing_rate(df, trace_column, time_axis=np.arange(0, 100)*0.1,
                             pop_name="Population", stim_column=None,
                              onset_column="burst_onsets_ms", group_column=None, 
                              window_ms=(-200, 200), dt=0.1, cmap="viridis",
                              ylims=[0, 250],
                              operator=np.mean, sort_by_group=True,
                              save_path=None, ax=None,
                              outlier_method="iqr", outlier_thresh=1.5):  # <--- nuovi parametri
    import numpy as np
    import matplotlib.pyplot as plt
    import matplotlib.colors as mcolors
    from pathlib import Path
    from mpl_toolkits.axes_grid1 import make_axes_locatable

    win_pre = int(abs(window_ms[0]) / dt)
    win_post = int(window_ms[1] / dt)
    #time_axis = np.arange(-win_pre, win_post) * dt
    #time_axis = np.arange(0, win_post+win_pre) * dt


    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    else:
        fig = ax.get_figure()

    cmap = plt.colormaps.get_cmap(cmap)
    norm = None
    if group_column:
        if sort_by_group:
            df = df.sort_values(group_column).reset_index(drop=True)
        group_vals = df[group_column].values
        norm = mcolors.Normalize(vmin=group_vals.min(), vmax=group_vals.max())

    for _, row in df.iterrows():
        trace = np.array(row[trace_column])
        onsets = row[onset_column]
        durations = row.get("burst_durations_ms", [float("inf")] * len(onsets))  # fallback se non presente
        stim = np.array(row[stim_column]) if stim_column else None
        group_val = row[group_column] if group_column else None
        color = cmap(norm(group_val)) if cmap and norm else "black"

        epochs, stim_epochs = [], []
        for onset, duration in zip(onsets, durations):
            if duration < 20:
                continue  # Salta se durata troppo breve
            center = int(onset / dt)
            start = center - win_pre
            end = center + win_post
            if start >= 0 and end <= len(trace):
                epochs.append(trace[start:end])
                if stim_column:
                    stim_epochs.append(stim[start:end])

        if not epochs:
            continue

        epochs = np.array(epochs)
        if stim_column and stim_epochs:
            stim_epochs = np.array(stim_epochs)

        if len(epochs) == 0:
            continue

        this_operator = operator
        
        if this_operator is None:
            for ep in epochs:
                ax.plot(time_axis, ep, color=color, alpha=0.5, linewidth=2)
                peak_idx = np.nanargmax(ep)
                peak_time = time_axis[peak_idx]
                peak_value = ep[peak_idx]
                ax.scatter(peak_time, peak_value, marker='.', color='k', s=0, zorder=5)
                # Interpola con modello di secondo ordine intorno al picco
                mask = (time_axis >= peak_time - 10) & (time_axis <= peak_time + 10)
                x_fit = time_axis[mask]
                y_fit = ep[mask]
                if len(x_fit) >= 3:
                    coeffs = np.polyfit(x_fit, y_fit, 2)
                    x_smooth = np.linspace(x_fit.min(), x_fit.max(), 100)
                    y_smooth = np.polyval(coeffs, x_smooth)
                    ax.plot(x_smooth, y_smooth, color='red', linestyle='--', linewidth=1, alpha=0)

        else:
            if this_operator == np.quantile:
                mean_trace = this_operator(epochs, axis=0, q=1)
            else:
                mean_trace = this_operator(epochs, axis=0)

            lower = np.percentile(epochs, 25, axis=0)
            upper = np.percentile(epochs, 75, axis=0)
            ax.plot(time_axis, mean_trace, linewidth=2, color=color, alpha=1)
            ax.fill_between(time_axis, lower, upper, color=color, alpha=0.5)

            
        if stim_column and stim_epochs is not None and len(stim_epochs) > 0:
            stim_mean = np.max(stim_epochs, axis=0)
            ax.plot(time_axis, stim_mean, linestyle='--', color="k", alpha=1, linewidth=1.5)


    k = 20
    ax.axvline(0-k, color='gray', linestyle='-', label="Saccade Burst Onset")
    ax.legend(fontsize=14, frameon=False)
    ax.set_title(f"{pop_name}")
    ax.set_xlabel("Time from Saccade Onset (ms)")
    ax.set_ylabel("Firing Rate (Hz)")
    ax.set_facecolor('#eeeeee')

    if group_column and norm:
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="3%", pad=0.1)
        sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
        sm.set_array([])
        cb = plt.colorbar(sm, cax=cax)
        cb.set_label("SZ Factor (%)")

    ax.legend(loc="upper right", bbox_to_anchor=(0.90, 1), fontsize=9)
    xticks = ax.get_xticks()
    ax.set_xticks(xticks)  # Imposta esplicitamente i tick
    ax.set_xticklabels([f"{int(x+k)}" for x in xticks])
    ax.set_ylim(ylims[0], ylims[1])

    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(_prefix_output(save_path), bbox_inches="tight", dpi=300)
        print(f"✅ Saved figure to: {save_path}")

    if ax is None:
        plt.close()
        
    return epochs


In [ ]:
df_all.keys()

In [ ]:
df_all['sigma'].sort_values().unique()

In [ ]:
df_all['SZ Factor (%)'] = df_all['Atrophy Factor (%)']

In [ ]:
df_all.keys()

In [ ]:
df_all.keys()

# pulse supporting

In [ ]:
import numpy as np
import pandas as pd

def extract_mf_pulses(df,
                      stim_column='stim_profile',
                      onset_column='burst_onsets_ms',
                      dt=0.1,
                      window_ms=(-50, 150),
                      thr_frac=0.1):
    """
    Estrae feature dei pulsi MF da stim_profile:
    - durata (ms)
    - intensità (peak, stessa unità di stim_profile)

    Ritorna:
        durations_ms, intensities, example_pulse_time, example_pulse
    """
    win_pre = int(abs(window_ms[0]) / dt)
    win_post = int(window_ms[1] / dt)

    durations = []
    intensities = []
    example_pulse = None
    example_time = None

    for _, row in df.iterrows():
        stim = np.array(row[stim_column], dtype=float)
        onsets = row[onset_column]

        for onset in onsets:
            center = int(onset / dt)
            start = center + int(window_ms[0] / dt)
            end = center + int(window_ms[1] / dt)

            if start < 0 or end > len(stim):
                continue

            seg = stim[start:end]
            peak = np.max(seg)
            if peak <= 0:
                continue

            # soglia relativa per definire il pulso (es. 10% del picco)
            thr = thr_frac * peak
            above = np.where(seg >= thr)[0]
            if above.size == 0:
                continue

            dur_ms = (above[-1] - above[0]) * dt
            durations.append(dur_ms)
            intensities.append(peak)

            # salva un esempio di pulso (il primo che troviamo)
            if example_pulse is None:
                example_pulse = seg.copy()
                example_time = np.arange(start, end) * dt  # tempo assoluto in ms
                # se lo vuoi relativo all'onset:
                # example_time = np.arange(len(seg)) * dt + window_ms[0]

    durations = np.array(durations, dtype=float)
    intensities = np.array(intensities, dtype=float)

    return durations, intensities, example_time, example_pulse


In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import linregress
import os
import numpy as np

sig = 100
df_mf = df_all[
    (df_all["resilience"] == "Without Resilience") &
    (df_all["sigma"] == sig)
]


def plot_mf_pulse_duration_intensity(df,
                                     stim_column='stim_profile',
                                     onset_column='burst_onsets_ms',
                                     dt=0.1,
                                     window_ms=(-50, 150),
                                     thr_frac=0.1,
                                     title_suffix="",
                                     save=True,
                                     outdir="./tempPlots",
                                     fname="mf_pulse_duration_intensity.png",
                                     df_pulses=None):
    if df_pulses is None:
        df_pulses = df

    durations, intensities, _, _ = extract_mf_pulses(
        df_pulses,
        stim_column=stim_column,
        onset_column=onset_column,
        dt=dt,
        window_ms=window_ms,
        thr_frac=thr_frac
    )

    if durations.size == 0:
        raise ValueError("Nessun pulso MF valido estratto.")

    df_example = df.sample(n=1, random_state=1).iloc[0]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    ax0 = axes[0]
    ax0.scatter(durations, intensities, s=15, alpha=0.7)
    ax0.set_xlabel("Pulse duration (ms)")
    ax0.set_ylabel("Pulse intensity (peak, Hz)")
    ax0.set_title(f"MF pulses {title_suffix}")

    # label pannello sinistra
    ax0.text(-0.12, 1.05, "a", transform=ax0.transAxes, fontsize=13, fontweight="bold")

    if durations.size >= 2:
        slope, intercept, r, p, _ = linregress(durations, intensities)
        x_fit = np.linspace(durations.min(), durations.max(), 100)
        y_fit = slope * x_fit + intercept
        ax0.plot(x_fit, y_fit, linewidth=1)
        #ax0.text(
        #    0.05, 0.95,
        #    f"r={r:.2f}, p={p:.2e}",
        #    transform=ax0.transAxes,
        #    ha="left", va="top",
        #    fontsize=9,
        #    bbox=dict(boxstyle="round", facecolor="white", alpha=0.7, linewidth=0)
        #)

    ax1 = axes[1]
    stim = np.asarray(df_example[stim_column], dtype=float)
    t = np.arange(len(stim)) * dt
    ax1.plot(t, stim, linewidth=1.2, alpha=0.9)

    onsets = df_example[onset_column]
    if isinstance(onsets, (list, np.ndarray)):
        for onset in onsets:
            ax1.axvline(onset, linestyle=':', linewidth=0.8, alpha=0.4)

    ax1.set_xlabel("Time (ms)")
    ax1.set_ylabel("Stimulus on MF (Hz)")
    ax1.set_title("Example of MF stimulus time series")

    # label pannello destra
    ax1.text(-0.12, 1.05, "b", transform=ax1.transAxes, fontsize=13, fontweight="bold")

    plt.tight_layout()

    if save:
        os.makedirs(outdir, exist_ok=True)
        fullpath = os.path.join(outdir, fname)
        plt.savefig(_prefix_output(fullpath), dpi=300, bbox_inches="tight")
        print("Salvato:", fullpath)

    plt.show()



sig = 100
df_mf = df_all[
    (df_all["resilience"] == "Without Resilience") &
    (df_all["sigma"] == sig)
]

plot_mf_pulse_duration_intensity(
    df=df_mf,
    df_pulses=df_all,
    stim_column="stim_profile",
    onset_column="burst_onsets_ms",
    dt=0.1,
    window_ms=(-50, 150),
    thr_frac=0.1,
    title_suffix=f"",
    save=True,
    outdir="./tempPlots",
    fname=f"mf_pulses_duration_intensity_sigma{sig}.png"
)




# raster and PSTH

In [ ]:
import numpy as np
import pandas as pd
from typing import Union

from scipy.stats import pearsonr
from collections import defaultdict, Counter
from pathlib import Path
from fooof import FOOOF
import matplotlib.pyplot as plt
from math import log2
import pickle
import gc
import neo
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict
from scipy.signal import convolve
from scipy.signal.windows import triang
from pylatex import Document, Section, Figure, NoEscape
import argparse
import re
import yaml
import numpy as np
from collections import Counter
from math import log2
from fooof import FOOOF
from scipy.signal import welch
from scipy.signal import hilbert
from scipy.stats import pearsonr
import seaborn as sns
import argparse
import matplotlib
import matplotlib.cm as cm
import matplotlib.colors as mcolors

def extract_spike_df(path, downsample_target_rate_hz=None, duration_ms=None):
    spikes = []
    for f in sorted(Path(path).glob("*.nio")):
        reader = neo.io.NixIO(filename=str(f), mode="ro")
        block = reader.read_block()
        for seg_idx, segment in enumerate(block.segments):
            for st_idx, st in enumerate(segment.spiketrains):
                times = st.times.rescale("ms").magnitude
                senders = st.annotations.get("senders", [f"unit{st_idx}"])
                if not isinstance(senders, (list, np.ndarray)):
                    senders = [senders]
                #pop = st.annotations.get("device", "unknown").split("_")[0]
                pop = st.annotations.get("device", "unknown").removesuffix("_record")

                for time, sid in zip(times, senders):
                    spikes.append({
                        "time_ms": time,
                        "sender_id": sid,
                        "pop": pop,
                        "file": f.name
                    })
        del reader, block
        gc.collect()

    df_spikes = pd.DataFrame(spikes)
    if downsample_target_rate_hz and not df_spikes.empty:
        if duration_ms is None:
            duration_ms = df_spikes["time_ms"].max()
        total_time_s = duration_ms / 1000.0
        max_spikes = downsample_target_rate_hz * total_time_s
        keep_fraction = min(1.0, max_spikes / len(df_spikes))
        df_spikes = df_spikes.sample(frac=keep_fraction, random_state=42).reset_index(drop=True)
        print(f"🔻 Downsampled: conservati {len(df_spikes)} spike su {len(spikes)} ({keep_fraction:.3f} = ~{downsample_target_rate_hz:.1f} Hz)")
    return df_spikes


def build_spike_dfs(base_dir, downsample_target_rate_hz=None):
    base_dir = Path(base_dir)
    spike_dfs = {}

    for path in base_dir.rglob("*_deg_*/3.function"):
        match = re.search(r"deg_(\d+\.\d+)", str(path))
        if not match:
            print(f"⚠️ Nessun k trovato in: {path}")
            continue

        k = float(match.group(1))
        print(f"🔎 Estraggo spike per k={k:.3f} da {path}")

        df = extract_spike_df(
            path,
            downsample_target_rate_hz=downsample_target_rate_hz,
            duration_ms=None
        )

        if df.empty:
            print(f"⚠️ Nessuno spike per k={k:.3f}")
            continue

        spike_dfs[k] = df.copy()

    return spike_dfs


from pathlib import Path
import re
import numpy as np

import numpy as np
import pandas as pd
import pickle
import os
from pathlib import Path
import re

def build_spike_df_for_k(base_dir, k_target, downsample_target_rate_hz=None,
                         save_dir="./tempPlots"):
    base_dir = Path(base_dir)
    k_target = float(k_target)

    os.makedirs(save_dir, exist_ok=True)
    save_path = _prefix_output(Path(save_dir) / f"spikes_k{k_target:.3f}.pkl")

    if save_path.exists():
        print(f"📦 Spike già estratti → caricati da: {save_path}")
        return pickle.load(open(save_path, "rb"))

    for path in base_dir.rglob("*_deg_*/3.function"):
        match = re.search(r"deg_(\d+\.\d+)", str(path))
        if not match:
            continue

        k = float(match.group(1))
        if not np.isclose(k, k_target, atol=1e-6):
            continue

        print(f"🔎 Estraggo spike per k={k:.3f} da: {path}")

        df_spikes = extract_spike_df(
            path,
            downsample_target_rate_hz=downsample_target_rate_hz,
            duration_ms=None
        )

        if df_spikes.empty:
            print(f"⚠️ Nessuno spike trovato per k={k_target:.3f}")
            return df_spikes

        with open(save_path, "wb") as f:
            pickle.dump(df_spikes, f)

        print(f"💾 Salvato spike_df in: {save_path}")
        return df_spikes

    raise FileNotFoundError(
        f"Nessuna cartella per k={k_target:.3f} trovata in {base_dir}"
    )





In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

def plot_classical_raster(df_spikes,
                          df_row_with_stim,
                          pop="mossy_fibers",
                          window=None,
                          save_path=None,
                          event_times=None,
                          title="Classical Raster Plot with Stimulus"):

    df_plot = df_spikes[df_spikes["pop"] == pop].copy()

    # Se c'è una finestra e una lista di eventi → allineo agli eventi
    if window and event_times is not None:
        spikes = []
        for t_ref in event_times:
            aligned = df_plot["time_ms"] - t_ref
            mask = (aligned >= window[0]) & (aligned <= window[1])
            spikes.append(df_plot.loc[mask].assign(time_aligned=aligned[mask]))

        if len(spikes) == 0:
            return

        df_plot = pd.concat(spikes, ignore_index=True)
        time_col = "time_aligned"
        stim_x = np.arange(len(df_row_with_stim["stim_profile"])) - event_times[0]
    else:
        time_col = "time_ms"
        stim_x = np.arange(len(df_row_with_stim["stim_profile"]))

    stim_y = np.array(df_row_with_stim["stim_profile"])

    atrophy = df_row_with_stim.get("atrophy_percent", None)
    if atrophy is not None:
        title = f"{pop.capitalize()} – Raster aligned to event – Atrophy {atrophy:.1f}%"

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.scatter(df_plot[time_col], df_plot["sender_id"], s=1, color="black", label="Spikes")
    ax.set_xlabel("Time (ms)" if not window else "Time from event (ms)")
    ax.set_ylabel("Neuron ID")
    ax.set_title(title)

    if window:
        ax.axvline(0, color="red", linestyle="--", label="Event onset")
        durations = df_row_with_stim.get("burst_durations_ms", [])
        if durations is not None and len(durations) > 0:
            stim_dur = np.mean(durations)
            ymin, ymax = ax.get_ylim()
            ax.fill_betweenx([ymin, ymax], 0, stim_dur, color="red", alpha=0.3, label="Stim duration")
        ax.legend()

    # inset con lo stimolo
    ax2 = ax.inset_axes([0, -0.15, 1, 0.1], transform=ax.transAxes)
    ax2.plot(stim_x, stim_y, color="orange")
    ax2.set_xlim(ax.get_xlim())
    ax2.axis("off")

    plt.tight_layout()

    # --- salvataggio in ./tempPlots di default ---
    if save_path is None:
        os.makedirs("./tempPlots", exist_ok=True)
        safe_pop = pop.replace(" ", "_")
        save_path = f"./tempPlots/raster_{safe_pop}.png"

    plt.savefig(_prefix_output(save_path), dpi=300)
    #plt.close()



def average_psth(df_spikes,
                 pop="mossy_fibers",
                 event_times=None,
                 window=(-200, 200),
                 bin_size=5.0,
                 dt=0.1,
                 n_min_events=10,
                 save_path=None,
                 condition_name="cond",
                 plot_raster=False,
                 df_row_with_stim=None):

    if event_times is None or len(event_times) < n_min_events:
        print("⚠️ Non ci sono eventi sufficienti per la PSTH.")
        return

    bins = np.arange(window[0], window[1] + bin_size, bin_size)
    all_counts = []
    raster_data = []

    pop_mask = df_spikes["pop"] == pop
    times_all = df_spikes.loc[pop_mask, "time_ms"].values

    for idx, t_ref in enumerate(event_times):
        aligned = times_all - t_ref
        counts, _ = np.histogram(aligned, bins=bins)
        all_counts.append(counts)

        if plot_raster:
            aligned_in_window = aligned[(aligned >= window[0]) & (aligned <= window[1])]
            for i, t in enumerate(aligned_in_window):
                if i % 10 == 0:
                    raster_data.append((idx, t))

    all_counts = np.array(all_counts)
    n_trials = len(event_times)
    centers = bins[:-1] + bin_size / 2

    # firing rate in spikes/s
    #mean_rate = all_counts.mean(axis=0) / (bin_size / 1000.0 * n_trials)
    #sem_rate = all_counts.std(axis=0) / np.sqrt(n_trials) / (bin_size / 1000.0 * n_trials)
    mean_rate = all_counts.mean(axis=0) / (bin_size * 2 / 1000) / n_trials
    sem_rate  = all_counts.std(axis=0)  / np.sqrt(n_trials) / (bin_size *2 / 1000) / n_trials


    colors = ["black" if c < 0 else "white" for c in centers]

    fig = plt.figure(figsize=(8, 6 if plot_raster else 4))
    atrophy = df_row_with_stim.get("atrophy_percent", "N/A") if df_row_with_stim is not None else "N/A"

    if plot_raster:
        ax1 = fig.add_subplot(211)
        ax1.scatter([t for _, t in raster_data], [i for i, _ in raster_data], s=2, color="black")
        ax1.axvline(0, color="red", linestyle="--")
        ax1.set_ylabel("Trial")
        ax1.set_title(f"Atrophy {atrophy:.1f}% – Raster & PSTH ({pop})")
        if df_row_with_stim is not None:
            durations = df_row_with_stim.get("burst_durations_ms", [])
            if durations is not None and len(durations) > 0:
                stim_dur = np.mean(durations)
                ax1.fill_betweenx([0, len(event_times)], 0, stim_dur, color="red", alpha=0.3)
        ax2 = fig.add_subplot(212, sharex=ax1)
    else:
        ax2 = fig.add_subplot(111)
        ax2.set_title(f"Atrophy {atrophy:.1f}% – PSTH ({pop}, n={n_trials} eventi)")

    ax2.bar(centers, mean_rate, width=bin_size, color=colors,
            edgecolor="black", alpha=0.9, label="mean PSTH")
    ax2.errorbar(centers, mean_rate, yerr=sem_rate,
                 fmt="none", ecolor="gray", elinewidth=1, capthick=1, alpha=0.7)

    ax2.axvline(0, color="red", linestyle="--", label="Onset")
    if df_row_with_stim is not None:
        durations = df_row_with_stim.get("burst_durations_ms", [])
        if durations is not None and len(durations) > 0:
            stim_dur = np.mean(durations)
            ax2.fill_betweenx([0, mean_rate.max() * 1.1], 0, stim_dur,
                              color="red", alpha=0.3, label="Stim duration")

    ax2.set_xlabel("Time from event (ms)")
    ax2.set_ylabel("Discharge rate (spikes/s)")
    ax2.legend()
    plt.tight_layout()

    # --- salvataggio in ./tempPlots di default ---
    if save_path is None:
        os.makedirs("./tempPlots", exist_ok=True)
        safe_pop = pop.replace(" ", "_")
        save_path = f"./tempPlots/psth_{safe_pop}.png"

    plt.savefig(_prefix_output(save_path), dpi=300)
    #plt.close()

import yaml
import numpy as np
from pathlib import Path

import yaml
import numpy as np
from pathlib import Path
import re

def get_df_row_with_stim_for_k(base_dir,
                               k_target,
                               stimName="basal_activity",
                               mainEventKey="burst",
                               dt=0.1):
    base_dir = Path(base_dir)
    k_target = float(k_target)

    # trova la cartella 3.function per quel k (stesso pattern di loadData)
    target_3func = None
    for path in base_dir.rglob("*_deg_*/3.function"):
        m = re.search(r"deg_(\d+\.\d+)", str(path))
        if not m:
            continue
        k = float(m.group(1))
        if np.isclose(k, k_target, atol=1e-6):
            target_3func = path
            break

    if target_3func is None:
        raise FileNotFoundError(f"Nessuna cartella 3.function trovata per k={k_target:.3f} sotto {base_dir}")

    # YAML come in loadData:
    yaml_file = target_3func.parent / "2.stimulus" / f"{stimName}_{k_target:.3f}.yaml"
    if not yaml_file.exists():
        raise FileNotFoundError(f"YAML stimolo non trovato: {yaml_file}")

    with open(yaml_file, "r") as f:
        yaml_data = yaml.safe_load(f)

    devices = yaml_data.get("simulations", {}).get(stimName, {}).get("devices", {})
    if not devices:
        raise ValueError(f"Nessun 'devices' trovato in {yaml_file} per stimName='{stimName}'")

    burst_onsets_ms = []
    burst_durations_ms = []
    burst_rates_hz = []

    # come in loadData: prendi solo i poisson che iniziano per mainEventKey
    for name, props in devices.items():
        if name.startswith(mainEventKey) and props.get("device") == "poisson_generator":
            start = float(props["start"])
            stop = float(props["stop"])
            rate = float(props.get("rate", 0.0))
            burst_onsets_ms.append(start)
            burst_durations_ms.append(stop - start)
            burst_rates_hz.append(rate)

    # durata totale per stim_profile
    duration = max(
        [float(p.get("stop", 0.0)) for p in devices.values()
         if p.get("device") == "poisson_generator"] + [0.0]
    )

    # usa la tua generate_stim_profile se è già definita,
    # altrimenti questa è identica:
    def _generate_stim_profile(devices, sim_duration, dt=0.1):
        stim_profile = np.zeros(int(sim_duration // dt))
        for _, props in devices.items():
            if props.get("device") != "poisson_generator":
                continue
            start = float(props.get("start", 0.0))
            stop = float(props.get("stop", sim_duration))
            rate = float(props.get("rate", 0.0))
            s_idx = int(start // dt)
            e_idx = int(stop // dt)
            stim_profile[s_idx:e_idx] += rate
        return stim_profile

    stim_profile = _generate_stim_profile(devices, duration, dt)

    atrophy = 100 * (1 - k_target)

    df_row_with_stim = {
        "stim_profile": stim_profile,
        "burst_onsets_ms": np.array(burst_onsets_ms, dtype=float),
        "burst_durations_ms": np.array(burst_durations_ms, dtype=float),
        "burst_rates_hz": np.array(burst_rates_hz, dtype=float),
        "atrophy_percent": atrophy,
        "k": k_target,
    }

    return df_row_with_stim



In [ ]:
base_dir = "20251113161643_kase_stim_vitro_2_0_glutamate_all"
k_desiderato = 0.707

df_spikes_k = build_spike_df_for_k(
    base_dir,
    k_target=k_desiderato,
    downsample_target_rate_hz=None
)

df_row_with_stim = get_df_row_with_stim_for_k(
    base_dir,
    k_target=k_desiderato,
    stimName="kase_stim_vitro",   # o quello che stai usando
    mainEventKey="burst",
    dt=0.1
)

average_psth(
    df_spikes_k,
    pop="purkinje",
    event_times=df_row_with_stim["burst_onsets_ms"],
    window=(-200, 200),
    bin_size=5,
    dt=0.1,
    n_min_events=5,
    save_path="./tempPlots/pc_psth_k1.000.png",
    plot_raster=True,
    df_row_with_stim=df_row_with_stim
)





In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

def _raster_on_ax(ax, df_spikes, df_row_with_stim,
                  pop="purkinje", window=(-200,200),
                  event_times=None, title=""):
    df_plot = df_spikes[df_spikes["pop"] == pop].copy()
    if window and event_times is not None:
        spikes = []
        for t_ref in event_times:
            aligned = df_plot["time_ms"] - t_ref
            mask = (aligned >= window[0]) & (aligned <= window[1])
            spikes.append(df_plot.loc[mask].assign(time_aligned=aligned[mask]))
        if len(spikes) == 0:
            return
        import pandas as pd
        df_plot = pd.concat(spikes, ignore_index=True)
        time_col = "time_aligned"
    else:
        time_col = "time_ms"
    atrophy = df_row_with_stim.get("atrophy_percent", None)
    if atrophy is not None and title == "":
        title = f"{pop.capitalize()} – Raster – Atrophy {atrophy:.1f}%"
    ax.scatter(df_plot[time_col], df_plot["sender_id"], s=1, color="black")
    ax.set_xlabel("Time from event (ms)")
    ax.set_ylabel("Neuron ID")
    ax.set_title(title)
    if window:
        ax.axvline(0, color="red", linestyle="--")
        durations = df_row_with_stim.get("burst_durations_ms", [])
        if durations is not None and len(durations) > 0:
            stim_dur = np.mean(durations)
            ymin, ymax = ax.get_ylim()
            ax.fill_betweenx([ymin, ymax], 0, stim_dur, color="red", alpha=0.3)

def _psth_on_ax(ax, df_spikes, df_row_with_stim,
                pop="purkinje",
                event_times=None,
                window=(-200,200),
                bin_size=5.0,
                dt=0.1,
                n_min_events=5,
                title=""):
    if event_times is None or len(event_times) < n_min_events:
        print("⚠️ Non ci sono eventi sufficienti per la PSTH.")
        return
    bins = np.arange(window[0], window[1] + bin_size, bin_size)
    all_counts = []
    pop_mask = df_spikes["pop"] == pop
    times_all = df_spikes.loc[pop_mask, "time_ms"].values
    for t_ref in event_times:
        aligned = times_all - t_ref
        counts, _ = np.histogram(aligned, bins=bins)
        all_counts.append(counts)
    all_counts = np.array(all_counts)
    n_trials = len(event_times)
    centers = bins[:-1] + bin_size/2
    # normalizzazione standard (spikes/s)
    mean_rate = all_counts.mean(axis=0) / (bin_size*2/1000.0 * n_trials)
    sem_rate  = all_counts.std(axis=0)  / np.sqrt(n_trials) / (bin_size*2/1000.0 * n_trials)
    colors = ["black" if c < 0 else "white" for c in centers]
    atrophy = df_row_with_stim.get("atrophy_percent", "N/A")
    if title == "":
        title = f"Atrophy {atrophy:.1f}% – PSTH ({pop}, n={n_trials} eventi)"
    ax.bar(centers, mean_rate, width=bin_size, color=colors,
           edgecolor="black", alpha=0.9, label="mean PSTH")
    ax.errorbar(centers, mean_rate, yerr=sem_rate,
                fmt="none", ecolor="gray", elinewidth=1, capthick=1, alpha=0.7)
    ax.axvline(0, color="red", linestyle="--", label="Onset")
    durations = df_row_with_stim.get("burst_durations_ms", [])
    if durations is not None and len(durations) > 0:
        stim_dur = np.mean(durations)
        ax.fill_betweenx([0, mean_rate.max()*1.1], 0, stim_dur,
                         color="red", alpha=0.3, label="Stim duration")
    ax.set_xlabel("Time from event (ms)")
    ax.set_ylabel("Discharge rate (spikes/s)")
    ax.set_title(title)
    ax.legend(fontsize=8)

def plot_3x2_raster_psth_grid(base_dir_no_res,
                              base_dir_res,
                              k_full=1.0,
                              k_partial=0.707,
                              pop="purkinje",
                              stimName="kase_stim_vitro",
                              mainEventKey="burst",
                              window=(-200,200),
                              bin_size=5.0,
                              dt=0.1,
                              save_path="./tempPlots/grid_raster_psth.png"):
    os.makedirs("./tempPlots", exist_ok=True)
    fig, axes = plt.subplots(3, 2, figsize=(10, 9), sharex="col")
    rows = [
        ("k=1.000 (no resilience)", base_dir_no_res, float(k_full)),
        ("k=0.707 (no resilience)", base_dir_no_res, float(k_partial)),
        ("k=0.707 (resilience)",    base_dir_res,    float(k_partial)),
    ]
    for i, (row_label, base_dir, k_val) in enumerate(rows):
        df_spikes_k = build_spike_df_for_k(
            base_dir,
            k_target=k_val,
            downsample_target_rate_hz=None
        )
        df_row_with_stim = get_df_row_with_stim_for_k(
            base_dir,
            k_target=k_val,
            stimName=stimName,
            mainEventKey=mainEventKey,
            dt=dt
        )
        event_times = df_row_with_stim["burst_onsets_ms"]
        ax_raster = axes[i, 0]
        ax_psth   = axes[i, 1]
        _raster_on_ax(
            ax_raster,
            df_spikes_k,
            df_row_with_stim,
            pop=pop,
            window=window,
            event_times=event_times,
            title=f"{row_label} – Raster"
        )
        _psth_on_ax(
            ax_psth,
            df_spikes_k,
            df_row_with_stim,
            pop=pop,
            event_times=event_times,
            window=window,
            bin_size=bin_size,
            dt=dt,
            n_min_events=5,
            title=f"{row_label} – PSTH"
        )
    plt.tight_layout()
    plt.savefig(_prefix_output(save_path), dpi=300)
    print(f"✅ Figura 3×2 salvata in: {save_path}")
    # plt.close()

base_dir_no_res = "20251113161643_kase_stim_vitro_2_0_glutamate_all"
base_dir_res    = "20251113214858_kase_stim_vitro_15_0_glutamate_all"
plot_3x2_raster_psth_grid(
    base_dir_no_res,
    base_dir_res,
    k_full=1.0,
    k_partial=0.707,
    pop="purkinje",
    stimName="kase_stim_vitro",
    save_path="./tempPlots/grid_purk_3x2.png"
)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os
from string import ascii_lowercase as _az
import pandas as pd

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os
from string import ascii_lowercase as _az
import pandas as pd


def compute_sigma_ranges(df,
                         lag_col="optimal_lag_ms",
                         te_col="te_mossy_fibers_to_purk",
                         lag_target=7.5,
                         top_quantile=0.8,
                         te_weight=1.0,
                         lag_weight=1.0,
                         cv_weight=0.5):
    """
    Sceglie, per ogni livello di 'resilience', un range di sigma plausibile
    usando lag e TE:

    - preferisce TE alta
    - penalizza lag lontani dal target fisiologico (lag_target)
    - penalizza alta variabilità (CV di lag e TE)

    Ritorna: dict[resilience] -> {
        'df_sigma': DataFrame per debug,
        'sigma_min': float,
        'sigma_max': float
    }
    """
    results = {}

    for res in df["resilience"].unique():
        sub = df[df["resilience"] == res].copy()
        if sub.empty:
            results[res] = None
            continue

        rows = []
        for sigma_val, g in sub.groupby("sigma"):
            lag_vals = g[lag_col].to_numpy(dtype=float)
            te_vals  = g[te_col].to_numpy(dtype=float)

            lag_vals = lag_vals[np.isfinite(lag_vals)]
            te_vals  = te_vals[np.isfinite(te_vals)]
            if lag_vals.size == 0 or te_vals.size == 0:
                continue

            mean_lag = float(np.mean(lag_vals))
            std_lag  = float(np.std(lag_vals, ddof=1)) if lag_vals.size > 1 else 0.0
            mean_te  = float(np.mean(te_vals))
            std_te   = float(np.std(te_vals, ddof=1)) if te_vals.size > 1 else 0.0

            cv_lag = std_lag / mean_lag if mean_lag != 0 else np.nan
            cv_te  = std_te  / mean_te  if mean_te  != 0 else np.nan

            rows.append(dict(
                resilience=res,
                sigma=float(sigma_val),
                mean_lag=mean_lag,
                std_lag=std_lag,
                cv_lag=cv_lag,
                mean_te=mean_te,
                std_te=std_te,
                cv_te=cv_te
            ))

        if not rows:
            results[res] = None
            continue

        df_sigma = pd.DataFrame(rows).sort_values("sigma").reset_index(drop=True)

        # normalizza TE tra 0 e 1
        te_min = df_sigma["mean_te"].min()
        te_max = df_sigma["mean_te"].max()
        te_norm = (df_sigma["mean_te"] - te_min) / (te_max - te_min + 1e-9)

        # errore relativo di lag rispetto al target
        err_lag = np.abs(df_sigma["mean_lag"] - lag_target) / (lag_target + 1e-9)

        cv_lag = df_sigma["cv_lag"].fillna(df_sigma["cv_lag"].max())
        cv_te  = df_sigma["cv_te"].fillna(df_sigma["cv_te"].max())

        score = (
            te_weight  * te_norm
            - lag_weight * err_lag
            - cv_weight * cv_lag
            - cv_weight * cv_te
        )
        df_sigma["score"] = score

        thr = df_sigma["score"].quantile(top_quantile)
        good = df_sigma[df_sigma["score"] >= thr]

        sigma_min = float(good["sigma"].min())
        sigma_max = float(good["sigma"].max())

        print(f"[RANGE σ] {res}: [{sigma_min:.1f}, {sigma_max:.1f}] ms "
              f"(score ≥ quantile {top_quantile:.2f})")

        results[res] = dict(
            df_sigma=df_sigma,
            sigma_min=sigma_min,
            sigma_max=sigma_max
        )

    return results

In [ ]:
df_all.keys()

In [ ]:
def plot_heatmaps_resilience_3cols(df, x_col=None, sigma_min=20,
                                   percentile_thr=95,   # ora NON usato (niente quadrati rossi)
                                   save=True, outdir="./tempPlots"):
    """
    VERSIONE MODIFICATA:
      - niente riquadri rossi dei massimi
      - seleziona un range plausibile di sigma usando TE e lag
      - ritorna sigma_ranges da passare a plot_sigma_collapsed_curves
    """

    # --- colonna atrofia ---
    if x_col is None:
        if "SZ Factor (%)" in df.columns:
            x_col = "SZ Factor (%)"
        elif "atrophy_percent" in df.columns:
            x_col = "atrophy_percent"
        else:
            raise ValueError("Nessuna colonna di atrofia trovata (SZ Factor (%) / atrophy_percent).")

    if "sigma" not in df.columns:
        raise ValueError("Manca la colonna 'sigma' nel DataFrame.")
    if "resilience" not in df.columns:
        raise ValueError("Manca la colonna 'resilience' nel DataFrame.")

    # --- filtro sigma ---
    df = df[df["sigma"] >= sigma_min].copy()
    if df.empty:
        raise ValueError(f"Nessun dato dopo il filtro sigma >= {sigma_min}")

    # 🔹 calcolo range di sigma plausibile per ogni resilience
    sigma_ranges = compute_sigma_ranges(df)

    os.makedirs(outdir, exist_ok=True)
    plt.rcParams.update({"font.size": 10})

    # metriche da plottare in heatmap
    metrics = [
        ("optimal_lag_ms", "Lag (ms)"),
        ("te_mossy_fibers_to_purk", r"TE (bit)"),
        
        #("cosine_similarity_mossy_fibers_purk", "Cosine Similarity MF–PC"),
        #("plv", "PLV"),
        #("cross_corr", "cross_corr")
    ]

    # --- scale colore globali per metrica (su df filtrato) ---
    color_ranges = {}
    for col, _ in metrics:
        if col in df.columns:
            vals = df[col].to_numpy(dtype=float)
            vals = vals[~np.isnan(vals)]
            if vals.size > 0:
                vmin, vmax = float(np.min(vals)), float(np.max(vals))
            else:
                vmin, vmax = None, None
        else:
            vmin, vmax = None, None
        color_ranges[col] = (vmin, vmax)

    print("=== Color scale ranges (after sigma filter) ===")
    for col, (vmin, vmax) in color_ranges.items():
        print(f"{col:30s} -> vmin={vmin}, vmax={vmax}")
    print("==============================================")

    resilience_levels = ["Without Resilience", "With Resilience"]

    n_rows = len(metrics)
    n_cols = 2

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(14, 4.0 * n_rows),
        sharey=False
    )

    if n_rows == 1:
        axes = np.array([axes])

    letter_idx = 0

    for r, (col, label) in enumerate(metrics):

        vmin, vmax = color_ranges.get(col, (None, None))

        pivots = {}
        stats = {}
        for res in resilience_levels:
            sub = df[df["resilience"] == res]

            piv = sub.pivot_table(
                index="sigma",
                columns=x_col,
                values=col,
                aggfunc="mean"
            )
            piv = piv.sort_index(axis=0).sort_index(axis=1)
            pivots[res] = piv

            # stats per pannello
            sub_vals = sub[col].to_numpy(dtype=float)
            sub_vals = sub_vals[np.isfinite(sub_vals)]
            if sub_vals.size > 0:
                avg_val = float(np.mean(sub_vals))
                sem_val = float(np.std(sub_vals, ddof=1) / np.sqrt(sub_vals.size))
            else:
                avg_val = np.nan
                sem_val = np.nan
            stats[res] = (avg_val, sem_val)

        for c, res in enumerate(resilience_levels):

            ax = axes[r, c]
            piv = pivots[res]

            hm = sns.heatmap(
                piv,
                ax=ax,
                cmap="viridis",
                cbar_kws={"shrink": 0.7},
                vmin=vmin,
                vmax=vmax,
                square=False
            )

            # 🔹 fascia orizzontale per il range di sigma scelto
            rng = sigma_ranges.get(res, None)
            if rng is not None:
                smin, smax = rng["sigma_min"], rng["sigma_max"]
                y_vals = piv.index.astype(float)
                mask_range = (y_vals >= smin - 1e-6) & (y_vals <= smax + 1e-6)
                if mask_range.any():
                    idx_min = np.where(mask_range)[0].min()
                    idx_max = np.where(mask_range)[0].max()

                    # banda chiara
                    ax.axhspan(
                        idx_min, idx_max + 1,
                        color="white", alpha=0.18, zorder=0
                    )

                    # 🔹 linee orizzontali ai bordi dell’isola σ*
                    x0, x1 = ax.get_xlim()
                    ax.hlines(
                        [idx_min, idx_max + 1],
                        x0, x1,
                        colors="white",
                        linestyles="--",
                        linewidth=1.0,
                        zorder=1
                    )

                    # etichetta laterale del range ottimale
                    ax.text(
                        1.01,
                        (idx_min + idx_max + 1) / 2 / len(y_vals),
                        f"σ* [{smin:.0f},{smax:.0f}]",
                        transform=ax.transAxes,
                        rotation=90,
                        va="center", ha="left",
                        fontsize=9
                    )


            # --- assi e label ---
            ax.set_ylabel(r"$\sigma$ (ms)", fontsize=12)
            ax.set_xlabel("SZ Factor (%)" if x_col == "SZ Factor (%)" else x_col, fontsize=12)

            # XTICKS
            x_vals = piv.columns.astype(float)
            x_max = np.max(x_vals)
            x_ticks_ideal = np.arange(0, x_max + 1, 10)
            x_tick_pos = []
            x_tick_lab = []
            for xt in x_ticks_ideal:
                idx = np.argmin(np.abs(x_vals - xt))
                if idx not in x_tick_pos:
                    x_tick_pos.append(idx)
                    x_tick_lab.append(f"{int(xt)}")
            ax.set_xticks(np.array(x_tick_pos) + 0.5)
            ax.set_xticklabels(x_tick_lab, rotation=0, ha="right", fontsize=10)

            # YTICKS
            y_vals = piv.index.astype(float)
            y_min, y_max = np.min(y_vals), np.max(y_vals)
            y_ticks_ideal = np.arange(y_min, y_max + 1, 20)
            y_tick_pos = []
            y_tick_lab = []
            for yt in y_ticks_ideal:
                idx = np.argmin(np.abs(y_vals - yt))
                if idx not in y_tick_pos:
                    y_tick_pos.append(idx)
                    y_tick_lab.append(f"{int(yt)}")
            ax.set_yticks(y_tick_pos)
            ax.set_yticklabels(y_tick_lab, fontsize=10)

            # titoli
            if r == 0:
                ax.set_title(res, fontsize=14, fontweight="bold", pad=25)
            else:
                ax.set_title("")

            # testo media ± SEM
            avg_val, sem_val = stats[res]
            if col == "optimal_lag_ms":
                avg_text = f"average lag MF→PC = {avg_val:.3f} ± {sem_val:.3f} ms"
            elif col == "te_mossy_fibers_to_purk":
                avg_text = f"average TE MF→PC = {avg_val:.3f} ± {sem_val:.3f} bit"
            else:
                avg_text = f"average = {avg_val:.3f} ± {sem_val:.3f}"

            ax.text(
                0.5, 1.02,
                avg_text,
                transform=ax.transAxes,
                ha="center",
                va="bottom",
                fontsize=11
            )

            # label metrica
            if c == 0:
                ax.text(
                    -0.25, 0.5, label,
                    transform=ax.transAxes,
                    rotation=90,
                    fontsize=12,
                    fontweight="bold",
                    va="center",
                    ha="right"
                )

            # lettera pannello
            ax.text(
                -0.10, 1.05, _az[letter_idx],
                transform=ax.transAxes,
                fontsize=14,
                fontweight='bold',
                va='bottom',
                ha='right'
            )
            letter_idx += 1

    plt.subplots_adjust(hspace=0.5)

    if save:
        fname = os.path.join(outdir, f"supporting_heatmaps_resilience_2cols_sigma{sigma_min}_paper.png")
        plt.savefig(_prefix_output(fname), dpi=300, bbox_inches="tight")
        print("Salvato:", fname)

    plt.show()

    # 🔙 restituisco i range di sigma per usarli poi nelle curve 2x2
    return sigma_ranges

# sigma range

In [ ]:
sigma_ranges = plot_heatmaps_resilience_3cols(df_all, sigma_min=10, outdir="./tempPlots")


# aperiodic

In [ ]:
from scipy.stats import ttest_ind
import matplotlib.pyplot as plt
import numpy as np
import os

import numpy as np

def add_fooof_aperiodic_cols(df):
    """
    Aggiunge colonne:
      - purk_offset, purk_slope
    estratte da df['fooof_purk'] (dict FOOOF).
    """
    def _get_offset(d):
        if isinstance(d, dict) and "aperiodic_params" in d:
            ap = d["aperiodic_params"]
            if isinstance(ap, (list, tuple)) and len(ap) >= 2:
                return float(ap[0])
        return np.nan

    def _get_slope(d):
        if isinstance(d, dict) and "aperiodic_params" in d:
            ap = d["aperiodic_params"]
            if isinstance(ap, (list, tuple)) and len(ap) >= 2:
                return float(ap[1])
        return np.nan

    df = df.copy()
    df["purk_offset"] = df["fooof_purk"].apply(_get_offset)
    df["purk_slope"]  = df["fooof_purk"].apply(_get_slope)

    return df


def plot_aperiodic_params_vs_sz(df,
                                sigma_ranges,
                                x_col=None,
                                outdir="./tempPlots",
                                fname="aperiodic_purk_vs_sz.png",
                                alpha=0.05):
    """
    Plotta offset e slope aperiodici di FOOOF (Purkinje) in funzione di SZ / atrofia,
    mediati sul range di sigma selezionato per ogni condizione (Without / With Resilience).

    df deve avere:
      - colonna 'sigma'
      - colonna 'resilience' (valori 'Without Resilience' / 'With Resilience')
      - colonne 'purk_offset', 'purk_slope' (create con add_fooof_aperiodic_cols)
      - colonna x_col = 'SZ Factor (%)' o 'atrophy_percent'
    sigma_ranges: dict con chiavi 'Without Resilience' e 'With Resilience', es:
      {
        "Without Resilience": {"sigma_min": 30, "sigma_max": 80},
        "With Resilience":    {"sigma_min": 25, "sigma_max": 90}
      }
    """

    if x_col is None:
        if "SZ Factor (%)" in df.columns:
            x_col = "SZ Factor (%)"
        elif "atrophy_percent" in df.columns:
            x_col = "atrophy_percent"
        else:
            raise ValueError("Nessuna colonna di atrofia trovata (SZ Factor (%) / atrophy_percent).")

    metrics = [
        ("purk_offset", "Purkinje FOOOF offset"),
        ("purk_slope",  "Purkinje FOOOF slope")
    ]

    resilience_levels = ["Without Resilience", "With Resilience"]
    colors = {
        "Without Resilience": "tab:blue",
        "With Resilience": "tab:orange"
    }

    # Dizionario dove accumulo mean/std per metrica e condizione
    metric_results = {
        col: {res: {} for res in resilience_levels}
        for col, _ in metrics
    }

    # --- Pre-calcolo per ogni metrica / condizione --- #
    for col, _ in metrics:
        if col not in df.columns:
            continue

        for res in resilience_levels:
            rng = sigma_ranges.get(res, None)
            if rng is None:
                continue

            smin, smax = rng["sigma_min"], rng["sigma_max"]

            sub = df[(df["resilience"] == res) &
                     (df["sigma"] >= smin) &
                     (df["sigma"] <= smax)].copy()
            if sub.empty:
                continue

            metric_results[col][res]["sub"] = sub

            grp = sub.groupby(x_col)[col]
            x_vals = np.array(sorted(grp.groups.keys()), dtype=float)

            mean_vals = grp.mean().reindex(x_vals).to_numpy(dtype=float)
            std_vals  = grp.std(ddof=1).reindex(x_vals).to_numpy(dtype=float)
            n_vals    = grp.count().reindex(x_vals).to_numpy(dtype=int)

            metric_results[col][res]["x"]    = x_vals
            metric_results[col][res]["mean"] = mean_vals
            metric_results[col][res]["std"]  = std_vals
            metric_results[col][res]["n"]    = n_vals

    # --- Figure 1x2: offset | slope --- #
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True)
    if len(axes.shape) == 0:
        axes = np.array([axes])

    for ax, (col, ylabel) in zip(axes, metrics):

        if not any(metric_results[col][res] for res in resilience_levels):
            ax.set_visible(False)
            continue

        # Curva per ogni condizione
        for res in resilience_levels:
            res_dict = metric_results[col].get(res, {})
            if not res_dict or "x" not in res_dict:
                continue

            x_vals    = res_dict["x"]
            mean_vals = res_dict["mean"]
            std_vals  = res_dict["std"]

            ax.errorbar(
                x_vals, mean_vals, yerr=std_vals,
                fmt="-o", capsize=3,
                label=res if col == "purk_offset" else None,
                color=colors.get(res, None),
                alpha=0.9
            )

        # --- test di significatività fra condizioni (Welch t-test per ogni SZ) --- #
        resA = "Without Resilience"
        resB = "With Resilience"
        if ("sub" in metric_results[col].get(resA, {}) and
            "sub" in metric_results[col].get(resB, {})):

            subA = metric_results[col][resA]["sub"]
            subB = metric_results[col][resB]["sub"]

            xA = np.array(sorted(subA[x_col].unique()), dtype=float)
            xB = np.array(sorted(subB[x_col].unique()), dtype=float)
            common_x = sorted(set(xA).intersection(set(xB)))

            pvals = {}
            for x0 in common_x:
                valsA = subA.loc[subA[x_col] == x0, col].to_numpy(dtype=float)
                valsB = subB.loc[subB[x_col] == x0, col].to_numpy(dtype=float)
                if len(valsA) >= 2 and len(valsB) >= 2:
                    _, p = ttest_ind(valsA, valsB, equal_var=False, nan_policy="omit")
                    pvals[x0] = p

            # Shading verticale dove p < alpha
            if pvals:
                xs_sorted = np.array(sorted(pvals.keys()), dtype=float)
                if len(xs_sorted) > 1:
                    dx = np.min(np.diff(xs_sorted))
                else:
                    dx = 1.0

                ymin, ymax = ax.get_ylim()
                for x0, p in pvals.items():
                    if p < alpha:
                        ax.axvspan(
                            x0 - dx * 0.5,
                            x0 + dx * 0.5,
                            color="grey",
                            alpha=0.15,
                            zorder=-1
                        )

        ax.set_ylabel(ylabel)
        ax.set_xlabel(x_col)
        ax.grid(True, linestyle="--", alpha=0.4)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper right")

    fig.suptitle("Purkinje aperiodic FOOOF params vs " + x_col, fontsize=14)
    plt.tight_layout(rect=[0, 0, 0.95, 0.95])

    os.makedirs(outdir, exist_ok=True)
    out_path = os.path.join(outdir, fname)
    plt.savefig(_prefix_output(out_path), dpi=300, bbox_inches="tight")
    print("Salvato:", out_path)
    plt.show()

# 1) estrai offset & slope
df_all = add_fooof_aperiodic_cols(df_all)

# 2) plottali vs SZ / atrofia usando lo stesso range di sigma delle altre variabili
plot_aperiodic_params_vs_sz(
    df_all,
    sigma_ranges=sigma_ranges,         # già calcolato dalle heatmap
    x_col="SZ Factor (%)",             # o None per auto-detect
    outdir="./tempPlots",
    fname="aperiodic_purk_vs_sz.png",
    alpha=0.05
)


def reconstruct_psd_from_fooof(fooof_dict, freqs):
    """
    Ricostruisce la PSD (in scala lineare) dai parametri FOOOF.

    Parametri FOOOF:
      log10(PSD) = offset - exponent * log10(f) + somma_gaussiane

    fooof_dict: dizionario tipo quello che hai in df['fooof_purk']
    freqs      : array 1D di frequenze (Hz)
    """
    if fooof_dict is None:
        return None

    ap = fooof_dict.get("aperiodic_params", None)
    if ap is None or len(ap) < 2:
        return None

    offset, exponent = ap

    # componente aperiodica in log10-power
    log10_bg = offset - exponent * np.log10(freqs)

    # aggiungi i picchi gaussiani (sempre in log10-power)
    log10_psd = log10_bg.copy()
    for peak in fooof_dict.get("peak_params", []):
        if peak is None or len(peak) < 3:
            continue
        cf, amp, bw = peak          # center frequency, amplitude, bandwidth
        log10_psd += amp * np.exp(-(freqs - cf)**2 / (2.0 * (bw**2)))

    # torna alla scala lineare
    psd = 10.0 ** log10_psd
    return psd


import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import os
import numpy as np

def plot_purk_psd_reconstructed(
        df_all,
        freq_min=1,
        freq_max=500,
        n_freqs=200,
        save_path="./tempPlots/purk_psd_reconstructed_SZ_fixed.png"):

    # griglia di frequenze su cui ricostruire
    freqs = np.linspace(freq_min, freq_max, n_freqs)

    # colonna SZ / atrofia
    if "SZ Factor (%)" in df_all.columns:
        sz_col = "SZ Factor (%)"
    elif "atrophy_percent" in df_all.columns:
        sz_col = "atrophy_percent"
    else:
        raise ValueError("Nel DataFrame non trovo né 'SZ Factor (%)' né 'atrophy_percent'.")

    if "fooof_purk" not in df_all.columns:
        raise ValueError("Nel DataFrame non c'è la colonna 'fooof_purk'.")

    if "resilience" not in df_all.columns:
        raise ValueError("Nel DataFrame non c'è la colonna 'resilience'.")

    resilience_levels = ["Without Resilience", "With Resilience"]

    # setup colori SZ
    sz_vals = df_all[sz_col].to_numpy(dtype=float)
    norm = mcolors.Normalize(vmin=np.nanmin(sz_vals), vmax=np.nanmax(sz_vals))
    cmap = cm.viridis

    fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

    for ax, res in zip(axes, resilience_levels):
        sub = df_all[df_all["resilience"] == res].copy()

        for _, row in sub.iterrows():
            fooof_dict = row["fooof_purk"]
            psd = reconstruct_psd_from_fooof(fooof_dict, freqs)
            if psd is None:
                continue

            color = cmap(norm(float(row[sz_col])))
            ax.plot(freqs, psd, color=color, alpha=0.5, linewidth=1.0)

        ax.set_title(res, fontsize=18, fontweight="bold")
        ax.set_xlabel("Frequency (Hz)", fontsize=12)
        ax.grid(True, alpha=0.3)
        ax.set_yscale("log")  # PSD realistica in scala log

    axes[0].set_ylabel("PSD (a.u.)", fontsize=12)

    # colorbar SZ Factor
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes.ravel().tolist(), pad=0.02)
    cbar.set_label("SZ Factor (%)", fontsize=12)

    plt.tight_layout()
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(_prefix_output(save_path), dpi=300, bbox_inches="tight")
    #plt.close()
    print(f"✅ Salvata figura PSD ricostruite in: {save_path}")


df_all = add_fooof_aperiodic_cols(df_all)  # se non già fatto

plot_purk_psd_reconstructed(df_all[df_all['sigma']==90])


# supporting OBD

## v1

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def plot_te_sz_and_hessian(
        df,
        sigma_ranges,
        x_col="SZ Factor (%)",
        te_col="te_mossy_fibers_to_purk",
        outdir="./tempPlots",
        fname="te_sz_and_hessian.png",
        use_delta=False
    ):
    os.makedirs(outdir, exist_ok=True)

    resilience_levels = ["Without Resilience", "With Resilience"]
    base_colors = {"Without Resilience": "tab:blue", "With Resilience": "tab:red"}

    te_results = {}

    for res in resilience_levels:
        rng = sigma_ranges.get(res, None)
        if rng is None:
            continue
        smin, smax = rng["sigma_min"], rng["sigma_max"]
        sub = df[(df["resilience"] == res) &
                 (df["sigma"] >= smin) &
                 (df["sigma"] <= smax)].copy()
        if sub.empty or te_col not in sub.columns:
            continue

        grp = sub.groupby(x_col)[te_col]
        x_vals = np.array(sorted(grp.groups.keys()), dtype=float)
        mean_vals = grp.mean().reindex(x_vals).to_numpy(dtype=float)
        std_vals  = grp.std(ddof=1).reindex(x_vals).to_numpy(dtype=float)

        if use_delta and len(x_vals) > 0:
            if 0 in x_vals:
                idx0 = np.where(x_vals == 0)[0][0]
            else:
                idx0 = 0
            baseline = mean_vals[idx0]
            mean_vals = mean_vals - baseline

        te_results[res] = dict(
            x=x_vals,
            mean=mean_vals,
            std=std_vals
        )

    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
    ax_no, ax_wi, ax_hess = axes

    # pannello TE senza resilience
    res = "Without Resilience"
    if res in te_results:
        x = te_results[res]["x"]
        m = te_results[res]["mean"]
        s = te_results[res]["std"]
        ax_no.errorbar(
            x, m, yerr=s,
            fmt="none", ecolor="0.7", elinewidth=1, capsize=3, alpha=0.9
        )
        ax_no.scatter(
            x, m, c=x, cmap="Blues_r",
            s=40, edgecolor="k", linewidth=0.4
        )
        ax_no.plot(x, m, color=base_colors[res], alpha=0.7, linewidth=1.5)
        if len(m) > 0:
            idx_max = int(np.argmax(m))
            ax_no.scatter(
                x[idx_max], m[idx_max],
                s=100, facecolors="none", edgecolors="k", linewidth=1.5, zorder=5
            )
    ax_no.set_title("TE MF→PC vs SZ\nWithout Resilience", fontweight="bold")
    ax_no.set_xlabel(x_col)
    ax_no.set_ylabel("TE MF→PC (bit)" + (" (Δ vs SZ=0)" if use_delta else ""))
    ax_no.spines["top"].set_visible(False)
    ax_no.spines["right"].set_visible(False)

    # pannello TE con resilience
    res = "With Resilience"
    if res in te_results:
        x = te_results[res]["x"]
        m = te_results[res]["mean"]
        s = te_results[res]["std"]
        ax_wi.errorbar(
            x, m, yerr=s,
            fmt="none", ecolor="0.7", elinewidth=1, capsize=3, alpha=0.9
        )
        ax_wi.scatter(
            x, m, c=x, cmap="Reds_r",
            s=40, edgecolor="k", linewidth=0.4
        )
        ax_wi.plot(x, m, color=base_colors[res], alpha=0.7, linewidth=1.5)
        if len(m) > 0:
            idx_max = int(np.argmax(m))
            ax_wi.scatter(
                x[idx_max], m[idx_max],
                s=100, facecolors="none", edgecolors="k", linewidth=1.5, zorder=5
            )
    ax_wi.set_title("TE MF→PC vs SZ\nWith Resilience", fontweight="bold")
    ax_wi.set_xlabel(x_col)
    ax_wi.set_ylabel("TE MF→PC (bit)" + (" (Δ vs SZ=0)" if use_delta else ""))
    ax_wi.spines["top"].set_visible(False)
    ax_wi.spines["right"].set_visible(False)

    # pannello Hessiano (seconda derivata discreta rispetto a SZ)
    for res, cmap in [("Without Resilience", "Blues_r"),
                      ("With Resilience", "Reds_r")]:
        if res not in te_results:
            continue
        x = te_results[res]["x"]
        m = te_results[res]["mean"]
        if len(x) < 3:
            continue
        hess = np.zeros_like(m)
        hess[1:-1] = m[2:] - 2*m[1:-1] + m[:-2]
        ax_hess.plot(
            x, hess,
            "-o",
            label=res,
            linewidth=1.5,
            markersize=4
        )
        idx_max_abs = int(np.argmax(np.abs(hess)))
        ax_hess.scatter(
            x[idx_max_abs], hess[idx_max_abs],
            s=100, facecolors="none", edgecolors="k", linewidth=1.5, zorder=5
        )

    ax_hess.axhline(0, color="k", linestyle="--", linewidth=1, alpha=0.8)
    ax_hess.set_title("Discrete Hessian of TE", fontweight="bold")
    ax_hess.set_xlabel(x_col)
    ax_hess.set_ylabel("2nd difference of TE")
    ax_hess.spines["top"].set_visible(False)
    ax_hess.spines["right"].set_visible(False)
    ax_hess.legend(frameon=False, fontsize=9)

    # panel labels
    labels = {ax_no: "a", ax_wi: "b", ax_hess: "c"}
    for ax, lab in labels.items():
        ax.text(
            -0.10, 1.08,
            lab,
            transform=ax.transAxes,
            fontsize=14,
            fontweight="bold",
            va="top",
            ha="left"
        )

    plt.tight_layout()
    out_path = os.path.join(outdir, fname)
    plt.savefig(_prefix_output(out_path), dpi=300, bbox_inches="tight")
    print("✅ Salvato:", out_path)
    plt.show()


In [ ]:
plot_te_sz_and_hessian(
    df=df_all,
    sigma_ranges=sigma_ranges
)


## v2

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def plot_te_sz_and_hessian(
        df,
        sigma_ranges,
        x_col="SZ Factor (%)",
        te_col="te_mossy_fibers_to_purk",
        outdir="./tempPlots",
        fname="te_hessian_4panels.png",
        use_delta=False
    ):
    os.makedirs(outdir, exist_ok=True)

    resilience_levels = ["Without Resilience", "With Resilience"]
    base_colors = {"Without Resilience": "tab:blue", "With Resilience": "tab:red"}
    cmap_colors = {"Without Resilience": "Blues_r", "With Resilience": "Reds_r"}

    te_results = {}
    sz_max_te = {}  # per salvare SZ del max TE per ogni resilience

    # --- AGGREGAZIONE TE MEDIATA SU SIGMA-RANGE ---
    for res in resilience_levels:
        rng = sigma_ranges.get(res, None)
        if rng is None:
            continue
        smin, smax = rng["sigma_min"], rng["sigma_max"]

        sub = df[(df["resilience"] == res) &
                 (df["sigma"] >= smin) &
                 (df["sigma"] <= smax)].copy()

        if sub.empty or te_col not in sub.columns:
            continue

        grp = sub.groupby(x_col)[te_col]

        x_vals = np.array(sorted(grp.groups.keys()), dtype=float)
        mean_vals = grp.mean().reindex(x_vals).to_numpy(float)
        std_vals  = grp.std(ddof=1).reindex(x_vals).to_numpy(float)

        if use_delta and len(x_vals) > 0:
            if 0 in x_vals:
                idx0 = np.where(x_vals == 0)[0][0]
            else:
                idx0 = 0
            baseline = mean_vals[idx0]
            mean_vals = mean_vals - baseline

        te_results[res] = dict(
            x=x_vals,
            mean=mean_vals,
            std=std_vals
        )

        if len(mean_vals) > 0:
            idx_max = int(np.argmax(mean_vals))
            sz_max_te[res] = float(x_vals[idx_max])

    # --- FIGURA: 2x2 ---
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    # RIGA 1: TE
    ax_no_te, ax_wi_te = axes[0]
    # RIGA 2: Hessian
    ax_no_hess, ax_wi_hess = axes[1]

    map_te   = {"Without Resilience": ax_no_te, "With Resilience": ax_wi_te}
    map_hess = {"Without Resilience": ax_no_hess, "With Resilience": ax_wi_hess}

    for res in resilience_levels:
        # ====================
        #        TE
        # ====================
        ax_te = map_te[res]
        ax_te.spines["top"].set_visible(False)
        ax_te.spines["right"].set_visible(False)

        if res not in te_results:
            ax_te.text(0.5, 0.5, "No data", transform=ax_te.transAxes,
                       ha="center", va="center")
            continue

        x  = te_results[res]["x"]
        m  = te_results[res]["mean"]
        sd = te_results[res]["std"]

        # TE vs SZ
        ax_te.errorbar(
            x, m, yerr=sd,
            fmt="none",
            ecolor="0.7",
            elinewidth=1,
            capsize=3,
            alpha=0.9
        )
        ax_te.scatter(
            x, m,
            c=x,
            cmap=cmap_colors[res],
            s=40, edgecolor="k", linewidth=0.4
        )
        ax_te.plot(x, m, color=base_colors[res], linewidth=1.5)

        if res in sz_max_te:
            x_max = sz_max_te[res]
            m_max = m[np.where(x == x_max)[0][0]]
            # evidenzio il massimo
            ax_te.scatter(
                x_max, m_max,
                s=100, facecolors="none", edgecolors="k", zorder=5
            )
            # linea verticale sul max TE
            ax_te.axvline(
                x_max,
                color=base_colors[res],
                linestyle="--",
                linewidth=1.2,
                alpha=0.8
            )

        ax_te.set_title(f"TE MF→PC vs SZ\n{res}", fontweight="bold")
        ax_te.set_xlabel(x_col)
        ax_te.set_ylabel("TE MF→PC (bit)" + (" (Δ vs SZ=0)" if use_delta else ""))

        # ====================
        #      HESSIAN
        # ====================
        ax_h = map_hess[res]
        ax_h.spines["top"].set_visible(False)
        ax_h.spines["right"].set_visible(False)

        if len(x) < 3:
            ax_h.text(0.5, 0.5, "Not enough points",
                      transform=ax_h.transAxes,
                      ha="center", va="center")
            continue

        m  = np.asarray(m, dtype=float)
        sd = np.asarray(sd, dtype=float)
        N = len(m)

        # 16 punti: 1..N-2 hanno valore, 0 e N-1 messi a 0 per allineamento
        h_mean = np.zeros(N, dtype=float)
        h_std  = np.zeros(N, dtype=float)

        # i = 1..N-2: vera seconda differenza
        for i in range(1, N-1):
            h_mean[i] = m[i+1] - 2*m[i] + m[i-1]
            var_i = sd[i+1]**2 + (2*sd[i])**2 + sd[i-1]**2
            h_std[i] = np.sqrt(var_i) if var_i > 0 else 0.0

        # 0 e N-1 restano 0 (per allineare ai 16 punti TE)

        # plot completo sui 16 punti di Hessian
        ax_h.errorbar(
            x, h_mean,
            yerr=h_std,
            fmt="none",
            ecolor="0.7",
            elinewidth=1,
            capsize=3,
            alpha=0.9
        )
        ax_h.plot(
            x, h_mean,
            "-o",
            color=base_colors[res],
            linewidth=1.5,
            markersize=4
        )

        # --- individua tre minimi interni dell'Hessiano ---
        if N > 2:
            idx_internal = np.arange(1, N-1)
            h_internal = h_mean[1:-1]

            if h_internal.size > 0:
                # caso "Without Resilience": i 3 più negativi
                if res == "Without Resilience":
                    n_min = min(3, h_internal.size)
                    base_sorted = np.argsort(h_internal)[:n_min]
                    idx_minima = idx_internal[base_sorted]

                else:  # "With Resilience": 2 minimi + 1 prima del primo
                    # 2 più negativi
                    n_base = min(2, h_internal.size)
                    base_sorted = np.argsort(h_internal)[:n_base]
                    idx_base = idx_internal[base_sorted]

                    # terzo: minimo tra i punti con indice < del più "a sinistra"
                    min_idx_left = idx_base.min()
                    left_mask = idx_internal < min_idx_left
                    if np.any(left_mask):
                        # tra quelli a sinistra, prendo il minimo (più negativo)
                        h_left = h_mean[idx_internal[left_mask]]
                        idx_left_rel = np.argsort(h_left)[0]
                        idx_left = idx_internal[left_mask][idx_left_rel]
                        idx_minima = np.concatenate([idx_base, [idx_left]])
                    else:
                        # se non esiste niente a sinistra, ripiego sui 3 più negativi
                        n_min = min(3, h_internal.size)
                        base_sorted_full = np.argsort(h_internal)[:n_min]
                        idx_minima = idx_internal[base_sorted_full]

                # rimuovo eventuali duplicati e ordino per SZ
                idx_minima = np.unique(idx_minima)
                # se per qualche motivo >3, tengo i 3 più negativi tra questi
                if idx_minima.size > 3:
                    vals = h_mean[idx_minima]
                    keep = np.argsort(vals)[:3]
                    idx_minima = idx_minima[keep]

                for idxm in idx_minima:
                    # cerchio in Hessian
                    ax_h.scatter(
                        x[idxm], h_mean[idxm],
                        s=90,
                        facecolors="none",
                        edgecolors="k",
                        linewidth=1.5,
                        zorder=5
                    )
                    # linea verticale fine (:) in Hessian
                    ax_h.axvline(
                        x[idxm],
                        color=base_colors[res],
                        linestyle=":",
                        linewidth=1.2,
                        alpha=0.8
                    )
                    # linea verticale fine (:) anche nel pannello TE
                    ax_te.axvline(
                        x[idxm],
                        color=base_colors[res],
                        linestyle=":",
                        linewidth=1.2,
                        alpha=0.8
                    )
                    # cerchio anche nel pannello TE
                    ax_te.scatter(
                        x[idxm], m[idxm],
                        s=90,
                        facecolors="none",
                        edgecolors="k",
                        linewidth=1.5,
                        zorder=5
                    )

        ax_h.axhline(0, color="k", linestyle="--", linewidth=1, alpha=0.8)
        ax_h.set_title(f"Discrete Hessian of TE\n{res}", fontweight="bold")
        ax_h.set_xlabel(x_col)
        ax_h.set_ylabel(r"$\dfrac{d^2\,\mathrm{TE}}{d\,\mathrm{SZ}^2}$ (bit)")

        # linea verticale sullo stesso SZ del max TE (resta, distinto dai minimi)
        if res in sz_max_te:
            x_max = sz_max_te[res]
            ax_h.axvline(
                x_max,
                color=base_colors[res],
                linestyle="--",
                linewidth=1.2,
                alpha=0.8
            )

    # --- PANEL LABELS ---
    labels = {
        ax_no_te:   "a",
        ax_wi_te:   "b",
        ax_no_hess: "c",
        ax_wi_hess: "d"
    }
    for ax, lab in labels.items():
        ax.text(
            -0.10, 1.10,
            lab,
            transform=ax.transAxes,
            fontsize=14,
            fontweight="bold"
        )

    plt.tight_layout()
    out_path = os.path.join(outdir, fname)
    plt.savefig(_prefix_output(out_path), dpi=300, bbox_inches="tight")
    print("✅ Salvato:", out_path)
    plt.show()


In [ ]:

plot_te_sz_and_hessian(
        df_all,
        sigma_ranges,
        x_col="SZ Factor (%)",
        te_col="te_mossy_fibers_to_purk",
        outdir="./tempPlots",
        fname="te_hessian_4panels.png",
        use_delta=False)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def plot_te_sz_and_hessian_norm(
        df,
        sigma_ranges,
        x_col="SZ Factor (%)",
        te_col="te_mossy_fibers_to_purk",
        outdir="./tempPlots",
        fname="te_hessian_4panels.png",
        use_delta_vs_sz0=True
    ):
    os.makedirs(outdir, exist_ok=True)

    resilience_levels=["Without Resilience","With Resilience"]
    base_colors={"Without Resilience":"tab:blue","With Resilience":"tab:red"}
    cmap_colors={"Without Resilience":"Blues_r","With Resilience":"Reds_r"}

    te_results={}
    sz_max_te={}

    for res in resilience_levels:
        rng=sigma_ranges.get(res,None)
        if rng is None:
            continue
        smin,smax=rng["sigma_min"],rng["sigma_max"]

        sub=df[(df["resilience"]==res)&(df["sigma"]>=smin)&(df["sigma"]<=smax)].copy()
        if sub.empty or te_col not in sub.columns:
            continue

        grp=sub.groupby(x_col)[te_col]
        x_vals=np.array(sorted(grp.groups.keys()),dtype=float)
        mean_vals=grp.mean().reindex(x_vals).to_numpy(float)
        std_vals=grp.std(ddof=1).reindex(x_vals).to_numpy(float)

        if use_delta_vs_sz0 and len(x_vals)>0:
            if 0 in x_vals:
                idx0=np.where(x_vals==0)[0][0]
            else:
                idx0=0
            baseline=float(mean_vals[idx0])
            if np.isfinite(baseline):
                mean_vals=mean_vals-baseline
            else:
                mean_vals[:]=np.nan
                std_vals[:]=np.nan

        te_results[res]=dict(x=x_vals,mean=mean_vals,std=std_vals)

        if len(mean_vals)>0 and np.any(np.isfinite(mean_vals)):
            idx_max=int(np.nanargmax(mean_vals))
            sz_max_te[res]=float(x_vals[idx_max])

    fig,axes=plt.subplots(2,2,figsize=(12,8))
    ax_no_te,ax_wi_te=axes[0]
    ax_no_hess,ax_wi_hess=axes[1]

    map_te={"Without Resilience":ax_no_te,"With Resilience":ax_wi_te}
    map_hess={"Without Resilience":ax_no_hess,"With Resilience":ax_wi_hess}

    for res in resilience_levels:
        ax_te=map_te[res]
        ax_te.spines["top"].set_visible(False)
        ax_te.spines["right"].set_visible(False)

        if res not in te_results:
            ax_te.text(0.5,0.5,"No data",transform=ax_te.transAxes,ha="center",va="center")
            continue

        x=te_results[res]["x"]
        m=te_results[res]["mean"]
        sd=te_results[res]["std"]

        ax_te.errorbar(x,m,yerr=sd,fmt="none",ecolor="0.7",elinewidth=1,capsize=3,alpha=0.9)
        ax_te.scatter(x,m,c=x,cmap=cmap_colors[res],s=40,edgecolor="k",linewidth=0.4)
        ax_te.plot(x,m,color=base_colors[res],linewidth=1.5)

        if res in sz_max_te:
            x_max=sz_max_te[res]
            m_max=m[np.where(x==x_max)[0][0]]
            ax_te.scatter(x_max,m_max,s=100,facecolors="none",edgecolors="k",zorder=5)
            ax_te.axvline(x_max,color=base_colors[res],linestyle="--",linewidth=1.2,alpha=0.8)

        ax_te.set_title(f"TE MF→PC vs SZ\n{res}",fontweight="bold")
        ax_te.set_xlabel(x_col)
        ax_te.set_ylabel("TE MF→PC (Δ vs SZ=0) (Δ bit)" if use_delta_vs_sz0 else "TE MF→PC (Δ bit)")
        ax_te.axhline(0,color="k",linestyle="--",linewidth=1,alpha=0.8)
        ax_h=map_hess[res]
        ax_h.spines["top"].set_visible(False)
        ax_h.spines["right"].set_visible(False)

        if len(x)<3 or np.sum(np.isfinite(m))<3:
            ax_h.text(0.5,0.5,"Not enough points",transform=ax_h.transAxes,ha="center",va="center")
            continue

        m=np.asarray(m,dtype=float)
        sd=np.asarray(sd,dtype=float)
        N=len(m)

        h_mean=np.zeros(N,dtype=float)
        h_std=np.zeros(N,dtype=float)

        for i in range(1,N-1):
            h_mean[i]=m[i+1]-2*m[i]+m[i-1]
            var_i=sd[i+1]**2+(2*sd[i])**2+sd[i-1]**2
            h_std[i]=np.sqrt(var_i) if var_i>0 else 0.0

        ax_h.errorbar(x,h_mean,yerr=h_std,fmt="none",ecolor="0.7",elinewidth=1,capsize=3,alpha=0.9)
        ax_h.plot(x,h_mean,"-o",color=base_colors[res],linewidth=1.5,markersize=4)

        if N>2:
            idx_internal=np.arange(1,N-1)
            h_internal=h_mean[1:-1]
            if h_internal.size>0:
                if res=="Without Resilience":
                    n_min=min(3,h_internal.size)
                    base_sorted=np.argsort(h_internal)[:n_min]
                    idx_minima=idx_internal[base_sorted]
                else:
                    n_base=min(2,h_internal.size)
                    base_sorted=np.argsort(h_internal)[:n_base]
                    idx_base=idx_internal[base_sorted]
                    min_idx_left=idx_base.min()
                    left_mask=idx_internal<min_idx_left
                    if np.any(left_mask):
                        h_left=h_mean[idx_internal[left_mask]]
                        idx_left_rel=np.argsort(h_left)[0]
                        idx_left=idx_internal[left_mask][idx_left_rel]
                        idx_minima=np.concatenate([idx_base,[idx_left]])
                    else:
                        n_min=min(3,h_internal.size)
                        base_sorted_full=np.argsort(h_internal)[:n_min]
                        idx_minima=idx_internal[base_sorted_full]

                idx_minima=np.unique(idx_minima)
                if idx_minima.size>3:
                    vals=h_mean[idx_minima]
                    keep=np.argsort(vals)[:3]
                    idx_minima=idx_minima[keep]

                for idxm in idx_minima:
                    ax_h.scatter(x[idxm],h_mean[idxm],s=90,facecolors="none",edgecolors="k",linewidth=1.5,zorder=5)
                    ax_h.axvline(x[idxm],color=base_colors[res],linestyle=":",linewidth=1.2,alpha=0.8)
                    ax_te.axvline(x[idxm],color=base_colors[res],linestyle=":",linewidth=1.2,alpha=0.8)
                    ax_te.scatter(x[idxm],m[idxm],s=90,facecolors="none",edgecolors="k",linewidth=1.5,zorder=5)

        ax_h.axhline(0,color="k",linestyle="--",linewidth=1,alpha=0.8)
        ax_h.set_title(f"Discrete Hessian of TE\n{res}",fontweight="bold")
        ax_h.set_xlabel(x_col)
        ax_h.set_ylabel(r"$d^2\,\mathrm{TE}/d\,\mathrm{SZ}^2$" + (" (Δ bit)" if use_delta_vs_sz0 else " (bit)"))

        if res in sz_max_te:
            x_max=sz_max_te[res]
            ax_h.axvline(x_max,color=base_colors[res],linestyle="--",linewidth=1.2,alpha=0.8)

    labels={ax_no_te:"a",ax_wi_te:"b",ax_no_hess:"c",ax_wi_hess:"d"}
    for ax,lab in labels.items():
        ax.text(-0.10,1.10,lab,transform=ax.transAxes,fontsize=14,fontweight="bold")

    plt.tight_layout()
    out_path=os.path.join(outdir,fname)
    plt.savefig(_prefix_output(out_path),dpi=300,bbox_inches="tight")
    print("✅ Salvato:",out_path)
    plt.show()


In [ ]:

plot_te_sz_and_hessian_norm(
        df_all,
        sigma_ranges,
        x_col="SZ Factor (%)",
        te_col="te_mossy_fibers_to_purk",
        outdir="./tempPlots",
        fname="te_hessian_4panels.png",)

In [ ]:
df_all.keys()

In [ ]:
import pickle

df_all_str = pickle.load(open("tempPlots/basal_df_all_str.pkl", "rb"))
df_all_weight = pickle.load(open("tempPlots/basal_df_all_weights.pkl", "rb"))

df_all_fr = df_all

In [ ]:
df_all_fr.keys()

In [ ]:
import numpy as np
import pandas as pd

def extract_TE_and_atrophy(df,sigma_ranges,
                           x_col="SZ Factor (%)",
                           res_col="resilience",
                           te_col="te_mossy_fibers_to_purk"):
    rows=[]
    for res in ["Without Resilience","With Resilience"]:
        if res not in sigma_ranges:
            continue
        smin=sigma_ranges[res]["sigma_min"]
        smax=sigma_ranges[res]["sigma_max"]
        sub=df[(df[res_col]==res)&(df["sigma"]>=smin)&(df["sigma"]<=smax)].copy()
        if sub.empty:
            continue
        grp=sub.groupby(x_col)[te_col]
        for sz,vals in grp:
            v=vals.astype(float).to_numpy()
            if v.size==0:
                continue
            rows.append({
                "Resilience":res,
                x_col:float(sz),
                "te_mean":float(np.mean(v)),
                "te_std":float(np.std(v,ddof=1)) if v.size>1 else 0.0,
                "n":int(v.size)
            })
    return pd.DataFrame(rows).sort_values(["Resilience",x_col])

# 1) TE media across sigma (df_all ha 'resilience' e 'sigma')
df_te=extract_TE_and_atrophy(df_all,sigma_ranges,
                             x_col="SZ Factor (%)",
                             res_col="resilience",
                             te_col="te_mossy_fibers_to_purk")


df_te

numeric_cols = df_all_str.select_dtypes("number").columns

df_collapsed = (
    df_all_str
    .groupby(["SZ Factor (%)", "Resilience"], as_index=False)[numeric_cols]
    .mean()
)
df_collapsed.shape

# --- Merge tra TE media (df_te) e struttura mediata (df_collapsed) ---
df_merged = df_collapsed.merge(
    df_te[["SZ Factor (%)", "Resilience", "te_mean", "te_std", "n"]],
    on=["SZ Factor (%)", "Resilience"],
    how="right"
)

df_merged[['te_mean', 'Resilience']]


In [ ]:
df_merged.keys()

In [ ]:
df_merged[["total_cells", "total_synapses", "Resilience", "te_mean"]].info()


In [ ]:
df_all_str.groupby(["SZ Factor (%)","dfType"])["synapses"].mean().shape


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def norm_res(val):
    s = str(val).strip()
    if s in ["15", "With Resilience"]:
        return "With Resilience"
    if s in ["2", "Without Resilience"]:
        return "Without Resilience"
    return s

def scatter_exc_inh_colorTE(
        df_all_str,
        df_te,
        measure_col="effective_convergence",
        outdir="./tempPlots",
        fname="scatter_exc_inh_colorTE.png"
    ):
    """
    Usa TUTTE le connessioni.
    Raggruppa eccitatorie/inibitorie per SZ × Resilience.
    Colore = TE_mean.
    Marker = Resilience.
    """
    os.makedirs(outdir, exist_ok=True)

    # --- copia e normalizza ---
    df = df_all_str.copy()
    df["Resilience"]    = df["Resilience"].apply(norm_res)
    df["SZ Factor (%)"] = df["SZ Factor (%)"].astype(float)

    df_te2 = df_te.copy()
    df_te2["Resilience"]    = df_te2["Resilience"].apply(norm_res)
    df_te2["SZ Factor (%)"] = df_te2["SZ Factor (%)"].astype(float)

    # --- check colonne ---
    needed = [measure_col, "SZ Factor (%)", "Resilience", "synType"]
    for c in needed:
        if c not in df.columns:
            raise ValueError(f"Column '{c}' not found in df_all_str")

    # ---- SUM UNIQUE ----
    def sum_unique(x):
        return np.unique(x).sum()

    exc_sum = (
        df[df["synType"] == "excitatory (glutamate)"]
        .groupby(["SZ Factor (%)", "Resilience"], as_index=False)
        .agg({measure_col: sum_unique})
        .rename(columns={measure_col: "exc_val"})
    )

    inh_sum = (
        df[df["synType"] == "inhibitory (GABA)"]
        .groupby(["SZ Factor (%)", "Resilience"], as_index=False)
        .agg({measure_col: sum_unique})
        .rename(columns={measure_col: "inh_val"})
    )

    # --- merge E/I ---
    df_EI = exc_sum.merge(inh_sum,
                          on=["SZ Factor (%)", "Resilience"],
                          how="outer").fillna(0)

    # --- MERGE con TE_mean ---
    df_EI = df_EI.merge(
        df_te2[["SZ Factor (%)", "Resilience", "te_mean"]],
        on=["SZ Factor (%)", "Resilience"],
        how="left"
    )

    # Se ci sono NaN in te_mean li metto a 0 o esclusi
    df_EI["te_mean"] = df_EI["te_mean"].fillna(0)

    # Marker mapping
    marker_map = {
        "With Resilience": "o",
        "Without Resilience": "s"
    }

    # ----------- PLOT -----------
    plt.figure(figsize=(10, 8))

    for res in df_EI["Resilience"].unique():
        sub = df_EI[df_EI["Resilience"] == res]

        plt.scatter(
            sub["exc_val"],
            sub["inh_val"],
            c=sub["te_mean"],
            cmap="coolwarm",
            s=150,
            edgecolor="black",
            linewidth=0.7,
            marker=marker_map.get(res, "o"),
            label=res,
            alpha=0.9
        )

    plt.xlabel(f"Excitatory sum ({measure_col})")
    plt.ylabel(f"Inhibitory sum ({measure_col})")
    plt.title("Exc vs Inh – colored by TE_mean")

    # Colorbar con TE
    cbar = plt.colorbar()
    cbar.set_label("TE_mean (bit)")

    plt.legend(title="Resilience")
    plt.tight_layout()

    outpath = os.path.join(outdir, fname)
    plt.savefig(_prefix_output(outpath), dpi=300)
    plt.show()
    print("Saved:", outpath)

    return df_EI



scatter_exc_inh_colorTE(
    df_all_str,
    df_te,
    measure_col="synapses",
    fname="scatter_TE_synapses.png"
)

scatter_exc_inh_colorTE(
    df_all_str,
    df_te,
    measure_col="effective_synapses",
    fname="scatter_TE_effectiveSyn.png"
)

scatter_exc_inh_colorTE(
    df_all_str,
    df_te,
    measure_col="convergence",
    fname="scatter_TE_convergence.png"
)


In [ ]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

def norm_res(val):
    s = str(val).strip()
    if s in ["15", "With Resilience"]:
        return "With Resilience"
    if s in ["2", "Without Resilience"]:
        return "Without Resilience"
    return s

def scatter_TE_vs_EIratio_weights_coloredSZ(
    df_all_str,
    df_te,
    outdir="./tempPlots",
    fname="scatter_TE_vs_EIratio_weights_coloredSZ.png"
):
    os.makedirs(outdir, exist_ok=True)

    df = df_all_str.copy()
    df["Resilience"] = df["Resilience"].apply(norm_res)
    df["SZ Factor (%)"] = df["SZ Factor (%)"].astype(float).round(2)

    df_te2 = df_te.copy()
    df_te2["Resilience"] = df_te2["Resilience"].apply(norm_res)
    df_te2["SZ Factor (%)"] = df_te2["SZ Factor (%)"].astype(float).round(2)

    df_te_agg = (
        df_te2.groupby(["SZ Factor (%)", "Resilience"], as_index=False)["te_mean"]
        .mean()
    )

    exc = (
        df[df["synType"] == "excitatory (glutamate)"]
        .groupby(["SZ Factor (%)", "Resilience"], as_index=False)["weight"]
        .sum()
        .rename(columns={"weight": "w_exc"})
    )

    inh = (
        df[df["synType"] == "inhibitory (GABA)"]
        .groupby(["SZ Factor (%)", "Resilience"], as_index=False)["weight"]
        .sum()
        .rename(columns={"weight": "w_inh"})
    )

    df_EI = exc.merge(
        inh,
        on=["SZ Factor (%)", "Resilience"],
        how="outer"
    ).fillna(0)

    df_merge = df_te_agg.merge(
        df_EI,
        on=["SZ Factor (%)", "Resilience"],
        how="left"
    )

    df_merge["w_exc"] = df_merge["w_exc"].fillna(0)
    df_merge["w_inh"] = df_merge["w_inh"].fillna(0)
    df_merge["EI_ratio"] = df_merge["w_exc"] / df_merge["w_inh"].replace(0, np.nan)

    cmap = {
        "With Resilience": plt.get_cmap("Reds"),
        "Without Resilience": plt.get_cmap("Blues")
    }

    sz_min = df_merge["SZ Factor (%)"].min()
    sz_max = df_merge["SZ Factor (%)"].max()
    norm = Normalize(vmin=sz_min, vmax=sz_max)

    plt.figure(figsize=(8, 6))

    for res in ["With Resilience", "Without Resilience"]:
        sub = df_merge[df_merge["Resilience"] == res]
        if sub.empty:
            continue

        colors = cmap[res](norm(sub["SZ Factor (%)"].values))

        plt.scatter(
            sub["te_mean"],
            sub["EI_ratio"],
            c=colors,
            s=150,
            edgecolor="black",
            linewidth=0.7,
            marker="o",
            label=res,
            alpha=0.9
        )

    plt.xlabel("TE_mean (bit)")
    plt.ylabel("Σ weights Exc / Σ weights Inh")
    plt.title("E/I weight ratio vs TE_mean\n(tinta = SZ Factor, colore = Resilience)")
    plt.legend(title="Resilience", loc="best")

    plt.tight_layout()
    outpath = os.path.join(outdir, fname)
    plt.savefig(_prefix_output(outpath), dpi=300)
    plt.show()
    print("Saved:", outpath)

    return df_merge

df_merge = scatter_TE_vs_EIratio_weights_coloredSZ(df_all_str, df_te)


In [ ]:
df_merge

# figure 2 bis

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind, spearmanr
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D

def plot_lag_te_and_epoched_5x2(
    df,
    sigma_ranges,
    x_col="SZ Factor (%)",
    outdir="./tempPlots",
    fname_abf="lag_te_epoched_abf.png",
    fname_ghijk="lag_te_epoched_ghijk.png",
    use_delta=False,
    alpha=0.05,
    dt=0.1,
    window_ms=(-100, 65),
    df_all_str=None,
    df_te=None,
    norm_res=None
):
    os.makedirs(outdir, exist_ok=True)

    if "purk_slope" not in df.columns and "fooof_purk" in df.columns:
        def _get_offset(d):
            if isinstance(d, dict) and "aperiodic_params" in d:
                ap = d["aperiodic_params"]
                if isinstance(ap, (list, tuple)) and len(ap) >= 2:
                    return float(ap[0])
            return np.nan
        def _get_slope(d):
            if isinstance(d, dict) and "aperiodic_params" in d:
                ap = d["aperiodic_params"]

                if isinstance(ap, (list, tuple)) and len(ap) >= 2:
                    return float(ap[1])
            return np.nan
        df = df.copy()
        df["purk_offset"] = df["fooof_purk"].apply(_get_offset)
        df["purk_slope"]  = df["fooof_purk"].apply(_get_slope)

    metrics = [
        ("optimal_lag_ms",          "Lag MF→PC (ms)"),
        ("te_mossy_fibers_to_purk", "TE MF→PC (bit)")
    ]
    resilience_levels = ["Without Resilience", "With Resilience"]
    base_colors = {"Without Resilience": "blue", "With Resilience": "red"}

    legend_handles = [
        Line2D([0],[0], color=base_colors["Without Resilience"], marker="o", linestyle="-", label="Without Resilience"),
        Line2D([0],[0], color=base_colors["With Resilience"],    marker="o", linestyle="-", label="With Resilience"),
    ]

    metric_results = {col: {res: {} for res in resilience_levels} for col, _ in metrics}
    lag_stats, te_stats = {}, {}

    for col, _ in metrics:
        if col not in df.columns:
            continue
        for res in resilience_levels:
            rng = sigma_ranges.get(res, None)
            if rng is None:
                continue
            smin, smax = rng["sigma_min"], rng["sigma_max"]
            sub = df[(df["resilience"] == res) &
                     (df["sigma"] >= smin) &
                     (df["sigma"] <= smax)].copy()
            if sub.empty:
                continue
            metric_results[col][res]["sub"] = sub
            grp = sub.groupby(x_col)[col]
            x_vals = np.array(sorted(grp.groups.keys()), dtype=float)
            mean_vals = grp.mean().reindex(x_vals).to_numpy(dtype=float)
            std_vals  = grp.std(ddof=1).reindex(x_vals).to_numpy(dtype=float)
            if use_delta:
                idx0 = np.where(x_vals == 0)[0][0] if np.any(x_vals == 0) else 0
                baseline = mean_vals[idx0]
                mean_vals = mean_vals - baseline
            metric_results[col][res]["x"] = x_vals
            metric_results[col][res]["mean"] = mean_vals
            metric_results[col][res]["std"] = std_vals

    win_pre  = int(abs(window_ms[0]) / dt)
    win_post = int(window_ms[1] / dt)
    time_axis = np.arange(-win_pre, win_post) * dt

    def _get_ylims(trace_column):
        if "mossy" in trace_column:
            return [-10, 200]
        if "purkinje" in trace_column:
            return [35, 110]
        return [0, 1]

    def _get_sigma_range_df(df_main, sigma_ranges, res):
        rng = sigma_ranges.get(res, None)
        if rng is None:
            return None, None, None
        smin, smax = rng["sigma_min"], rng["sigma_max"]
        df_sub = df_main[(df_main["resilience"] == res) &
                         (df_main["sigma"] >= smin) &
                         (df_main["sigma"] <= smax)].copy()
        if df_sub.empty:
            return None, None, None
        return df_sub, smin, smax

    def _make_sigma_averaged_df(df_sub, trace_columns, onset_column="burst_onsets_ms", group_column="atrophy_percent"):
        if df_sub is None or df_sub.empty:
            return None
        rows = []
        for grp_val, g in df_sub.groupby(group_column):
            row = {group_column: grp_val}
            if "resilience" in g.columns:
                row["resilience"] = g["resilience"].iloc[0]
            row[onset_column] = g[onset_column].iloc[0]
            for trace_col in trace_columns:
                traces = [np.asarray(v, dtype=float) for v in g[trace_col]]
                if not traces:
                    row[trace_col] = np.array([])
                    continue
                L = min(t.shape[0] for t in traces)
                if L == 0:
                    row[trace_col] = np.array([])
                    continue
                stack = np.vstack([t[:L] for t in traces])
                row[trace_col] = stack.mean(axis=0)
            rows.append(row)
        return pd.DataFrame(rows) if rows else None

    def _compute_peak_vs_sz(df_sub, trace_column, peak_window_ms=(0, 75)):
        if df_sub is None or df_sub.empty:
            return (np.array([]), np.array([]), np.array([]))
        rows = []
        for _, row in df_sub.iterrows():
            trace = np.array(row[trace_column], dtype=float)
            onsets = row["burst_onsets_ms"]
            sz = float(row[x_col])
            single_peaks = []
            for onset in onsets:
                center = int(onset / dt)
                start = center + int(peak_window_ms[0] / dt)
                end   = center + int(peak_window_ms[1] / dt)
                if start < 0 or end > len(trace):
                    continue
                seg = trace[start:end]
                if seg.size == 0:
                    continue
                single_peaks.append(float(np.max(seg)))
            if single_peaks:
                rows.append({"SZ": sz, "peak": float(np.mean(single_peaks))})
        if not rows:
            return (np.array([]), np.array([]), np.array([]))
        tmp = pd.DataFrame(rows)
        sz_vals, peak_mean, peak_std = [], [], []
        for sz in sorted(tmp["SZ"].unique()):
            vals = tmp.loc[tmp["SZ"] == sz, "peak"].to_numpy(dtype=float)
            sz_vals.append(sz)
            peak_mean.append(np.mean(vals))
            peak_std.append(np.std(vals, ddof=1) if vals.size > 1 else 0.0)
        return (np.array(sz_vals, dtype=float),
                np.array(peak_mean, dtype=float),
                np.array(peak_std, dtype=float))

    def _compute_te_and_peak(df_sub, trace_column, peak_window_ms=(0, 75)):
        if df_sub is None or df_sub.empty:
            return (np.array([]), np.array([]), np.array([]))
        te_vals, peak_vals, sz_vals = [], [], []
        for _, row in df_sub.iterrows():
            trace = np.array(row[trace_column], dtype=float)
            onsets = row["burst_onsets_ms"]
            sz = float(row[x_col])
            te = float(row["te_mossy_fibers_to_purk"])
            single_peaks = []
            for onset in onsets:
                center = int(onset / dt)
                start = center + int(peak_window_ms[0] / dt)
                end   = center + int(peak_window_ms[1] / dt)
                if start < 0 or end > len(trace):
                    continue
                seg = trace[start:end]
                if seg.size == 0:
                    continue
                single_peaks.append(float(np.max(seg)))
            if single_peaks:
                te_vals.append(te)
                peak_vals.append(np.mean(single_peaks))
                sz_vals.append(sz)
        return (np.array(te_vals, dtype=float),
                np.array(peak_vals, dtype=float),
                np.array(sz_vals, dtype=float))

    def _compute_te_and_slope(df_sub):
        if df_sub is None or df_sub.empty or "purk_slope" not in df_sub.columns:
            return (np.array([]), np.array([]), np.array([]))
        te_vals = df_sub["te_mossy_fibers_to_purk"].astype(float).to_numpy()
        slope_vals = df_sub["purk_slope"].astype(float).to_numpy()
        sz_vals = df_sub[x_col].astype(float).to_numpy()
        return te_vals, slope_vals, sz_vals

    df_no_range, _, _ = _get_sigma_range_df(df, sigma_ranges, "Without Resilience")
    df_wi_range, _, _ = _get_sigma_range_df(df, sigma_ranges, "With Resilience")

    df_no_avg = _make_sigma_averaged_df(
        df_no_range,
        trace_columns=["mossy_fibers_timeseries", "purkinje_timeseries"],
        onset_column="burst_onsets_ms",
        group_column="atrophy_percent"
    )
    df_wi_avg = _make_sigma_averaged_df(
        df_wi_range,
        trace_columns=["mossy_fibers_timeseries", "purkinje_timeseries"],
        onset_column="burst_onsets_ms",
        group_column="atrophy_percent"
    )

    fig1, axes1 = plt.subplots(3, 2, figsize=(12, 14))
    ax_lag      = axes1[0, 0]
    ax_te       = axes1[0, 1]
    ax_mf_nores = axes1[1, 0]
    ax_pc_nores = axes1[1, 1]
    ax_mf_with  = axes1[2, 0]
    ax_pc_with  = axes1[2, 1]
    fig1.legend(handles=legend_handles, loc="upper center", bbox_to_anchor=(0.5, 1.02), ncol=2, frameon=False)

    for ax in (ax_lag, ax_te):
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    col = metrics[0][0]
    for res in resilience_levels:
        res_dict = metric_results[col].get(res, {})
        if "x" not in res_dict:
            continue
        x_vals, mean_vals, std_vals = res_dict["x"], res_dict["mean"], res_dict["std"]
        ax_lag.errorbar(x_vals, mean_vals, yerr=std_vals, fmt="none", ecolor="k", elinewidth=1, capsize=3, alpha=0.25, zorder=1)
        ax_lag.scatter(x_vals, mean_vals, color=base_colors[res], s=60, edgecolor="k", linewidth=0.4, zorder=3)
        ax_lag.plot(x_vals, mean_vals, color=base_colors[res], alpha=0.7, linewidth=1.5, zorder=2)
        if len(mean_vals) > 0:
            idx = int(np.argmax(mean_vals))
            lag_stats[res] = (float(mean_vals[idx]), float(x_vals[idx]))

    resA, resB = resilience_levels
    if ("sub" in metric_results[col].get(resA, {}) and "sub" in metric_results[col].get(resB, {})):
        subA = metric_results[col][resA]["sub"]
        subB = metric_results[col][resB]["sub"]
        common_x = sorted(set(subA[x_col].astype(float).unique()).intersection(set(subB[x_col].astype(float).unique())))
        pvals = {}
        for x0 in common_x:
            valsA = subA.loc[subA[x_col] == x0, col].to_numpy(dtype=float)
            valsB = subB.loc[subB[x_col] == x0, col].to_numpy(dtype=float)
            if len(valsA) >= 2 and len(valsB) >= 2:
                _, p = ttest_ind(valsA, valsB, equal_var=False, nan_policy="omit")
                pvals[x0] = p
        if pvals:
            xs_sorted = np.array(sorted(pvals.keys()), dtype=float)
            dx = np.min(np.diff(xs_sorted)) if len(xs_sorted) > 1 else 1.0
            for x0, p in pvals.items():
                if p < alpha:
                    ax_lag.axvspan(x0 - dx*0.5, x0 + dx*0.5, color="#fff7b2", alpha=0.0, zorder=-1)

    ax_lag.axhline(0, color="k", linestyle="--", linewidth=1, alpha=0.8)
    ax_lag.set_ylabel("ms" + (" (Δ vs SZ=0)" if use_delta else ""))
    ax_lag.set_xlabel(x_col)
    ax_lag.grid(False)
    ax_lag.set_title("MF→PC lag (Saccadic dysmetria)", fontweight="bold", pad=34)
    y_text = 1.05
    if "Without Resilience" in lag_stats:
        lag_val, sz_val = lag_stats["Without Resilience"]
        ax_lag.text(0.5, y_text, f"Without Resilience: lag={int(round(lag_val))} ms at SZ={int(round(sz_val))}%", transform=ax_lag.transAxes, fontsize=12, color="blue", va="bottom", ha="center")
        y_text -= 0.07
    if "With Resilience" in lag_stats:
        lag_val, sz_val = lag_stats["With Resilience"]
        ax_lag.text(0.5, y_text, f"With Resilience: lag={int(round(lag_val))} ms at SZ={int(round(sz_val))}%", transform=ax_lag.transAxes, fontsize=12, color="red", va="bottom", ha="center")

    col = metrics[1][0]
    for res in resilience_levels:
        res_dict = metric_results[col].get(res, {})
        if "x" not in res_dict:
            continue
        x_vals, mean_vals, std_vals = res_dict["x"], res_dict["mean"], res_dict["std"]
        ax_te.errorbar(x_vals, mean_vals, yerr=std_vals, fmt="none", ecolor="k", elinewidth=1, capsize=3, alpha=0.25, zorder=1)
        ax_te.scatter(x_vals, mean_vals, color=base_colors[res], s=60, edgecolor="k", linewidth=0.4, zorder=3)
        ax_te.plot(x_vals, mean_vals, color=base_colors[res], alpha=0.7, linewidth=1.5, zorder=2)
        if len(mean_vals) > 0:
            idx = int(np.argmax(mean_vals))
            te_stats[res] = (float(mean_vals[idx]), float(x_vals[idx]))

    if ("sub" in metric_results[col].get(resA, {}) and "sub" in metric_results[col].get(resB, {})):
        subA = metric_results[col][resA]["sub"]
        subB = metric_results[col][resB]["sub"]
        common_x = sorted(set(subA[x_col].astype(float).unique()).intersection(set(subB[x_col].astype(float).unique())))
        pvals = {}
        for x0 in common_x:
            valsA = subA.loc[subA[x_col] == x0, col].to_numpy(dtype=float)
            valsB = subB.loc[subB[x_col] == x0, col].to_numpy(dtype=float)
            if len(valsA) >= 2 and len(valsB) >= 2:
                _, p = ttest_ind(valsA, valsB, equal_var=False, nan_policy="omit")
                pvals[x0] = p
        if pvals:
            xs_sorted = np.array(sorted(pvals.keys()), dtype=float)
            dx = np.min(np.diff(xs_sorted)) if len(xs_sorted) > 1 else 1.0
            for x0, p in pvals.items():
                if p < alpha:
                    ax_te.axvspan(x0 - dx*0.5, x0 + dx*0.5, color="#fff7b2", alpha=0.0, zorder=-1)

    ax_te.axhline(0, color="k", linestyle="--", linewidth=1, alpha=0.8)
    ax_te.set_ylabel("bit" + (" (Δ vs SZ=0)" if use_delta else ""))
    ax_te.set_xlabel(x_col)
    ax_te.grid(False)
    ax_te.set_title("MF→PC transfer entropy (TE)", fontweight="bold", pad=34)
    y_text = 1.05
    if "Without Resilience" in te_stats:
        te_val, sz_val = te_stats["Without Resilience"]
        ax_te.text(0.5, y_text, f"Without Resilience: max TE={te_val:.3f} bit at SZ={int(round(sz_val))}%", transform=ax_te.transAxes, fontsize=12, color="blue", va="bottom", ha="center")
        y_text -= 0.07
    if "With Resilience" in te_stats:
        te_val, sz_val = te_stats["With Resilience"]
        ax_te.text(0.5, y_text, f"With Resilience: max TE={te_val:.3f} bit at SZ={int(round(sz_val))}%", transform=ax_te.transAxes, fontsize=12, color="red", va="bottom", ha="center")

    if df_no_avg is not None:
        plot_epoched_firing_rate(df=df_no_avg, trace_column="mossy_fibers_timeseries", time_axis=time_axis, pop_name="Mossy Fibers (input) \n Without Resilience", stim_column=None, onset_column="burst_onsets_ms", group_column="atrophy_percent", cmap="Blues_r", ylims=_get_ylims("mossy_fibers_timeseries"), operator=None, window_ms=window_ms, dt=dt, save_path=None, ax=ax_mf_nores)
        plot_epoched_firing_rate(df=df_no_avg, trace_column="purkinje_timeseries", time_axis=time_axis, pop_name="Purkinje Cells (output) \n Without Resilience", stim_column=None, onset_column="burst_onsets_ms", group_column="atrophy_percent", cmap="Blues_r", ylims=_get_ylims("purkinje_timeseries"), operator=None, window_ms=window_ms, dt=dt, save_path=None, ax=ax_pc_nores)

        INSET_TITLE_FS = 10  # era 8
        INSET_LABEL_FS = 9   # era 7
        INSET_TICK_FS  = 11  # era 9


        
        inset_mf_no = ax_mf_nores.inset_axes([0.595, 0.40, 0.35, 0.35])
        sz_mf, peak_mf_mean, peak_mf_std = _compute_peak_vs_sz(df_no_range, "mossy_fibers_timeseries")
        if len(sz_mf) > 0:
            baseline_mf = np.mean(peak_mf_mean[sz_mf == 0]) if np.any(sz_mf == 0) else peak_mf_mean[np.argmin(sz_mf)]
            peak_mf_delta = peak_mf_mean - baseline_mf
            inset_mf_no.errorbar(sz_mf, peak_mf_delta, yerr=peak_mf_std, fmt="none", ecolor="k", elinewidth=1, capsize=2, alpha=0.25)
            inset_mf_no.scatter(sz_mf, peak_mf_delta, c=sz_mf, cmap="Blues_r", s=15, edgecolor="k", linewidth=0.3)
            rho, pval = spearmanr(sz_mf, peak_mf_delta)
            inset_mf_no.set_title(f"ρ={rho:.2f}, p={pval:.2e}", fontsize=INSET_TITLE_FS)
            inset_mf_no.set_xlabel("SZ Factor (%)", fontsize=INSET_LABEL_FS)
            inset_mf_no.set_ylabel("Δ Peak FR (Hz)", fontsize=INSET_LABEL_FS)
            inset_mf_no.tick_params(labelsize=INSET_TICK_FS)

            inset_mf_no.set_ylim(-30, 30)
            inset_mf_no.axhline(0, color="k", linestyle="--", linewidth=0.8, alpha=0.8)
        else:
            inset_mf_no.set_visible(False)

        inset_pc_no = ax_pc_nores.inset_axes([0.595, 0.40, 0.35, 0.35])
        sz_pc, peak_pc_mean, peak_pc_std = _compute_peak_vs_sz(df_no_range, "purkinje_timeseries")
        if len(sz_pc) > 0:
            baseline_pc = np.mean(peak_pc_mean[sz_pc == 0]) if np.any(sz_pc == 0) else peak_pc_mean[np.argmin(sz_pc)]
            peak_pc_delta = peak_pc_mean - baseline_pc
            inset_pc_no.errorbar(sz_pc, peak_pc_delta, yerr=peak_pc_std, fmt="none", ecolor="k", elinewidth=1, capsize=2, alpha=0.25)
            inset_pc_no.scatter(sz_pc, peak_pc_delta, c=sz_pc, cmap="Blues_r", s=30, edgecolor="k", linewidth=0.3)
            rho, pval = spearmanr(sz_pc, peak_pc_delta)
            inset_pc_no.set_title(f"ρ={rho:.2f}, p={pval:.2e}", fontsize=INSET_TITLE_FS)
            inset_pc_no.set_xlabel("SZ Factor (%)", fontsize=INSET_LABEL_FS)
            inset_pc_no.set_ylabel("Δ Peak FR (Hz)", fontsize=INSET_LABEL_FS)
            inset_pc_no.tick_params(labelsize=INSET_TICK_FS)
            inset_pc_no.set_ylim(-30, 30)
            inset_pc_no.axhline(0, color="k", linestyle="--", linewidth=0.8, alpha=0.8)
        else:
            inset_pc_no.set_visible(False)
    else:
        ax_mf_nores.set_title("Mossy Fibers – Without Resilience (no σ)")
        ax_pc_nores.set_title("Purkinje – Without Resilience (no σ)")

    if df_wi_avg is not None:
        plot_epoched_firing_rate(df=df_wi_avg, trace_column="mossy_fibers_timeseries", time_axis=time_axis, pop_name="Mossy Fibers (input) \n With Resilience", stim_column=None, onset_column="burst_onsets_ms", group_column="atrophy_percent", cmap="Reds_r", ylims=_get_ylims("mossy_fibers_timeseries"), operator=None, window_ms=window_ms, dt=dt, save_path=None, ax=ax_mf_with)
        plot_epoched_firing_rate(df=df_wi_avg, trace_column="purkinje_timeseries", time_axis=time_axis, pop_name="Purkinje Cells (output) \n With Resilience", stim_column=None, onset_column="burst_onsets_ms", group_column="atrophy_percent", cmap="Reds_r", ylims=_get_ylims("purkinje_timeseries"), operator=None, window_ms=window_ms, dt=dt, save_path=None, ax=ax_pc_with)

        inset_mf_wi = ax_mf_with.inset_axes([0.595, 0.40, 0.35, 0.35])
        sz_mf, peak_mf_mean, peak_mf_std = _compute_peak_vs_sz(df_wi_range, "mossy_fibers_timeseries")
        if len(sz_mf) > 0:
            baseline_mf = np.mean(peak_mf_mean[sz_mf == 0]) if np.any(sz_mf == 0) else peak_mf_mean[np.argmin(sz_mf)]
            peak_mf_delta = peak_mf_mean - baseline_mf
            inset_mf_wi.errorbar(sz_mf, peak_mf_delta, yerr=peak_mf_std, fmt="none", ecolor="k", elinewidth=1, capsize=2, alpha=0.25)
            inset_mf_wi.scatter(sz_mf, peak_mf_delta, c=sz_mf, cmap="Reds_r", s=30, edgecolor="k", linewidth=0.3)
            rho, pval = spearmanr(sz_mf, peak_mf_delta)
            inset_mf_wi.set_title(f"ρ={rho:.2f}, p={pval:.2e}", fontsize=INSET_TITLE_FS)
            inset_mf_wi.set_xlabel("SZ Factor (%)", fontsize=INSET_LABEL_FS)
            inset_mf_wi.set_ylabel("Δ Peak FR (Hz)", fontsize=INSET_LABEL_FS)
            inset_mf_wi.tick_params(labelsize=INSET_TICK_FS)
            inset_mf_wi.set_ylim(-30, 30)
            inset_mf_wi.axhline(0, color="k", linestyle="--", linewidth=0.8, alpha=0.8)
        else:
            inset_mf_wi.set_visible(False)

        inset_pc_wi = ax_pc_with.inset_axes([0.595, 0.40, 0.35, 0.35])
        sz_pc, peak_pc_mean, peak_pc_std = _compute_peak_vs_sz(df_wi_range, "purkinje_timeseries")
        if len(sz_pc) > 0:
            baseline_pc = np.mean(peak_pc_mean[sz_pc == 0]) if np.any(sz_pc == 0) else peak_pc_mean[np.argmin(sz_pc)]
            peak_pc_delta = peak_pc_mean - baseline_pc
            inset_pc_wi.errorbar(sz_pc, peak_pc_delta, yerr=peak_pc_std, fmt="none", ecolor="k", elinewidth=1, capsize=2, alpha=0.25)
            inset_pc_wi.scatter(sz_pc, peak_pc_delta, c=sz_pc, cmap="Reds_r", s=30, edgecolor="k", linewidth=0.3)
            rho, pval = spearmanr(sz_pc, peak_pc_delta)
            inset_pc_wi.set_title(f"ρ={rho:.2f}, p={pval:.2e}", fontsize=INSET_TITLE_FS)
            inset_pc_wi.set_xlabel("SZ Factor (%)", fontsize=INSET_LABEL_FS)
            inset_pc_wi.set_ylabel("Δ Peak FR (Hz)", fontsize=INSET_LABEL_FS)
            inset_pc_wi.tick_params(labelsize=INSET_TICK_FS)
            inset_pc_wi.set_ylim(-30, 30)
            inset_pc_wi.axhline(0, color="k", linestyle="--", linewidth=0.8, alpha=0.8)
        else:
            inset_pc_wi.set_visible(False)
    else:
        ax_mf_with.set_title("Mossy Fibers – With Resilience (no σ)")
        ax_pc_with.set_title("Purkinje – With Resilience (no σ)")

    panel_labels_1 = {ax_lag:"a", ax_te:"b", ax_mf_nores:"c", ax_pc_nores:"d", ax_mf_with:"e", ax_pc_with:"f"}
    for ax, lab in panel_labels_1.items():
        ax.text(-0.10, 1.15, lab, transform=ax.transAxes, fontsize=14, fontweight="bold", va="top", ha="left")

    fig1.subplots_adjust(left=0.07, right=0.98, bottom=0.06, top=0.93, hspace=0.55, wspace=0.35)
    out1 = os.path.join(outdir, fname_abf)
    fig1.savefig(_prefix_output(out1), dpi=600, bbox_inches="tight")
    print("✅ Salvato:", out1)
    plt.close(fig1)

    fig2, axes2 = plt.subplots(2, 2, figsize=(12, 12))
    ax_te_pc    = axes2[0, 0]
    ax_te_slope = axes2[0, 1]
    #ax_struct   = axes2[1, 0]
    ax_te = axes2[1,1]
    #axes2[1, 1].axis("off")

    for ax in (ax_te_pc, ax_te_slope, ax_te):
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    stats_pc = {}
    for res, cmap in [("Without Resilience","Blues_r"), ("With Resilience","Reds_r")]:
        rng = sigma_ranges.get(res, None)
        if rng is None:
            continue
        smin, smax = rng["sigma_min"], rng["sigma_max"]
        df_range = df[(df["resilience"] == res) & (df["sigma"] >= smin) & (df["sigma"] <= smax)].copy()
        if df_range.empty:
            continue
        te_vals, peak_vals, sz_vals = _compute_te_and_peak(df_range, "purkinje_timeseries")
        if len(peak_vals) == 0:
            continue
        if np.any(sz_vals == 0):
            baseline_peak = np.mean(peak_vals[sz_vals == 0])
            baseline_te   = np.mean(te_vals[sz_vals == 0])
        else:
            idx0 = np.argmin(sz_vals)
            baseline_peak = peak_vals[idx0]
            baseline_te   = te_vals[idx0]
        peak_delta = peak_vals - baseline_peak
        te_delta   = te_vals   - baseline_te
        rho, pval = spearmanr(te_delta, peak_delta) if len(te_delta) > 1 else (np.nan, np.nan)
        stats_pc[res] = (rho, pval)

        sz_uniq = np.sort(np.unique(sz_vals))
        means_te, means_peak, std_te, std_peak, sz_for_color = [], [], [], [], []
        for sz0 in sz_uniq:
            m = (sz_vals == sz0)
            if np.sum(m) == 0:
                continue
            means_te.append(np.mean(te_delta[m]))
            means_peak.append(np.mean(peak_delta[m]))
            std_te.append(np.std(te_delta[m], ddof=1) if np.sum(m) > 1 else 0.0)
            std_peak.append(np.std(peak_delta[m], ddof=1) if np.sum(m) > 1 else 0.0)
            sz_for_color.append(sz0)

        if not means_te:
            continue

        means_te = np.array(means_te); means_peak = np.array(means_peak)
        std_te = np.array(std_te); std_peak = np.array(std_peak)
        sz_for_color = np.array(sz_for_color)

        ax_te_pc.scatter(means_te, means_peak, c=sz_for_color, cmap=cmap, s=60, edgecolor="k", linewidth=0.4, alpha=0.9, label=res, zorder=4)
        ax_te_pc.errorbar(means_te, means_peak, xerr=std_te, yerr=std_peak, fmt="none", ecolor="k", elinewidth=1, capsize=2, alpha=0.25, zorder=2)

    ax_te_pc.axhline(0, color="k", linestyle="--", linewidth=0.8, alpha=0.8)
    ax_te_pc.axvline(0, color="k", linestyle="--", linewidth=0.8, alpha=0.8)
    ax_te_pc.set_xlabel("Δ TE MF→PC (bit)")
    ax_te_pc.set_ylabel("Δ Peak PC FR (Hz)")
    ax_te_pc.grid(False)

    y0 = 1.10
    for res, colr in [("Without Resilience","blue"), ("With Resilience","red")]:
        if res in stats_pc and not np.isnan(stats_pc[res][0]):
            rho, pval = stats_pc[res]
            ax_te_pc.text(0.5, y0, f"{res}: ρ={rho:.2f}, p={pval:.2e}", transform=ax_te_pc.transAxes, fontsize=12, color=colr, va="bottom", ha="center")
            y0 -= 0.08

    stats_slope = {}
    for res, cmap in [("Without Resilience","Blues_r"), ("With Resilience","Reds_r")]:
        rng = sigma_ranges.get(res, None)
        if rng is None:
            continue
        smin, smax = rng["sigma_min"], rng["sigma_max"]
        df_range = df[(df["resilience"] == res) & (df["sigma"] >= smin) & (df["sigma"] <= smax)].copy()
        te_vals, slope_vals, sz_vals = _compute_te_and_slope(df_range)
        if len(slope_vals) == 0:
            continue
        if np.any(sz_vals == 0):
            baseline_te = np.mean(te_vals[sz_vals == 0])
            baseline_slope = np.mean(slope_vals[sz_vals == 0])
        else:
            idx0 = np.argmin(sz_vals)
            baseline_te = te_vals[idx0]
            baseline_slope = slope_vals[idx0]
        te_delta = te_vals - baseline_te
        slope_delta = slope_vals - baseline_slope
        rho, pval = spearmanr(te_delta, slope_delta) if len(te_delta) > 1 else (np.nan, np.nan)
        stats_slope[res] = (rho, pval)

        sz_uniq = np.sort(np.unique(sz_vals))
        means_te, means_sl, std_te, std_sl, sz_for_color = [], [], [], [], []
        for sz0 in sz_uniq:
            m = (sz_vals == sz0)
            if np.sum(m) == 0:
                continue
            means_te.append(np.mean(te_delta[m]))
            means_sl.append(np.mean(slope_delta[m]))
            std_te.append(np.std(te_delta[m], ddof=1) if np.sum(m) > 1 else 0.0)
            std_sl.append(np.std(slope_delta[m], ddof=1) if np.sum(m) > 1 else 0.0)
            sz_for_color.append(sz0)

        if not means_te:
            continue

        means_te = np.array(means_te); means_sl = np.array(means_sl)
        std_te = np.array(std_te); std_sl = np.array(std_sl)
        sz_for_color = np.array(sz_for_color)

        ax_te_slope.scatter(means_te, means_sl, c=sz_for_color, cmap=cmap, s=60, edgecolor="k", linewidth=0.4, alpha=0.9, label=res, zorder=4)
        ax_te_slope.errorbar(means_te, means_sl, xerr=std_te, yerr=std_sl, fmt="none", ecolor="k", elinewidth=1, capsize=2, alpha=0.25, zorder=2)

    ax_te_slope.axhline(0, color="k", linestyle="--", linewidth=0.8, alpha=0.8)
    ax_te_slope.axvline(0, color="k", linestyle="--", linewidth=0.8, alpha=0.8)
    ax_te_slope.set_xlabel("Δ TE MF→PC (bit)")
    ax_te_slope.set_ylabel("Δ Purkinje aperiodic slope \n (proxy of E/I balance)")
    ax_te_slope.grid(False)

    y0 = 1.10
    for res, colr in [("Without Resilience","blue"), ("With Resilience","red")]:
        if res in stats_slope and not np.isnan(stats_slope[res][0]):
            rho, pval = stats_slope[res]
            ax_te_slope.text(0.5, y0, f"{res}: ρ={rho:.2f}, p={pval:.2e}", transform=ax_te_slope.transAxes, fontsize=12, color=colr, va="bottom", ha="center")
            y0 -= 0.08

    cmap_te = plt.cm.viridis
    te_all = np.array([])
    df_te_struct = None

    if df_all_str is None or df_te is None or norm_res is None:
        ax_te.text(0.5, 0.5, "STRUCT SKIPPED\n(pass df_all_str, df_te, norm_res)", transform=ax_te.transAxes, ha="center", va="center", fontsize=10)
        ax_te.axis("off")
    else:
        try:
            df_str = df_all_str.copy()
            df_str["Resilience"] = df_str["Resilience"].apply(norm_res)
            df_str["SZ Factor (%)"] = df_str["SZ Factor (%)"].astype(float).round(2)

            df_te_struct = df_te.copy()
            df_te_struct["Resilience"] = df_te_struct["Resilience"].apply(norm_res)
            df_te_struct["SZ Factor (%)"] = df_te_struct["SZ Factor (%)"].astype(float).round(2)

            df_te_struct["dte_mean"] = np.nan
            for res in df_te_struct["Resilience"].unique():
                sub = df_te_struct[df_te_struct["Resilience"] == res].sort_values("SZ Factor (%)")
                if sub.empty:
                    continue
                base = sub.loc[sub["SZ Factor (%)"] == 0, "te_mean"].mean() if np.any(sub["SZ Factor (%)"] == 0) else sub["te_mean"].iloc[0]
                m = (df_te_struct["Resilience"] == res)
                df_te_struct.loc[m, "dte_mean"] = df_te_struct.loc[m, "te_mean"] - base

            te_all = df_te_struct["dte_mean"].astype(float).values

            def sum_unique_synapses(x):
                return np.unique(x).sum()

            df_struct = (
                df_str.groupby(["SZ Factor (%)", "Resilience"], as_index=False)
                .agg({"total_cells": "first", "synapses": sum_unique_synapses})
            )

            df_baseline = df_struct[df_struct["SZ Factor (%)"] == 0].set_index("Resilience")
            rows = []
            for _, row in df_struct.iterrows():
                res = row["Resilience"]
                base_cells = df_baseline.loc[res, "total_cells"]
                base_syn = df_baseline.loc[res, "synapses"]
                rows.append({
                    "Resilience": res,
                    "SZ Factor (%)": row["SZ Factor (%)"],
                    "cells_pct": 100*(row["total_cells"]-base_cells)/base_cells,
                    "syn_pct": 100*(row["synapses"]-base_syn)/base_syn,
                })
            df_struct_pct = pd.DataFrame(rows)

            df_plot_struct = df_struct_pct.merge(
                df_te_struct[["SZ Factor (%)", "Resilience", "dte_mean"]],
                on=["SZ Factor (%)", "Resilience"],
                how="inner"
            )

            dte_min = np.nanmin(te_all) if te_all.size > 0 else 0.0
            dte_max = np.nanmax(te_all) if te_all.size > 0 else 1.0
            norm_dte = Normalize(vmin=dte_min, vmax=dte_max)

            marker_map_struct = {"With Resilience": "s", "Without Resilience": "o"}
            for res in ["With Resilience", "Without Resilience"]:
                sub = df_plot_struct[df_plot_struct["Resilience"] == res]
                if sub.empty:
                    continue
                colors_s = cmap_te(norm_dte(sub["dte_mean"].values))
                ax_te.scatter(sub["cells_pct"], sub["syn_pct"], c=colors_s, s=70, edgecolor="k", linewidth=0.6, marker=marker_map_struct[res], label=res)

            ax_te.set_xlabel("Δ Cells (%)")
            ax_te.set_ylabel("Δ Synapses (%)")
            ax_te.legend(title="Resilience", loc="upper left", fontsize=8)
            ax_te.axhspan(-30, -60, facecolor="k", alpha=0.2, zorder=-2)
        except Exception as e:
            ax_te.text(0.5, 0.5, f"STRUCT ERROR:\n{e}", transform=ax_te.transAxes, ha="center", va="center", fontsize=10)
            ax_te.axis("off")

    fig2.subplots_adjust(left=0.07, right=0.98, bottom=0.10, top=0.93, hspace=0.55, wspace=0.35)

    pos_g = ax_te_pc.get_position()
    
    x_shift = pos_g.x0
    y_base  = pos_g.y0 - 0.38 - 0.15
    
    total_h = pos_g.height * 1.25
    gap_y   = 0.12 * total_h
    h_sub   = (total_h - gap_y) / 2.0
    
    shrink_w = 0.03*2.5          # 3% più stretto (puoi provare 0.02–0.04)
    new_w    = pos_g.width * (1 - shrink_w)
    
    ax_mfpc_no = fig2.add_axes([
        x_shift,
        y_base + h_sub + gap_y,
        new_w,
        h_sub
    ])
    
    ax_mfpc_wi = fig2.add_axes([
        x_shift,
        y_base,
        new_w,
        h_sub
    ], sharex=ax_mfpc_no)


    def _plot_mf_pc_peak_relation(df_range, res_label, ax, title, ecolor):
        sz_mf, mf_mean, mf_std = _compute_peak_vs_sz(df_range, "mossy_fibers_timeseries")
        sz_pc, pc_mean, pc_std = _compute_peak_vs_sz(df_range, "purkinje_timeseries")
        if len(sz_mf) == 0 or len(sz_pc) == 0:
            ax.text(0.5, 0.5, "NO DATA", transform=ax.transAxes, ha="center", va="center", fontsize=10)
            ax.axis("off")
            return

        base_mf = np.mean(mf_mean[sz_mf == 0]) if np.any(sz_mf == 0) else mf_mean[np.argmin(sz_mf)]
        base_pc = np.mean(pc_mean[sz_pc == 0]) if np.any(sz_pc == 0) else pc_mean[np.argmin(sz_pc)]
        mf_delta = mf_mean - base_mf
        pc_delta = pc_mean - base_pc

        df_mf = pd.DataFrame({"SZ": sz_mf, "mf_delta": mf_delta, "mf_std": mf_std})
        df_pc = pd.DataFrame({"SZ": sz_pc, "pc_delta": pc_delta, "pc_std": pc_std})
        df_mf["SZ_round"] = df_mf["SZ"].round(2)
        df_pc["SZ_round"] = df_pc["SZ"].round(2)
        df_peaks = df_mf.merge(df_pc, on="SZ_round", how="inner").rename(columns={"SZ_round": "SZ"}).sort_values("SZ")
        if df_peaks.empty:
            ax.text(0.5, 0.5, "NO COMMON SZ", transform=ax.transAxes, ha="center", va="center", fontsize=10)
            ax.axis("off")
            return

        x = df_peaks["mf_delta"].to_numpy(dtype=float)
        y = df_peaks["pc_delta"].to_numpy(dtype=float)
        xerr = df_peaks["mf_std"].to_numpy(dtype=float)
        yerr = df_peaks["pc_std"].to_numpy(dtype=float)

        dte_vals = None
        if df_te_struct is not None:
            te_sub = df_te_struct[df_te_struct["Resilience"] == res_label][["SZ Factor (%)", "dte_mean"]].rename(columns={"SZ Factor (%)": "SZ"})
            te_sub["SZ"] = te_sub["SZ"].round(2)
            df_tmp = df_peaks.merge(te_sub, on="SZ", how="left")
            dte_vals = df_tmp["dte_mean"].to_numpy(dtype=float)

        norm_te = Normalize(vmin=np.nanmin(te_all) if te_all.size else 0.0, vmax=np.nanmax(te_all) if te_all.size else 1.0)
        colors = cmap_te(norm_te(dte_vals)) if (dte_vals is not None and np.any(np.isfinite(dte_vals))) else cmap_te(norm_te(np.zeros_like(x)))

        ax.errorbar(x, y, xerr=xerr, yerr=yerr, fmt="none", ecolor="k", elinewidth=1.0, capsize=2, alpha=0.25, zorder=1)
        ax.scatter(x, y, c=colors, s=60, edgecolor="k", linewidth=0.4, alpha=0.9, zorder=2)

        ax.axhline(0, color="k", linestyle="--", linewidth=0.8, alpha=0.8)
        ax.axvline(0, color="k", linestyle="--", linewidth=0.8, alpha=0.8)

        rho, pval = spearmanr(x, y) if len(x) > 1 else (np.nan, np.nan)
        ax.set_title(f"{title}: ρ={rho:.2f}, p={pval:.2e}" if not np.isnan(rho) else title, fontsize=12, color=ecolor)
        ax.grid(False)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    _plot_mf_pc_peak_relation(df_no_range, "Without Resilience", ax_mfpc_no, "Without Resilience", "blue")
    _plot_mf_pc_peak_relation(df_wi_range, "With Resilience", ax_mfpc_wi, "With Resilience", "red")

    ax_mfpc_no.tick_params(labelbottom=False)
    ax_mfpc_wi.set_xlabel("Δ Peak MF FR (Hz)")

    ymin = min(ax_mfpc_no.get_ylim()[0], ax_mfpc_wi.get_ylim()[0])
    ymax = max(ax_mfpc_no.get_ylim()[1], ax_mfpc_wi.get_ylim()[1])
    ax_mfpc_no.set_ylim(ymin, ymax)
    ax_mfpc_wi.set_ylim(ymin, ymax)

    pos_no = ax_mfpc_no.get_position()
    pos_wi = ax_mfpc_wi.get_position()
    fig2.text(pos_no.x0 - 0.05, (pos_no.y1 + pos_wi.y0)/2.0, "Δ Peak PC FR (Hz)", rotation="vertical", va="center", ha="center")

    for ax in (ax_mfpc_no, ax_mfpc_wi):
        x_min, x_max = ax.get_xlim()
        y_min, y_max = ax.get_ylim()
        x0_rel = np.clip((0 - x_min) / (x_max - x_min), 0, 1)
        y0_rel = np.clip((0 - y_min) / (y_max - y_min), 0, 1)
        ax.add_patch(plt.Rectangle((x0_rel, y0_rel), 1-x0_rel, 1-y0_rel, transform=ax.transAxes, facecolor="#fff7b2", edgecolor="none", alpha=0.25, zorder=-3))

    for ax in (ax_te_pc, ax_te_slope):
        x_min, x_max = ax.get_xlim()
        y_min, y_max = ax.get_ylim()
        if (x_max - x_min) == 0 or (y_max - y_min) == 0:
            continue
        x0_rel = np.clip((0 - x_min) / (x_max - x_min), 0, 1)
        y0_rel = np.clip((0 - y_min) / (y_max - y_min), 0, 1)
        ax.add_patch(plt.Rectangle((x0_rel, y0_rel), 1-x0_rel, 1-y0_rel, transform=ax.transAxes, facecolor="#fff7b2", edgecolor="none", alpha=0.25, zorder=-3))

    panel_labels_2 = {ax_te_pc:"a", ax_te_slope:"b", ax_te:"e", ax_mfpc_no:"c", ax_mfpc_wi:"d"}
    for ax, lab in panel_labels_2.items():
        ax.text(-0.10, 1.15, lab, transform=ax.transAxes, fontsize=14, fontweight="bold", va="top", ha="left")

    if te_all.size > 0:
        norm_te = Normalize(vmin=np.nanmin(te_all), vmax=np.nanmax(te_all))
        sm_te = ScalarMappable(norm=norm_te, cmap=cmap_te)
        sm_te.set_array(te_all)
        pos_no = ax_mfpc_no.get_position()
        pos_wi = ax_mfpc_wi.get_position()
        x_cbar = pos_no.x1 + 0.02
        y_cbar = pos_wi.y0
        h_cbar = pos_no.y1 - pos_wi.y0
        w_cbar = 0.018
        cax_te = fig2.add_axes([x_cbar, y_cbar, w_cbar, h_cbar])
        cb_te = fig2.colorbar(sm_te, cax=cax_te, orientation="vertical")
        cb_te.ax.set_title("Δ TE MF→PC \n (bit)", fontsize=14, pad=10)
        cb_te.ax.tick_params(labelsize=14)

    axes2[1,0].remove()

    # =========================
    # CENTRAL "HAT" COLORBARS
    # =========================
    
    # bounding box dei due pannelli a–b
    pos_a = ax_te_pc.get_position()
    pos_b = ax_te_slope.get_position()
    
    # centro orizzontale dei due pannelli
    x_center = 0.5 * (pos_a.x0 + pos_b.x1)
    
    # larghezza del cappello (70% del totale a–b)
    hat_width = 0.70 * (pos_b.x1 - pos_a.x0)
    x0 = x_center - hat_width / 2
    
    # altezze e spazi
    bar_h   = 0.018     # più sottile
    gap     = 0.006*4     # spazio tra le due barre
    y_top   = max(pos_a.y1, pos_b.y1) + 0.065*2.5  # più in alto (chiave!)
    
    # assi colorbar
    cax_red  = fig2.add_axes([x0, y_top,              hat_width, bar_h])
    cax_blue = fig2.add_axes([x0, y_top - bar_h - gap, hat_width, bar_h])
    
    # normalizzazione SZ
    norm_sz = Normalize(
        vmin=df[x_col].min(),
        vmax=df[x_col].max()
    )
    
    # mappabili
    sm_red  = ScalarMappable(norm=norm_sz, cmap="Reds")
    sm_blue = ScalarMappable(norm=norm_sz, cmap="Blues")
    
    # colorbar
    cb_red = fig2.colorbar(sm_red,  cax=cax_red,  orientation="horizontal")
    cb_blue = fig2.colorbar(sm_blue, cax=cax_blue, orientation="horizontal")
    
    # titoli delle barre
    cb_red.ax.set_title("With Resilience", fontsize=14, color="red", pad=2)
    cb_blue.ax.set_title("Without Resilience", fontsize=14, color="blue", pad=2)
    
    # label comune (solo sotto)
    cb_blue.set_label("SZ Factor (%)", fontsize=12)
    
    # tick più piccoli
    cb_red.ax.tick_params(labelsize=10)
    cb_blue.ax.tick_params(labelsize=10)

    cb_red.ax.tick_params(
    bottom=False,
    top=False,
    labelbottom=False
    )

    cb_red.ax.tick_params(labelsize=9)


    out2 = os.path.join(outdir, fname_ghijk)
    fig2.savefig(_prefix_output(out2), dpi=600, bbox_inches="tight")
    print("✅ Salvato:", out2)
    plt.show()

plot_lag_te_and_epoched_5x2(
    df_all,
    sigma_ranges=sigma_ranges,
    x_col="SZ Factor (%)",
    outdir="./tempPlots",
    fname_abf="lag_te_epoched_a_f.png",
    fname_ghijk="lag_te_epoched_g_k.png",
    use_delta=True,
    alpha=0.001,
    dt=0.1,
    window_ms=(-70, 200),
    df_all_str=df_all_str,
    df_te=df_te,
    norm_res=norm_res
)


## v7

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind, spearmanr
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
from scipy.stats import linregress


def plot_lag_te_and_epoched_5x2(
    df,
    sigma_ranges,
    x_col="SZ Factor (%)",
    outdir="./tempPlots",
    fname_abf="lag_te_epoched_abf.png",
    fname_ghijk="lag_te_epoched_ghijk.png",
    use_delta=False,
    alpha=0.05,
    dt=0.1,
    window_ms=(-100, 65),
    df_all_str=None,
    df_te=None,
    norm_res=None
):
    os.makedirs(outdir, exist_ok=True)

    if "purk_slope" not in df.columns and "fooof_purk" in df.columns:
        def _get_offset(d):
            if isinstance(d, dict) and "aperiodic_params" in d:
                ap = d["aperiodic_params"]
                if isinstance(ap, (list, tuple)) and len(ap) >= 2:
                    return float(ap[0])
            return np.nan
        def _get_slope(d):
            if isinstance(d, dict) and "aperiodic_params" in d:
                ap = d["aperiodic_params"]

                if isinstance(ap, (list, tuple)) and len(ap) >= 2:
                    return float(ap[1])
            return np.nan
        df = df.copy()
        df["purk_offset"] = df["fooof_purk"].apply(_get_offset)
        df["purk_slope"]  = df["fooof_purk"].apply(_get_slope)

    metrics = [
        ("optimal_lag_ms",          "Lag MF→PC (ms)"),
        ("te_mossy_fibers_to_purk", "TE MF→PC (bit)")
    ]
    resilience_levels = ["Without Resilience", "With Resilience"]
    base_colors = {"Without Resilience": "blue", "With Resilience": "red"}

    legend_handles = [
        Line2D([0],[0], color=base_colors["Without Resilience"], marker="o", linestyle="-", label="Without Resilience"),
        Line2D([0],[0], color=base_colors["With Resilience"],    marker="o", linestyle="-", label="With Resilience"),
    ]

    metric_results = {col: {res: {} for res in resilience_levels} for col, _ in metrics}
    lag_stats, te_stats = {}, {}

    for col, _ in metrics:
        if col not in df.columns:
            continue
        for res in resilience_levels:
            rng = sigma_ranges.get(res, None)
            if rng is None:
                continue
            smin, smax = rng["sigma_min"], rng["sigma_max"]
            sub = df[(df["resilience"] == res) &
                     (df["sigma"] >= smin) &
                     (df["sigma"] <= smax)].copy()
            if sub.empty:
                continue
            metric_results[col][res]["sub"] = sub
            grp = sub.groupby(x_col)[col]
            x_vals = np.array(sorted(grp.groups.keys()), dtype=float)
            mean_vals = grp.mean().reindex(x_vals).to_numpy(dtype=float)
            std_vals  = grp.std(ddof=1).reindex(x_vals).to_numpy(dtype=float)
            if use_delta:
                idx0 = np.where(x_vals == 0)[0][0] if np.any(x_vals == 0) else 0
                baseline = mean_vals[idx0]
                mean_vals = mean_vals - baseline
            metric_results[col][res]["x"] = x_vals
            metric_results[col][res]["mean"] = mean_vals
            metric_results[col][res]["std"] = std_vals

    win_pre  = int(abs(window_ms[0]) / dt)
    win_post = int(window_ms[1] / dt)
    time_axis = np.arange(-win_pre, win_post) * dt

    def _fit_slope(x, y):
        m = np.isfinite(x) & np.isfinite(y)
        x = np.asarray(x)[m]
        y = np.asarray(y)[m]
        if x.size < 3 or np.all(x == x[0]):
            return None
        lr = linregress(x, y)
        return {
            "slope": lr.slope,
            "intercept": lr.intercept,
            "r": lr.rvalue,
            "p": lr.pvalue,
            "stderr": lr.stderr,
            "ci95": (lr.slope - 1.96*lr.stderr, lr.slope + 1.96*lr.stderr),
            "r2": lr.rvalue**2
        }


    def _get_ylims(trace_column):
        if "mossy" in trace_column:
            return [-10, 200]
        if "purkinje" in trace_column:
            return [35, 110]
        return [0, 1]

    def _get_sigma_range_df(df_main, sigma_ranges, res):
        rng = sigma_ranges.get(res, None)
        if rng is None:
            return None, None, None
        smin, smax = rng["sigma_min"], rng["sigma_max"]
        df_sub = df_main[(df_main["resilience"] == res) &
                         (df_main["sigma"] >= smin) &
                         (df_main["sigma"] <= smax)].copy()
        if df_sub.empty:
            return None, None, None
        return df_sub, smin, smax

    def _make_sigma_averaged_df(df_sub, trace_columns, onset_column="burst_onsets_ms", group_column="atrophy_percent"):
        if df_sub is None or df_sub.empty:
            return None
        rows = []
        for grp_val, g in df_sub.groupby(group_column):
            row = {group_column: grp_val}
            if "resilience" in g.columns:
                row["resilience"] = g["resilience"].iloc[0]
            row[onset_column] = g[onset_column].iloc[0]
            for trace_col in trace_columns:
                traces = [np.asarray(v, dtype=float) for v in g[trace_col]]
                if not traces:
                    row[trace_col] = np.array([])
                    continue
                L = min(t.shape[0] for t in traces)
                if L == 0:
                    row[trace_col] = np.array([])
                    continue
                stack = np.vstack([t[:L] for t in traces])
                row[trace_col] = stack.mean(axis=0)
            rows.append(row)
        return pd.DataFrame(rows) if rows else None

    def _compute_peak_vs_sz(df_sub, trace_column, peak_window_ms=(0, 75)):
        if df_sub is None or df_sub.empty:
            return (np.array([]), np.array([]), np.array([]))
        rows = []
        for _, row in df_sub.iterrows():
            trace = np.array(row[trace_column], dtype=float)
            onsets = row["burst_onsets_ms"]
            sz = float(row[x_col])
            single_peaks = []
            for onset in onsets:
                center = int(onset / dt)
                start = center + int(peak_window_ms[0] / dt)
                end   = center + int(peak_window_ms[1] / dt)
                if start < 0 or end > len(trace):
                    continue
                seg = trace[start:end]
                if seg.size == 0:
                    continue
                single_peaks.append(float(np.max(seg)))
            if single_peaks:
                rows.append({"SZ": sz, "peak": float(np.mean(single_peaks))})
        if not rows:
            return (np.array([]), np.array([]), np.array([]))
        tmp = pd.DataFrame(rows)
        sz_vals, peak_mean, peak_std = [], [], []
        for sz in sorted(tmp["SZ"].unique()):
            vals = tmp.loc[tmp["SZ"] == sz, "peak"].to_numpy(dtype=float)
            sz_vals.append(sz)
            peak_mean.append(np.mean(vals))
            peak_std.append(np.std(vals, ddof=1) if vals.size > 1 else 0.0)
        return (np.array(sz_vals, dtype=float),
                np.array(peak_mean, dtype=float),
                np.array(peak_std, dtype=float))

    def _compute_te_and_peak(df_sub, trace_column, peak_window_ms=(0, 75)):
        if df_sub is None or df_sub.empty:
            return (np.array([]), np.array([]), np.array([]))
        te_vals, peak_vals, sz_vals = [], [], []
        for _, row in df_sub.iterrows():
            trace = np.array(row[trace_column], dtype=float)
            onsets = row["burst_onsets_ms"]
            sz = float(row[x_col])
            te = float(row["te_mossy_fibers_to_purk"])
            single_peaks = []
            for onset in onsets:
                center = int(onset / dt)
                start = center + int(peak_window_ms[0] / dt)
                end   = center + int(peak_window_ms[1] / dt)
                if start < 0 or end > len(trace):
                    continue
                seg = trace[start:end]
                if seg.size == 0:
                    continue
                single_peaks.append(float(np.max(seg)))
            if single_peaks:
                te_vals.append(te)
                peak_vals.append(np.mean(single_peaks))
                sz_vals.append(sz)
        return (np.array(te_vals, dtype=float),
                np.array(peak_vals, dtype=float),
                np.array(sz_vals, dtype=float))

    def _compute_te_and_slope(df_sub):
        if df_sub is None or df_sub.empty or "purk_slope" not in df_sub.columns:
            return (np.array([]), np.array([]), np.array([]))
        te_vals = df_sub["te_mossy_fibers_to_purk"].astype(float).to_numpy()
        slope_vals = df_sub["purk_slope"].astype(float).to_numpy()
        sz_vals = df_sub[x_col].astype(float).to_numpy()
        return te_vals, slope_vals, sz_vals

    df_no_range, _, _ = _get_sigma_range_df(df, sigma_ranges, "Without Resilience")
    df_wi_range, _, _ = _get_sigma_range_df(df, sigma_ranges, "With Resilience")

    df_no_avg = _make_sigma_averaged_df(
        df_no_range,
        trace_columns=["mossy_fibers_timeseries", "purkinje_timeseries"],
        onset_column="burst_onsets_ms",
        group_column="atrophy_percent"
    )
    df_wi_avg = _make_sigma_averaged_df(
        df_wi_range,
        trace_columns=["mossy_fibers_timeseries", "purkinje_timeseries"],
        onset_column="burst_onsets_ms",
        group_column="atrophy_percent"
    )

    fig1, axes1 = plt.subplots(3, 2, figsize=(12, 14))
    ax_lag      = axes1[0, 0]
    ax_te       = axes1[0, 1]
    ax_mf_nores = axes1[1, 0]
    ax_pc_nores = axes1[1, 1]
    ax_mf_with  = axes1[2, 0]
    ax_pc_with  = axes1[2, 1]
    fig1.legend(handles=legend_handles, loc="upper center", bbox_to_anchor=(0.5, 1.02), ncol=2, frameon=False)

    for ax in (ax_lag, ax_te):
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    col = metrics[0][0]
    for res in resilience_levels:
        res_dict = metric_results[col].get(res, {})
        if "x" not in res_dict:
            continue
        x_vals, mean_vals, std_vals = res_dict["x"], res_dict["mean"], res_dict["std"]
        ax_lag.errorbar(x_vals, mean_vals, yerr=std_vals, fmt="none", ecolor="k", elinewidth=1, capsize=3, alpha=0.25, zorder=1)
        ax_lag.scatter(x_vals, mean_vals, color=base_colors[res], s=60, edgecolor="k", linewidth=0.4, zorder=3)
        ax_lag.plot(x_vals, mean_vals, color=base_colors[res], alpha=0.7, linewidth=1.5, zorder=2)
        if len(mean_vals) > 0:
            idx = int(np.argmax(mean_vals))
            lag_stats[res] = (float(mean_vals[idx]), float(x_vals[idx]))

    resA, resB = resilience_levels
    if ("sub" in metric_results[col].get(resA, {}) and "sub" in metric_results[col].get(resB, {})):
        subA = metric_results[col][resA]["sub"]
        subB = metric_results[col][resB]["sub"]
        common_x = sorted(set(subA[x_col].astype(float).unique()).intersection(set(subB[x_col].astype(float).unique())))
        pvals = {}
        for x0 in common_x:
            valsA = subA.loc[subA[x_col] == x0, col].to_numpy(dtype=float)
            valsB = subB.loc[subB[x_col] == x0, col].to_numpy(dtype=float)
            if len(valsA) >= 2 and len(valsB) >= 2:
                _, p = ttest_ind(valsA, valsB, equal_var=False, nan_policy="omit")
                pvals[x0] = p
        if pvals:
            xs_sorted = np.array(sorted(pvals.keys()), dtype=float)
            dx = np.min(np.diff(xs_sorted)) if len(xs_sorted) > 1 else 1.0
            for x0, p in pvals.items():
                if p < alpha:
                    ax_lag.axvspan(x0 - dx*0.5, x0 + dx*0.5, color="#fff7b2", alpha=0.0, zorder=-1)

    ax_lag.axhline(0, color="k", linestyle="--", linewidth=1, alpha=0.8)
    ax_lag.set_ylabel("ms" + (" (Δ vs SZ=0)" if use_delta else ""))
    ax_lag.set_xlabel(x_col)
    ax_lag.grid(False)
    ax_lag.set_title("MF→PC lag (Saccadic dysmetria)", fontweight="bold", pad=34)
    y_text = 1.05
    if "Without Resilience" in lag_stats:
        lag_val, sz_val = lag_stats["Without Resilience"]
        ax_lag.text(0.5, y_text, f"Without Resilience: lag={int(round(lag_val))} ms at SZ={int(round(sz_val))}%", transform=ax_lag.transAxes, fontsize=12, color="k", va="bottom", ha="center")
        y_text -= 0.07
    if "With Resilience" in lag_stats:
        lag_val, sz_val = lag_stats["With Resilience"]
        ax_lag.text(0.5, y_text, f"With Resilience: lag={int(round(lag_val))} ms at SZ={int(round(sz_val))}%", transform=ax_lag.transAxes, fontsize=12, color="k", va="bottom", ha="center")


    col = metrics[1][0]
    for res in resilience_levels:
        res_dict = metric_results[col].get(res, {})
        if "x" not in res_dict:
            continue
        x_vals, mean_vals, std_vals = res_dict["x"], res_dict["mean"], res_dict["std"]
        ax_te.errorbar(x_vals, mean_vals, yerr=std_vals, fmt="none", ecolor="k", elinewidth=1, capsize=3, alpha=0.25, zorder=1)
        ax_te.scatter(x_vals, mean_vals, color=base_colors[res], s=60, edgecolor="k", linewidth=0.4, zorder=3)
        ax_te.plot(x_vals, mean_vals, color=base_colors[res], alpha=0.7, linewidth=1.5, zorder=2)
        if len(mean_vals) > 0:
            idx = int(np.argmax(mean_vals))
            te_stats[res] = (float(mean_vals[idx]), float(x_vals[idx]))

    if ("sub" in metric_results[col].get(resA, {}) and "sub" in metric_results[col].get(resB, {})):
        subA = metric_results[col][resA]["sub"]
        subB = metric_results[col][resB]["sub"]
        common_x = sorted(set(subA[x_col].astype(float).unique()).intersection(set(subB[x_col].astype(float).unique())))
        pvals = {}
        for x0 in common_x:
            valsA = subA.loc[subA[x_col] == x0, col].to_numpy(dtype=float)
            valsB = subB.loc[subB[x_col] == x0, col].to_numpy(dtype=float)
            if len(valsA) >= 2 and len(valsB) >= 2:
                _, p = ttest_ind(valsA, valsB, equal_var=False, nan_policy="omit")
                pvals[x0] = p
        if pvals:
            xs_sorted = np.array(sorted(pvals.keys()), dtype=float)
            dx = np.min(np.diff(xs_sorted)) if len(xs_sorted) > 1 else 1.0
            for x0, p in pvals.items():
                if p < alpha:
                    ax_te.axvspan(x0 - dx*0.5, x0 + dx*0.5, color="#fff7b2", alpha=0.0, zorder=-1)

    # =========================
    # SLOPE STATS (LAG vs SZ)
    # =========================
    col_lag = "optimal_lag_ms"
    slope_stats_lag = {}
    
    for res in resilience_levels:
        res_dict = metric_results[col_lag].get(res, {})
        if "x" not in res_dict:
            continue
        x_vals = res_dict["x"]
        y_vals = res_dict["mean"]  # stai fittando la curva media (come nel plot)
        fit = _fit_slope(x_vals, y_vals)
        if fit is None:
            continue
        slope_stats_lag[res] = fit
    
    # stampa a terminale
    for res in resilience_levels:
        if res not in slope_stats_lag:
            continue
        fit = slope_stats_lag[res]
        print(f"[LAG vs SZ] {res}: slope={fit['slope']:.4f} ms/%  "
              f"(95% CI [{fit['ci95'][0]:.4f},{fit['ci95'][1]:.4f}])  "
              f"p={fit['p']:.3e}  r={fit['r']:.3f}  R²={fit['r2']:.3f}")

    
    ax_te.axhline(0, color="k", linestyle="--", linewidth=1, alpha=0.8)
    ax_te.set_ylabel("bit" + (" (Δ vs SZ=0)" if use_delta else ""))
    ax_te.set_xlabel(x_col)
    ax_te.grid(False)
    ax_te.set_title("MF→PC transfer entropy (TE)", fontweight="bold", pad=34)
    y_text = 1.05
    if "Without Resilience" in te_stats:
        te_val, sz_val = te_stats["Without Resilience"]
        ax_te.text(0.5, y_text, f"Without Resilience: max TE={te_val:.3f} bit at SZ={int(round(sz_val))}%", transform=ax_te.transAxes, fontsize=12, color="k", va="bottom", ha="center")
        y_text -= 0.07
    if "With Resilience" in te_stats:
        te_val, sz_val = te_stats["With Resilience"]
        ax_te.text(0.5, y_text, f"With Resilience: max TE={te_val:.3f} bit at SZ={int(round(sz_val))}%", transform=ax_te.transAxes, fontsize=12, color="k", va="bottom", ha="center")

    if df_no_avg is not None:
        plot_epoched_firing_rate(df=df_no_avg, trace_column="mossy_fibers_timeseries", time_axis=time_axis, pop_name="Mossy Fibers (input) \n Without Resilience", stim_column=None, onset_column="burst_onsets_ms", group_column="atrophy_percent", cmap="Blues_r", ylims=_get_ylims("mossy_fibers_timeseries"), operator=None, window_ms=window_ms, dt=dt, save_path=None, ax=ax_mf_nores)
        plot_epoched_firing_rate(df=df_no_avg, trace_column="purkinje_timeseries", time_axis=time_axis, pop_name="Purkinje Cells (output) \n Without Resilience", stim_column=None, onset_column="burst_onsets_ms", group_column="atrophy_percent", cmap="Blues_r", ylims=_get_ylims("purkinje_timeseries"), operator=None, window_ms=window_ms, dt=dt, save_path=None, ax=ax_pc_nores)

        INSET_TITLE_FS = 10  # era 8
        INSET_LABEL_FS = 9   # era 7
        INSET_TICK_FS  = 11  # era 9


        
        inset_mf_no = ax_mf_nores.inset_axes([0.595, 0.40, 0.35, 0.35])
        sz_mf, peak_mf_mean, peak_mf_std = _compute_peak_vs_sz(df_no_range, "mossy_fibers_timeseries")
        if len(sz_mf) > 0:
            baseline_mf = np.mean(peak_mf_mean[sz_mf == 0]) if np.any(sz_mf == 0) else peak_mf_mean[np.argmin(sz_mf)]
            peak_mf_delta = peak_mf_mean - baseline_mf
            inset_mf_no.errorbar(sz_mf, peak_mf_delta, yerr=peak_mf_std, fmt="none", ecolor="k", elinewidth=1, capsize=2, alpha=0.25)
            inset_mf_no.scatter(sz_mf, peak_mf_delta, c=sz_mf, cmap="Blues_r", s=15, edgecolor="k", linewidth=0.3)
            rho, pval = spearmanr(sz_mf, peak_mf_delta)
            inset_mf_no.set_title(f"ρ={rho:.2f}, p={pval:.2e}", fontsize=INSET_TITLE_FS)
            inset_mf_no.set_xlabel("SZ Factor (%)", fontsize=INSET_LABEL_FS)
            inset_mf_no.set_ylabel("Δ Peak FR (Hz)", fontsize=INSET_LABEL_FS)
            inset_mf_no.tick_params(labelsize=INSET_TICK_FS)

            inset_mf_no.set_ylim(-30, 30)
            inset_mf_no.axhline(0, color="k", linestyle="--", linewidth=0.8, alpha=0.8)
        else:
            inset_mf_no.set_visible(False)

        inset_pc_no = ax_pc_nores.inset_axes([0.595, 0.40, 0.35, 0.35])
        sz_pc, peak_pc_mean, peak_pc_std = _compute_peak_vs_sz(df_no_range, "purkinje_timeseries")
        if len(sz_pc) > 0:
            baseline_pc = np.mean(peak_pc_mean[sz_pc == 0]) if np.any(sz_pc == 0) else peak_pc_mean[np.argmin(sz_pc)]
            peak_pc_delta = peak_pc_mean - baseline_pc
            inset_pc_no.errorbar(sz_pc, peak_pc_delta, yerr=peak_pc_std, fmt="none", ecolor="k", elinewidth=1, capsize=2, alpha=0.25)
            inset_pc_no.scatter(sz_pc, peak_pc_delta, c=sz_pc, cmap="Blues_r", s=30, edgecolor="k", linewidth=0.3)
            rho, pval = spearmanr(sz_pc, peak_pc_delta)
            inset_pc_no.set_title(f"ρ={rho:.2f}, p={pval:.2e}", fontsize=INSET_TITLE_FS)
            inset_pc_no.set_xlabel("SZ Factor (%)", fontsize=INSET_LABEL_FS)
            inset_pc_no.set_ylabel("Δ Peak FR (Hz)", fontsize=INSET_LABEL_FS)
            inset_pc_no.tick_params(labelsize=INSET_TICK_FS)
            inset_pc_no.set_ylim(-30, 30)
            inset_pc_no.axhline(0, color="k", linestyle="--", linewidth=0.8, alpha=0.8)
        else:
            inset_pc_no.set_visible(False)
    else:
        ax_mf_nores.set_title("Mossy Fibers – Without Resilience (no σ)")
        ax_pc_nores.set_title("Purkinje – Without Resilience (no σ)")

    if df_wi_avg is not None:
        plot_epoched_firing_rate(df=df_wi_avg, trace_column="mossy_fibers_timeseries", time_axis=time_axis, pop_name="Mossy Fibers (input) \n With Resilience", stim_column=None, onset_column="burst_onsets_ms", group_column="atrophy_percent", cmap="Reds_r", ylims=_get_ylims("mossy_fibers_timeseries"), operator=None, window_ms=window_ms, dt=dt, save_path=None, ax=ax_mf_with)
        plot_epoched_firing_rate(df=df_wi_avg, trace_column="purkinje_timeseries", time_axis=time_axis, pop_name="Purkinje Cells (output) \n With Resilience", stim_column=None, onset_column="burst_onsets_ms", group_column="atrophy_percent", cmap="Reds_r", ylims=_get_ylims("purkinje_timeseries"), operator=None, window_ms=window_ms, dt=dt, save_path=None, ax=ax_pc_with)

        inset_mf_wi = ax_mf_with.inset_axes([0.595, 0.40, 0.35, 0.35])
        sz_mf, peak_mf_mean, peak_mf_std = _compute_peak_vs_sz(df_wi_range, "mossy_fibers_timeseries")
        if len(sz_mf) > 0:
            baseline_mf = np.mean(peak_mf_mean[sz_mf == 0]) if np.any(sz_mf == 0) else peak_mf_mean[np.argmin(sz_mf)]
            peak_mf_delta = peak_mf_mean - baseline_mf
            inset_mf_wi.errorbar(sz_mf, peak_mf_delta, yerr=peak_mf_std, fmt="none", ecolor="k", elinewidth=1, capsize=2, alpha=0.25)
            inset_mf_wi.scatter(sz_mf, peak_mf_delta, c=sz_mf, cmap="Reds_r", s=30, edgecolor="k", linewidth=0.3)
            rho, pval = spearmanr(sz_mf, peak_mf_delta)
            inset_mf_wi.set_title(f"ρ={rho:.2f}, p={pval:.2e}", fontsize=INSET_TITLE_FS)
            inset_mf_wi.set_xlabel("SZ Factor (%)", fontsize=INSET_LABEL_FS)
            inset_mf_wi.set_ylabel("Δ Peak FR (Hz)", fontsize=INSET_LABEL_FS)
            inset_mf_wi.tick_params(labelsize=INSET_TICK_FS)
            inset_mf_wi.set_ylim(-30, 30)
            inset_mf_wi.axhline(0, color="k", linestyle="--", linewidth=0.8, alpha=0.8)
        else:
            inset_mf_wi.set_visible(False)

        inset_pc_wi = ax_pc_with.inset_axes([0.595, 0.40, 0.35, 0.35])
        sz_pc, peak_pc_mean, peak_pc_std = _compute_peak_vs_sz(df_wi_range, "purkinje_timeseries")
        if len(sz_pc) > 0:
            baseline_pc = np.mean(peak_pc_mean[sz_pc == 0]) if np.any(sz_pc == 0) else peak_pc_mean[np.argmin(sz_pc)]
            peak_pc_delta = peak_pc_mean - baseline_pc
            inset_pc_wi.errorbar(sz_pc, peak_pc_delta, yerr=peak_pc_std, fmt="none", ecolor="k", elinewidth=1, capsize=2, alpha=0.25)
            inset_pc_wi.scatter(sz_pc, peak_pc_delta, c=sz_pc, cmap="Reds_r", s=30, edgecolor="k", linewidth=0.3)
            rho, pval = spearmanr(sz_pc, peak_pc_delta)
            inset_pc_wi.set_title(f"ρ={rho:.2f}, p={pval:.2e}", fontsize=INSET_TITLE_FS)
            inset_pc_wi.set_xlabel("SZ Factor (%)", fontsize=INSET_LABEL_FS)
            inset_pc_wi.set_ylabel("Δ Peak FR (Hz)", fontsize=INSET_LABEL_FS)
            inset_pc_wi.tick_params(labelsize=INSET_TICK_FS)
            inset_pc_wi.set_ylim(-30, 30)
            inset_pc_wi.axhline(0, color="k", linestyle="--", linewidth=0.8, alpha=0.8)
        else:
            inset_pc_wi.set_visible(False)
    else:
        ax_mf_with.set_title("Mossy Fibers – With Resilience (no σ)")
        ax_pc_with.set_title("Purkinje – With Resilience (no σ)")

    panel_labels_1 = {ax_lag:"a", ax_te:"b", ax_mf_nores:"c", ax_pc_nores:"d", ax_mf_with:"e", ax_pc_with:"f"}
    for ax, lab in panel_labels_1.items():
        ax.text(-0.10, 1.15, lab, transform=ax.transAxes, fontsize=14, fontweight="bold", va="top", ha="left")

    fig1.subplots_adjust(left=0.07, right=0.98, bottom=0.06, top=0.93, hspace=0.55, wspace=0.35)
    out1 = os.path.join(outdir, fname_abf)
    fig1.savefig(_prefix_output(out1), dpi=600, bbox_inches="tight")
    print("✅ Salvato:", out1)
    plt.close(fig1)

    fig2, axes2 = plt.subplots(2, 2, figsize=(12, 12))
    ax_te_pc    = axes2[0, 0]
    ax_te_slope = axes2[0, 1]
    #ax_struct   = axes2[1, 0]
    ax_te = axes2[1,1]
    #axes2[1, 1].axis("off")

    for ax in (ax_te_pc, ax_te_slope, ax_te):
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    stats_pc = {}
    for res, cmap in [("Without Resilience","Blues_r"), ("With Resilience","Reds_r")]:
        rng = sigma_ranges.get(res, None)
        if rng is None:
            continue
        smin, smax = rng["sigma_min"], rng["sigma_max"]
        df_range = df[(df["resilience"] == res) & (df["sigma"] >= smin) & (df["sigma"] <= smax)].copy()
        if df_range.empty:
            continue
        te_vals, peak_vals, sz_vals = _compute_te_and_peak(df_range, "purkinje_timeseries")
        if len(peak_vals) == 0:
            continue
        if np.any(sz_vals == 0):
            baseline_peak = np.mean(peak_vals[sz_vals == 0])
            baseline_te   = np.mean(te_vals[sz_vals == 0])
        else:
            idx0 = np.argmin(sz_vals)
            baseline_peak = peak_vals[idx0]
            baseline_te   = te_vals[idx0]
        peak_delta = peak_vals - baseline_peak
        te_delta   = te_vals   - baseline_te
        rho, pval = spearmanr(te_delta, peak_delta) if len(te_delta) > 1 else (np.nan, np.nan)
        stats_pc[res] = (rho, pval)

        sz_uniq = np.sort(np.unique(sz_vals))
        means_te, means_peak, std_te, std_peak, sz_for_color = [], [], [], [], []
        for sz0 in sz_uniq:
            m = (sz_vals == sz0)
            if np.sum(m) == 0:
                continue
            means_te.append(np.mean(te_delta[m]))
            means_peak.append(np.mean(peak_delta[m]))
            std_te.append(np.std(te_delta[m], ddof=1) if np.sum(m) > 1 else 0.0)
            std_peak.append(np.std(peak_delta[m], ddof=1) if np.sum(m) > 1 else 0.0)
            sz_for_color.append(sz0)

        if not means_te:
            continue

        means_te = np.array(means_te); means_peak = np.array(means_peak)
        std_te = np.array(std_te); std_peak = np.array(std_peak)
        sz_for_color = np.array(sz_for_color)

        ax_te_pc.scatter(means_te, means_peak, c=sz_for_color, cmap=cmap, s=60, edgecolor="k", linewidth=0.4, alpha=0.9, label=res, zorder=4)
        ax_te_pc.errorbar(means_te, means_peak, xerr=std_te, yerr=std_peak, fmt="none", ecolor="k", elinewidth=1, capsize=2, alpha=0.25, zorder=2)

    ax_te_pc.axhline(0, color="k", linestyle="--", linewidth=0.8, alpha=0.8)
    ax_te_pc.axvline(0, color="k", linestyle="--", linewidth=0.8, alpha=0.8)
    ax_te_pc.set_xlabel("Δ TE MF→PC (bit)")
    ax_te_pc.set_ylabel("Δ Peak PC FR (Hz)")
    ax_te_pc.grid(False)

    y0 = 1.10
    for res, colr in [("Without Resilience","k"), ("With Resilience","k")]:
        if res in stats_pc and not np.isnan(stats_pc[res][0]):
            rho, pval = stats_pc[res]
            ax_te_pc.text(0.5, y0, f"{res}: ρ={rho:.2f}, p={pval:.2e}", transform=ax_te_pc.transAxes, fontsize=12, color=colr, va="bottom", ha="center")
            y0 -= 0.08

    stats_slope = {}
    for res, cmap in [("Without Resilience","Blues_r"), ("With Resilience","Reds_r")]:
        rng = sigma_ranges.get(res, None)
        if rng is None:
            continue
        smin, smax = rng["sigma_min"], rng["sigma_max"]
        df_range = df[(df["resilience"] == res) & (df["sigma"] >= smin) & (df["sigma"] <= smax)].copy()
        te_vals, slope_vals, sz_vals = _compute_te_and_slope(df_range)
        if len(slope_vals) == 0:
            continue
        if np.any(sz_vals == 0):
            baseline_te = np.mean(te_vals[sz_vals == 0])
            baseline_slope = np.mean(slope_vals[sz_vals == 0])
        else:
            idx0 = np.argmin(sz_vals)
            baseline_te = te_vals[idx0]
            baseline_slope = slope_vals[idx0]
        te_delta = te_vals - baseline_te
        slope_delta = slope_vals - baseline_slope
        rho, pval = spearmanr(te_delta, slope_delta) if len(te_delta) > 1 else (np.nan, np.nan)
        stats_slope[res] = (rho, pval)

        sz_uniq = np.sort(np.unique(sz_vals))
        means_te, means_sl, std_te, std_sl, sz_for_color = [], [], [], [], []
        for sz0 in sz_uniq:
            m = (sz_vals == sz0)
            if np.sum(m) == 0:
                continue
            means_te.append(np.mean(te_delta[m]))
            means_sl.append(np.mean(slope_delta[m]))
            std_te.append(np.std(te_delta[m], ddof=1) if np.sum(m) > 1 else 0.0)
            std_sl.append(np.std(slope_delta[m], ddof=1) if np.sum(m) > 1 else 0.0)
            sz_for_color.append(sz0)

        if not means_te:
            continue

        means_te = np.array(means_te); means_sl = np.array(means_sl)
        std_te = np.array(std_te); std_sl = np.array(std_sl)
        sz_for_color = np.array(sz_for_color)

        ax_te_slope.scatter(means_te, means_sl, c=sz_for_color, cmap=cmap, s=60, edgecolor="k", linewidth=0.4, alpha=0.9, label=res, zorder=4)
        ax_te_slope.errorbar(means_te, means_sl, xerr=std_te, yerr=std_sl, fmt="none", ecolor="k", elinewidth=1, capsize=2, alpha=0.25, zorder=2)

    ax_te_slope.axhline(0, color="k", linestyle="--", linewidth=0.8, alpha=0.8)
    ax_te_slope.axvline(0, color="k", linestyle="--", linewidth=0.8, alpha=0.8)
    ax_te_slope.set_xlabel("Δ TE MF→PC (bit)")
    ax_te_slope.set_ylabel("Δ Purkinje aperiodic slope \n (proxy of E/I balance)")
    ax_te_slope.grid(False)

    y0 = 1.10
    for res, colr in [("Without Resilience","k"), ("With Resilience","k")]:
        if res in stats_slope and not np.isnan(stats_slope[res][0]):
            rho, pval = stats_slope[res]
            ax_te_slope.text(0.5, y0, f"{res}: ρ={rho:.2f}, p={pval:.2e}", transform=ax_te_slope.transAxes, fontsize=12, color=colr, va="bottom", ha="center")
            y0 -= 0.08

    cmap_te = plt.cm.viridis
    te_all = np.array([])
    df_te_struct = None

    if df_all_str is None or df_te is None or norm_res is None:
        ax_te.text(0.5, 0.5, "STRUCT SKIPPED\n(pass df_all_str, df_te, norm_res)", transform=ax_te.transAxes, ha="center", va="center", fontsize=10)
        ax_te.axis("off")
    else:
        try:
            df_str = df_all_str.copy()
            df_str["Resilience"] = df_str["Resilience"].apply(norm_res)
            df_str["SZ Factor (%)"] = df_str["SZ Factor (%)"].astype(float).round(2)

            df_te_struct = df_te.copy()
            df_te_struct["Resilience"] = df_te_struct["Resilience"].apply(norm_res)
            df_te_struct["SZ Factor (%)"] = df_te_struct["SZ Factor (%)"].astype(float).round(2)

            df_te_struct["dte_mean"] = np.nan
            for res in df_te_struct["Resilience"].unique():
                sub = df_te_struct[df_te_struct["Resilience"] == res].sort_values("SZ Factor (%)")
                if sub.empty:
                    continue
                base = sub.loc[sub["SZ Factor (%)"] == 0, "te_mean"].mean() if np.any(sub["SZ Factor (%)"] == 0) else sub["te_mean"].iloc[0]
                m = (df_te_struct["Resilience"] == res)
                df_te_struct.loc[m, "dte_mean"] = df_te_struct.loc[m, "te_mean"] - base

            te_all = df_te_struct["dte_mean"].astype(float).values

            def sum_unique_synapses(x):
                return np.unique(x).sum()

            df_struct = (
                df_str.groupby(["SZ Factor (%)", "Resilience"], as_index=False)
                .agg({"total_cells": "first", "synapses": sum_unique_synapses})
            )

            df_baseline = df_struct[df_struct["SZ Factor (%)"] == 0].set_index("Resilience")
            rows = []
            for _, row in df_struct.iterrows():
                res = row["Resilience"]
                base_cells = df_baseline.loc[res, "total_cells"]
                base_syn = df_baseline.loc[res, "synapses"]
                rows.append({
                    "Resilience": res,
                    "SZ Factor (%)": row["SZ Factor (%)"],
                    "cells_pct": 100*(row["total_cells"]-base_cells)/base_cells,
                    "syn_pct": 100*(row["synapses"]-base_syn)/base_syn,
                })
            df_struct_pct = pd.DataFrame(rows)

            df_plot_struct = df_struct_pct.merge(
                df_te_struct[["SZ Factor (%)", "Resilience", "dte_mean"]],
                on=["SZ Factor (%)", "Resilience"],
                how="inner"
            )

            dte_min = np.nanmin(te_all) if te_all.size > 0 else 0.0
            dte_max = np.nanmax(te_all) if te_all.size > 0 else 1.0
            norm_dte = Normalize(vmin=dte_min, vmax=dte_max)

            marker_map_struct = {"With Resilience": "s", "Without Resilience": "o"}
            for res in ["With Resilience", "Without Resilience"]:
                sub = df_plot_struct[df_plot_struct["Resilience"] == res]
                if sub.empty:
                    continue
                colors_s = cmap_te(norm_dte(sub["dte_mean"].values))
                ax_te.scatter(sub["cells_pct"], sub["syn_pct"], c=colors_s, s=70, edgecolor="k", linewidth=0.6, marker=marker_map_struct[res], label=res)

            ax_te.set_xlabel("Δ Cells (%)")
            ax_te.set_ylabel("Δ Synapses (%)")
            ax_te.legend(title="Resilience", loc="upper left", fontsize=8)
            ax_te.axhspan(-30, -60, facecolor="k", alpha=0.2, zorder=-2)
        except Exception as e:
            ax_te.text(0.5, 0.5, f"STRUCT ERROR:\n{e}", transform=ax_te.transAxes, ha="center", va="center", fontsize=10)
            ax_te.axis("off")

    fig2.subplots_adjust(left=0.07, right=0.98, bottom=0.10, top=0.93, hspace=0.55, wspace=0.35)

    pos_g = ax_te_pc.get_position()
    
    x_shift = pos_g.x0
    y_base  = pos_g.y0 - 0.38 - 0.15
    
    total_h = pos_g.height * 1.25
    gap_y   = 0.12 * total_h
    h_sub   = (total_h - gap_y) / 2.0
    
    shrink_w = 0.03*2.5          # 3% più stretto (puoi provare 0.02–0.04)
    new_w    = pos_g.width * (1 - shrink_w)
    
    ax_mfpc_no = fig2.add_axes([
        x_shift,
        y_base + h_sub + gap_y,
        new_w,
        h_sub
    ])
    
    ax_mfpc_wi = fig2.add_axes([
        x_shift,
        y_base,
        new_w,
        h_sub
    ], sharex=ax_mfpc_no)


    def _plot_mf_pc_peak_relation(df_range, res_label, ax, title, ecolor):
        sz_mf, mf_mean, mf_std = _compute_peak_vs_sz(df_range, "mossy_fibers_timeseries")
        sz_pc, pc_mean, pc_std = _compute_peak_vs_sz(df_range, "purkinje_timeseries")
        if len(sz_mf) == 0 or len(sz_pc) == 0:
            ax.text(0.5, 0.5, "NO DATA", transform=ax.transAxes, ha="center", va="center", fontsize=10)
            ax.axis("off")
            return

        base_mf = np.mean(mf_mean[sz_mf == 0]) if np.any(sz_mf == 0) else mf_mean[np.argmin(sz_mf)]
        base_pc = np.mean(pc_mean[sz_pc == 0]) if np.any(sz_pc == 0) else pc_mean[np.argmin(sz_pc)]
        mf_delta = mf_mean - base_mf
        pc_delta = pc_mean - base_pc

        df_mf = pd.DataFrame({"SZ": sz_mf, "mf_delta": mf_delta, "mf_std": mf_std})
        df_pc = pd.DataFrame({"SZ": sz_pc, "pc_delta": pc_delta, "pc_std": pc_std})
        df_mf["SZ_round"] = df_mf["SZ"].round(2)
        df_pc["SZ_round"] = df_pc["SZ"].round(2)
        df_peaks = df_mf.merge(df_pc, on="SZ_round", how="inner").rename(columns={"SZ_round": "SZ"}).sort_values("SZ")
        if df_peaks.empty:
            ax.text(0.5, 0.5, "NO COMMON SZ", transform=ax.transAxes, ha="center", va="center", fontsize=10)
            ax.axis("off")
            return

        x = df_peaks["mf_delta"].to_numpy(dtype=float)
        y = df_peaks["pc_delta"].to_numpy(dtype=float)
        xerr = df_peaks["mf_std"].to_numpy(dtype=float)
        yerr = df_peaks["pc_std"].to_numpy(dtype=float)

        dte_vals = None
        if df_te_struct is not None:
            te_sub = df_te_struct[df_te_struct["Resilience"] == res_label][["SZ Factor (%)", "dte_mean"]].rename(columns={"SZ Factor (%)": "SZ"})
            te_sub["SZ"] = te_sub["SZ"].round(2)
            df_tmp = df_peaks.merge(te_sub, on="SZ", how="left")
            dte_vals = df_tmp["dte_mean"].to_numpy(dtype=float)

        norm_te = Normalize(vmin=np.nanmin(te_all) if te_all.size else 0.0, vmax=np.nanmax(te_all) if te_all.size else 1.0)
        colors = cmap_te(norm_te(dte_vals)) if (dte_vals is not None and np.any(np.isfinite(dte_vals))) else cmap_te(norm_te(np.zeros_like(x)))

        ax.errorbar(x, y, xerr=xerr, yerr=yerr, fmt="none", ecolor="k", elinewidth=1.0, capsize=2, alpha=0.25, zorder=1)
        ax.scatter(x, y, c=colors, s=60, edgecolor="k", linewidth=0.4, alpha=0.9, zorder=2)

        ax.axhline(0, color="k", linestyle="--", linewidth=0.8, alpha=0.8)
        ax.axvline(0, color="k", linestyle="--", linewidth=0.8, alpha=0.8)

        rho, pval = spearmanr(x, y) if len(x) > 1 else (np.nan, np.nan)
        ax.set_title(f"{title}: ρ={rho:.2f}, p={pval:.2e}" if not np.isnan(rho) else title, fontsize=12, color=ecolor)
        ax.grid(False)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    _plot_mf_pc_peak_relation(df_no_range, "Without Resilience", ax_mfpc_no, "Without Resilience", "k")
    _plot_mf_pc_peak_relation(df_wi_range, "With Resilience", ax_mfpc_wi, "With Resilience", "k")

    ax_mfpc_no.tick_params(labelbottom=False)
    ax_mfpc_wi.set_xlabel("Δ Peak MF FR (Hz)")

    ymin = min(ax_mfpc_no.get_ylim()[0], ax_mfpc_wi.get_ylim()[0])
    ymax = max(ax_mfpc_no.get_ylim()[1], ax_mfpc_wi.get_ylim()[1])
    ax_mfpc_no.set_ylim(ymin, ymax)
    ax_mfpc_wi.set_ylim(ymin, ymax)

    pos_no = ax_mfpc_no.get_position()
    pos_wi = ax_mfpc_wi.get_position()
    fig2.text(pos_no.x0 - 0.05, (pos_no.y1 + pos_wi.y0)/2.0, "Δ Peak PC FR (Hz)", rotation="vertical", va="center", ha="center")

    for ax in (ax_mfpc_no, ax_mfpc_wi):
        x_min, x_max = ax.get_xlim()
        y_min, y_max = ax.get_ylim()
        x0_rel = np.clip((0 - x_min) / (x_max - x_min), 0, 1)
        y0_rel = np.clip((0 - y_min) / (y_max - y_min), 0, 1)
        ax.add_patch(plt.Rectangle((x0_rel, y0_rel), 1-x0_rel, 1-y0_rel, transform=ax.transAxes, facecolor="#fff7b2", edgecolor="none", alpha=0.25, zorder=-3))

    for ax in (ax_te_pc, ax_te_slope):
        x_min, x_max = ax.get_xlim()
        y_min, y_max = ax.get_ylim()
        if (x_max - x_min) == 0 or (y_max - y_min) == 0:
            continue
        x0_rel = np.clip((0 - x_min) / (x_max - x_min), 0, 1)
        y0_rel = np.clip((0 - y_min) / (y_max - y_min), 0, 1)
        ax.add_patch(plt.Rectangle((x0_rel, y0_rel), 1-x0_rel, 1-y0_rel, transform=ax.transAxes, facecolor="#fff7b2", edgecolor="none", alpha=0.25, zorder=-3))

    panel_labels_2 = {ax_te_pc:"a", ax_te_slope:"b", ax_te:"e", ax_mfpc_no:"c", ax_mfpc_wi:"d"}
    for ax, lab in panel_labels_2.items():
        ax.text(-0.10, 1.15, lab, transform=ax.transAxes, fontsize=14, fontweight="bold", va="top", ha="left")

    if te_all.size > 0:
        norm_te = Normalize(vmin=np.nanmin(te_all), vmax=np.nanmax(te_all))
        sm_te = ScalarMappable(norm=norm_te, cmap=cmap_te)
        sm_te.set_array(te_all)
        pos_no = ax_mfpc_no.get_position()
        pos_wi = ax_mfpc_wi.get_position()
        x_cbar = pos_no.x1 + 0.02
        y_cbar = pos_wi.y0
        h_cbar = pos_no.y1 - pos_wi.y0
        w_cbar = 0.018
        cax_te = fig2.add_axes([x_cbar, y_cbar, w_cbar, h_cbar])
        cb_te = fig2.colorbar(sm_te, cax=cax_te, orientation="vertical")
        cb_te.ax.set_title("Δ TE MF→PC \n (bit)", fontsize=14, pad=10)
        cb_te.ax.tick_params(labelsize=14)

    axes2[1,0].remove()

    # =========================
    # CENTRAL "HAT" COLORBARS
    # =========================
    
    # bounding box dei due pannelli a–b
    pos_a = ax_te_pc.get_position()
    pos_b = ax_te_slope.get_position()
    
    # centro orizzontale dei due pannelli
    x_center = 0.5 * (pos_a.x0 + pos_b.x1)
    
    # larghezza del cappello (70% del totale a–b)
    hat_width = 0.70 * (pos_b.x1 - pos_a.x0)
    x0 = x_center - hat_width / 2
    
    # altezze e spazi
    bar_h   = 0.018     # più sottile
    gap     = 0.006*4     # spazio tra le due barre
    y_top   = max(pos_a.y1, pos_b.y1) + 0.065*2.5  # più in alto (chiave!)
    
    # assi colorbar
    cax_red  = fig2.add_axes([x0, y_top,              hat_width, bar_h])
    cax_blue = fig2.add_axes([x0, y_top - bar_h - gap, hat_width, bar_h])
    
    # normalizzazione SZ
    norm_sz = Normalize(
        vmin=df[x_col].min(),
        vmax=df[x_col].max()
    )
    
    # mappabili
    sm_red  = ScalarMappable(norm=norm_sz, cmap="Reds_r")
    sm_blue = ScalarMappable(norm=norm_sz, cmap="Blues_r")
    
    # colorbar
    cb_red = fig2.colorbar(sm_red,  cax=cax_red,  orientation="horizontal")
    cb_blue = fig2.colorbar(sm_blue, cax=cax_blue, orientation="horizontal")
    
    # titoli delle barre
    cb_red.ax.set_title("With Resilience", fontsize=12, color="k", pad=2)
    cb_blue.ax.set_title("Without Resilience", fontsize=12, color="k", pad=2)
    
    # label comune (solo sotto)
    cb_blue.set_label("SZ Factor (%)", fontsize=14)
    
    # tick più piccoli
    cb_red.ax.tick_params(labelsize=10)
    cb_blue.ax.tick_params(labelsize=10)

    cb_red.ax.tick_params(
    bottom=False,
    top=False,
    labelbottom=False
    )

    cb_red.ax.tick_params(labelsize=9)


    out2 = os.path.join(outdir, fname_ghijk)
    fig2.savefig(_prefix_output(out2), dpi=600, bbox_inches="tight")
    print("✅ Salvato:", out2)
    plt.show()

plot_lag_te_and_epoched_5x2(
    df_all,
    sigma_ranges=sigma_ranges,
    x_col="SZ Factor (%)",
    outdir="./tempPlots",
    fname_abf="lag_te_epoched_a_f.png",
    fname_ghijk="lag_te_epoched_g_k.png",
    use_delta=True,
    alpha=0.001,
    dt=0.1,
    window_ms=(-70, 200),
    df_all_str=df_all_str,
    df_te=df_te,
    norm_res=norm_res
)


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

def norm_res(val):
    s = str(val).strip()
    if s in ["15", "With Resilience"]:
        return "With Resilience"
    if s in ["2", "Without Resilience"]:
        return "Without Resilience"
    return s


def plot_second_derivative_panels_slim(
        df_te,
        outdir="./tempPlots",
        fname="second_derivative_two_slim_panels.png"
    ):

    os.makedirs(outdir, exist_ok=True)

    # ---------- PREPARAZIONE ----------
    df = df_te.copy()
    df["Resilience"]    = df["Resilience"].apply(norm_res)
    df["SZ Factor (%)"] = df["SZ Factor (%)"].astype(float)

    res_levels = ["Without Resilience", "With Resilience"]
    markers = {"Without Resilience": "o", "With Resilience": "s"}

    # Calcolo derivata seconda
    rows = []
    for res in res_levels:
        sub = df[df["Resilience"] == res].sort_values("SZ Factor (%)")
        x = sub["SZ Factor (%)"].to_numpy(float)
        y = sub["te_mean"].to_numpy(float)

        if len(x) < 3:
            continue

        dydx = np.gradient(y, x)
        d2 = np.gradient(dydx, x)

        for xi, yi, d2v in zip(x, y, d2):
            rows.append({
                "Resilience": res,
                "SZ Factor (%)": xi,
                "TE_mean": yi,
                "d2_TE": d2v
            })

    df_dd = pd.DataFrame(rows)

    # ---------- PLOT: due pannelli stretti ----------
    fig, axes = plt.subplots(
        1, 2,
        figsize=(9, 3.2),    # <-- PANNELLI MOLTO STRETTI
        sharey=True
    )

    norm = Normalize(vmin=df_dd["TE_mean"].min(), vmax=df_dd["TE_mean"].max())
    cmap = plt.cm.viridis

    for i, res in enumerate(res_levels):
        ax = axes[i]
        sub = df_dd[df_dd["Resilience"] == res]

        ax.plot(
            sub["SZ Factor (%)"], sub["d2_TE"],
            color="gray", alpha=0.4, linewidth=1.0
        )

        ax.scatter(
            sub["SZ Factor (%)"], sub["d2_TE"],
            c=cmap(norm(sub["TE_mean"].values)),
            s=90,
            marker=markers[res],
            edgecolor="black",
            linewidth=0.5,
            alpha=0.95
        )

        ax.axhline(0, color="k", linestyle="--", linewidth=1, alpha=0.7)
        ax.set_title(res)
        ax.set_xlabel("SZ (%)")
        if i == 0:
            ax.set_ylabel("d²TE / d(SZ)²")

        ax.grid(True, linestyle="--", alpha=0.25)

    # Colorbar comune sotto
    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = fig.colorbar(
        sm,
        ax=axes,
        orientation="horizontal",
        fraction=0.1,
        pad=0.2
    )
    cbar.set_label("TE_mean (bit)")

    plt.tight_layout()
    outpath = os.path.join(outdir, fname)
    plt.savefig(_prefix_output(outpath), dpi=300)
    plt.show()
    print("✔ Saved:", outpath)

    return df_dd


# --- Uso ---
df_dd = plot_second_derivative_panels_slim(df_te)


In [ ]:
df_all_weight.keys().sort_values()

In [ ]:
df_all_weight['SZ Factor (%)'] = 100-(df_all_weight['k']*100)
df_all_weight['connection'].unique()
# Lista connessioni IO-related
io_conn = [
'basket_to_basket'
]

# Filtra via tutte le connessioni che compaiono in io_conn
df_all_weight[df_all_weight["connection"].isin(io_conn)].copy()


In [ ]:
df_all_str[(df_all_str["Resilience"] == '2') & (df_all_str['synType']=='excitatory (glutamate)')][["weight", "Resilience", "synType"]]


In [ ]:
import numpy as np
import pandas as pd

def norm_res(val):
    s = str(val).strip()
    if s in ["15", "With Resilience"]:
        return "With Resilience"
    if s in ["2", "Without Resilience"]:
        return "Without Resilience"
    return s

# ==========================
# 1) Prepara df e somma pesi
# ==========================
df = df_all_str.copy()
df["Resilience"]    = df["Resilience"].apply(norm_res)
df["SZ Factor (%)"] = df["SZ Factor (%)"].astype(float)

# Somma pesi eccitatori
exc_w = (
    df[df["synType"] == "excitatory (glutamate)"]
    .groupby(["SZ Factor (%)", "Resilience"], as_index=False)["weight"]
    .sum()
    .rename(columns={"weight": "w_exc"})
)

# Somma pesi inibitori
inh_w = (
    df[df["synType"] == "inhibitory (GABA)"]
    .groupby(["SZ Factor (%)", "Resilience"], as_index=False)["weight"]
    .sum()
    .rename(columns={"weight": "w_inh"})
)

# Merge
df_wEI = exc_w.merge(
    inh_w,
    on=["SZ Factor (%)", "Resilience"],
    how="outer"
).fillna(0.0)

# Rapporto E/I
df_wEI["EI_ratio"] = df_wEI["w_exc"] / df_wEI["w_inh"].replace(0, np.nan)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

def norm_res(val):
    s = str(val).strip()
    if s in ["15", "With Resilience"]:
        return "With Resilience"
    if s in ["2", "Without Resilience"]:
        return "Without Resilience"
    return s

# Ordino come fai tu
A = df_wEI.sort_values(['Resilience','SZ Factor (%)'])
B = df_te.sort_values(['Resilience','SZ Factor (%)'])

te = B.copy()
w  = A.copy()

# Normalizzazione
for df in (te, w):
    df["Resilience"]    = df["Resilience"].apply(norm_res)
    df["SZ Factor (%)"] = df["SZ Factor (%)"].astype(float).round(2)

# Merge pulito
df_merge = te.merge(
    w,
    on=["SZ Factor (%)", "Resilience"],
    how="inner"
)

# ——————————————————————————
#  PALETTE custom RESILIENCE
# ——————————————————————————
def color_from_resilience(res, sz):
    if res == "With Resilience":
        cmap = plt.cm.Reds
    else:
        cmap = plt.cm.Blues
    sz_norm = (sz - df_merge["SZ Factor (%)"].min()) / \
              (df_merge["SZ Factor (%)"].max() - df_merge["SZ Factor (%)"].min())
    return cmap(sz_norm)

df_merge["color"] = [
    color_from_resilience(r, sz)
    for r, sz in zip(df_merge["Resilience"], df_merge["SZ Factor (%)"])
]

# ——————————————————————————
#        PLOT
# ——————————————————————————
plt.figure(figsize=(9,7))

plt.scatter(
    df_merge["te_mean"],
    df_merge["EI_ratio"],
    c=df_merge["color"],
    s=150,
    edgecolors="black",
    linewidth=0.7
)

# Aggiungi etichette SZ vicino ai punti
for _, row in df_merge.iterrows():
    plt.text(
        row["te_mean"] + 1,          # piccolo offset X
        row["EI_ratio"] + 0.005,     # piccolo offset Y
        f"{row['SZ Factor (%)']}",
        fontsize=8,
        ha='left',
        va='bottom'
    )

plt.xlabel("TE_mean (bit)")
plt.ylabel("E/I ratio (Σ w_exc / Σ w_inh)")
plt.title("E/I weight ratio vs TE_mean\nRed = With Resilience, Blue = Without Resilience")

# legende artificiali
import matplotlib.patches as mpatches
red_patch = mpatches.Patch(color=plt.cm.Reds(0.6), label="With Resilience")
blue_patch = mpatches.Patch(color=plt.cm.Blues(0.6), label="Without Resilience")
plt.legend(handles=[red_patch, blue_patch], title="Resilience")

plt.tight_layout()
plt.show()


In [ ]:
sns.scatterplot(data=df_merge, x='w_exc', y='w_inh')
df_merge

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def norm_res(val):
    s=str(val).strip()
    if s in ["15","With Resilience"]:
        return "With Resilience"
    if s in ["2","Without Resilience"]:
        return "Without Resilience"
    return s

def plot_SZ_vs_EIratio(
        df_all_str,
        measure_col="effective_convergence",
        target_connections=None,
        outdir="./tempPlots",
        fname_prefix="SZ_vs_EIratio"
    ):
    """
    Calcola, per ciascuna combinazione SZ×Resilience:
      - somma eccitatoria del 'measure_col'
      - somma inibitoria del 'measure_col'
      - rapporto E/I = exc / inh
    e plottat SZ Factor (%) vs E/I in due pannelli (With / Without Resilience).
    
    measure_col può essere ad es.:
      - "effective_convergence"
      - "effective_synapses"
      - "synapses"
    """
    os.makedirs(outdir,exist_ok=True)

    df=df_all_str.copy()
    df["Resilience"]=df["Resilience"].apply(norm_res)
    df["SZ Factor (%)"]=df["SZ Factor (%)"].astype(float)

    if target_connections is not None:
        df=df[df["connection"].isin(target_connections)].copy()

    cols_needed=[measure_col,"connection","SZ Factor (%)","Resilience","synType"]
    for c in cols_needed:
        if c not in df.columns:
            raise ValueError(f"Column '{c}' not found in df_all_str")

    def sum_unique(x):
        return np.unique(x).sum()

    exc_sum=(
        df[df["synType"]=="excitatory (glutamate)"]
        .groupby(["SZ Factor (%)","Resilience"],as_index=False)
        .agg({measure_col:sum_unique})
        .rename(columns={measure_col:"exc_val"})
    )

    inh_sum=(
        df[df["synType"]=="inhibitory (GABA)"]
        .groupby(["SZ Factor (%)","Resilience"],as_index=False)
        .agg({measure_col:sum_unique})
        .rename(columns={measure_col:"inh_val"})
    )

    df_EI=exc_sum.merge(
        inh_sum,
        on=["SZ Factor (%)","Resilience"],
        how="outer"
    ).fillna(0)

    df_EI["EI_ratio"]=df_EI["exc_val"]/df_EI["inh_val"].replace(0,np.nan)

    df_with=df_EI[df_EI["Resilience"]=="With Resilience"]
    df_without=df_EI[df_EI["Resilience"]=="Without Resilience"]

    fig,axs=plt.subplots(1,2,figsize=(14,6),sharey=True)

    sns.scatterplot(
        data=df_with,
        x="SZ Factor (%)",
        y="EI_ratio",
        hue="exc_val",
        palette="Reds",
        s=120,
        edgecolor="black",
        linewidth=0.6,
        ax=axs[0]
    )
    axs[0].set_title("With Resilience")
    axs[0].set_xlabel("SZ Factor (%)")
    axs[0].set_ylabel(f"Exc/Inh ratio ({measure_col})")

    sns.scatterplot(
        data=df_without,
        x="SZ Factor (%)",
        y="EI_ratio",
        hue="exc_val",
        palette="Blues",
        s=120,
        edgecolor="black",
        linewidth=0.6,
        ax=axs[1]
    )
    axs[1].set_title("Without Resilience")
    axs[1].set_xlabel("SZ Factor (%)")
    axs[1].set_ylabel("")

    vmin=df_EI["exc_val"].min()
    vmax=df_EI["exc_val"].max()
    norm=plt.Normalize(vmin,vmax)
    sm=plt.cm.ScalarMappable(norm=norm,cmap="viridis")
    sm.set_array([])
    cbar=fig.colorbar(sm,ax=axs,shrink=0.8)
    cbar.set_label(f"Excitatory sum ({measure_col})")

    plt.tight_layout()
    fname=f"{fname_prefix}_{measure_col}.png"
    outpath=os.path.join(outdir,fname)
    plt.savefig(_prefix_output(outpath),dpi=300)
    print("Saved:",outpath)
    plt.show()

    return df_EI


In [ ]:
pc_conn=[
    "ascending_axon_to_purkinje",
    "basket_to_purkinje",
    "basket_to_basket",
    "parallel_fiber_to_purkinje",
    "stellate_to_purkinje",
    "stellate_to_stellate"
]

# 1) Usando la convergence
df_EI_conv = plot_SZ_vs_EIratio(
    df_all_str,
    measure_col="effective_convergence",
    target_connections=pc_conn,
    outdir="./tempPlots",
    fname_prefix="SZ_vs_EIratio_PC"
)

# 2) Usando le effective synapses
df_EI_effsyn = plot_SZ_vs_EIratio(
    df_all_str,
    measure_col="effective_synapses",
    target_connections=pc_conn,
    outdir="./tempPlots",
    fname_prefix="SZ_vs_EIratio_PC"
)

# 2) Usando le effective synapses
df_EI_effsyn = plot_SZ_vs_EIratio(
    df_all_str,
    measure_col="effective_divergence",
    target_connections=pc_conn,
    outdir="./tempPlots",
    fname_prefix="SZ_vs_EIratio_PC"
)


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def norm_res(val):
    s = str(val).strip()
    if s in ["15", "With Resilience"]:
        return "With Resilience"
    if s in ["2", "Without Resilience"]:
        return "Without Resilience"
    return s

def scatter_exc_inh_colorSZ(
        df_all_str,
        measure_col="effective_convergence",
        outdir="./tempPlots",
        fname="scatter_exc_inh_colorSZ.png"
    ):
    """
    Usa TUTTE le connessioni della rete.
    Raggruppa per (SZ, Resilience), somma Eccitatorie e Inibitorie.
    Scatter Exc vs Inh colorato per SZ, con marker = Resilience.
    """
    os.makedirs(outdir, exist_ok=True)

    df = df_all_str.copy()
    df["Resilience"]    = df["Resilience"].apply(norm_res)
    df["SZ Factor (%)"] = df["SZ Factor (%)"].astype(float)

    needed = [measure_col, "SZ Factor (%)", "Resilience", "synType"]
    for c in needed:
        if c not in df.columns:
            raise ValueError(f"Column '{c}' not found in df_all_str")

    def sum_unique(x):
        return np.unique(x).sum()

    # ECCITATORIE
    exc_sum = (
        df[df["synType"] == "excitatory (glutamate)"]
        .groupby(["SZ Factor (%)", "Resilience"], as_index=False)
        .agg({measure_col: sum_unique})
        .rename(columns={measure_col: "exc_val"})
    )

    # INIBITORIE
    inh_sum = (
        df[df["synType"] == "inhibitory (GABA)"]
        .groupby(["SZ Factor (%)", "Resilience"], as_index=False)
        .agg({measure_col: sum_unique})
        .rename(columns={measure_col: "inh_val"})
    )

    # Merge
    df_EI = exc_sum.merge(inh_sum,
                          on=["SZ Factor (%)", "Resilience"],
                          how="outer").fillna(0)

    # Marker mapping
    marker_map = {
        "With Resilience": "o",
        "Without Resilience": "s"
    }

    # ----------- PLOT -----------
    plt.figure(figsize=(10, 8))

    for res in df_EI["Resilience"].unique():
        sub = df_EI[df_EI["Resilience"] == res]

        plt.scatter(
            sub["exc_val"],
            sub["inh_val"],
            c=sub["SZ Factor (%)"],
            cmap="viridis",
            s=140,
            edgecolor="black",
            linewidth=0.7,
            marker=marker_map.get(res, "o"),
            label=res,
            alpha=0.9,
        )

    plt.xlabel(f"Excitatory sum ({measure_col})")
    plt.ylabel(f"Inhibitory sum ({measure_col})")
    plt.title("Global Exc vs Inh – colored by SZ Factor")

    # Colorbar
    cbar = plt.colorbar()
    cbar.set_label("SZ Factor (%)")

    # Legend per Resilience
    plt.legend(title="Resilience")

    plt.tight_layout()
    outpath = os.path.join(outdir, fname)
    plt.savefig(_prefix_output(outpath), dpi=300)
    plt.show()
    print("Saved:", outpath)

    return df_EI


scatter_exc_inh_colorSZ(df_all_str, measure_col="synapses")
scatter_exc_inh_colorSZ(df_all_str, measure_col="effective_synapses")

scatter_exc_inh_colorSZ(df_all_str, measure_col="convergence")
scatter_exc_inh_colorSZ(df_all_str, measure_col="divergence")


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def norm_res(val):
    s = str(val).strip()
    if s in ["15", "With Resilience"]:
        return "With Resilience"
    if s in ["2", "Without Resilience"]:
        return "Without Resilience"
    return s

def scatter_exc_inh_colorTE(
        df_all_str,
        df_te,
        measure_col="effective_convergence",
        outdir="./tempPlots",
        fname="scatter_exc_inh_colorTE.png"
    ):
    """
    Usa TUTTE le connessioni.
    Raggruppa eccitatorie/inibitorie per SZ × Resilience.
    Colore = TE_mean.
    Marker = Resilience.
    """
    os.makedirs(outdir, exist_ok=True)

    # --- copia e normalizza ---
    df = df_all_str.copy()
    df["Resilience"]    = df["Resilience"].apply(norm_res)
    df["SZ Factor (%)"] = df["SZ Factor (%)"].astype(float)

    df_te2 = df_te.copy()
    df_te2["Resilience"]    = df_te2["Resilience"].apply(norm_res)
    df_te2["SZ Factor (%)"] = df_te2["SZ Factor (%)"].astype(float)

    # --- check colonne ---
    needed = [measure_col, "SZ Factor (%)", "Resilience", "synType"]
    for c in needed:
        if c not in df.columns:
            raise ValueError(f"Column '{c}' not found in df_all_str")

    # ---- SUM UNIQUE ----
    def sum_unique(x):
        return np.unique(x).sum()

    exc_sum = (
        df[df["synType"] == "excitatory (glutamate)"]
        .groupby(["SZ Factor (%)", "Resilience"], as_index=False)
        .agg({measure_col: sum_unique})
        .rename(columns={measure_col: "exc_val"})
    )

    inh_sum = (
        df[df["synType"] == "inhibitory (GABA)"]
        .groupby(["SZ Factor (%)", "Resilience"], as_index=False)
        .agg({measure_col: sum_unique})
        .rename(columns={measure_col: "inh_val"})
    )

    # --- merge E/I ---
    df_EI = exc_sum.merge(inh_sum,
                          on=["SZ Factor (%)", "Resilience"],
                          how="outer").fillna(0)

    # --- MERGE con TE_mean ---
    df_EI = df_EI.merge(
        df_te2[["SZ Factor (%)", "Resilience", "te_mean"]],
        on=["SZ Factor (%)", "Resilience"],
        how="left"
    )

    # Se ci sono NaN in te_mean li metto a 0 o esclusi
    df_EI["te_mean"] = df_EI["te_mean"].fillna(0)

    # Marker mapping
    marker_map = {
        "With Resilience": "o",
        "Without Resilience": "s"
    }

    # ----------- PLOT -----------
    plt.figure(figsize=(10, 8))

    for res in df_EI["Resilience"].unique():
        sub = df_EI[df_EI["Resilience"] == res]

        plt.scatter(
            sub["exc_val"],
            sub["inh_val"],
            c=sub["te_mean"],
            cmap="coolwarm",
            s=150,
            edgecolor="black",
            linewidth=0.7,
            marker=marker_map.get(res, "o"),
            label=res,
            alpha=0.9
        )

    plt.xlabel(f"Excitatory sum ({measure_col})")
    plt.ylabel(f"Inhibitory sum ({measure_col})")
    plt.title("Exc vs Inh – colored by TE_mean")

    # Colorbar con TE
    cbar = plt.colorbar()
    cbar.set_label("TE_mean (bit)")

    plt.legend(title="Resilience")
    plt.tight_layout()

    outpath = os.path.join(outdir, fname)
    plt.savefig(_prefix_output(outpath), dpi=300)
    plt.show()
    print("Saved:", outpath)

    return df_EI



scatter_exc_inh_colorTE(
    df_all_str,
    df_te,
    measure_col="synapses",
    fname="scatter_TE_synapses.png"
)

scatter_exc_inh_colorTE(
    df_all_str,
    df_te,
    measure_col="effective_synapses",
    fname="scatter_TE_effectiveSyn.png"
)

scatter_exc_inh_colorTE(
    df_all_str,
    df_te,
    measure_col="convergence",
    fname="scatter_TE_convergence.png"
)


In [ ]:
plt.plot(df_all_str[(df_all_str['synType']=='excitatory (glutamate)')&(df_all_str['Resilience']=='2')][['synType', 'weight', 'Resilience', 'SZ Factor (%)']]['weight'])


In [ ]:
plt.plot(df_all_str[(df_all_str['synType']=='excitatory (glutamate)')&(df_all_str['Resilience']=='15')][['synType', 'weight', 'Resilience', 'SZ Factor (%)']]['weight'])

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Se parti da df_all_str:
A = df_all_str[["synType", "weight", "Resilience", "SZ Factor (%)"]].copy()

# Normalizza Resilience (15/2 → stringhe)
def norm_res(val):
    s = str(val).strip()
    if s in ["15", "With Resilience"]:
        return "With Resilience"
    if s in ["2", "Without Resilience"]:
        return "Without Resilience"
    return s

A["Resilience"]    = A["Resilience"].apply(norm_res)
A["SZ Factor (%)"] = A["SZ Factor (%)"].astype(float)

# --- Somme pesi eccitatori / inibitori per SZ × Resilience ---

# eccitatorie
exc_sum = (
    A[A["synType"] == "excitatory (glutamate)"]
    .groupby(["SZ Factor (%)", "Resilience"], as_index=False)
    .agg({"weight": "sum"})
    .rename(columns={"weight": "exc_weight_sum"})
)

# inibitorie
inh_sum = (
    A[A["synType"] == "inhibitory (GABA)"]
    .groupby(["SZ Factor (%)", "Resilience"], as_index=False)
    .agg({"weight": "sum"})
    .rename(columns={"weight": "inh_weight_sum"})
)

# merge
df_w = exc_sum.merge(
    inh_sum,
    on=["SZ Factor (%)", "Resilience"],
    how="outer"
).fillna(0)

print(df_w.head())

# --- Scatter: x = somma pesi eccitatori, y = somma pesi inibitori ---
fig, axs = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)

res_order = ["With Resilience", "Without Resilience"]
titles = {
    "With Resilience": "With Resilience",
    "Without Resilience": "Without Resilience"
}

# per colorbar comune
sz_min = df_w["SZ Factor (%)"].min()
sz_max = df_w["SZ Factor (%)"].max()

for ax, res in zip(axs, res_order):
    sub = df_w[df_w["Resilience"] == res]
    sc = ax.scatter(
        sub["exc_weight_sum"],
        sub["inh_weight_sum"],
        c=sub["SZ Factor (%)"],
        cmap="viridis",
        s=120,
        edgecolor="k",
        linewidth=0.6
    )
    ax.set_title(titles.get(res, res))
    ax.set_xlabel("Sum of excitatory weights")
    if ax is axs[0]:
        ax.set_ylabel("Sum of inhibitory weights")
    else:
        ax.set_ylabel("")

# colorbar condivisa
norm = plt.Normalize(vmin=sz_min, vmax=sz_max)
sm = plt.cm.ScalarMappable(cmap="viridis", norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=axs, shrink=0.8)
cbar.set_label("SZ Factor (%)")

plt.tight_layout()
plt.savefig(_prefix_output("./tempPlots/scatter_sum_exc_inh_weights_two_panels.png"), dpi=300)
plt.show()


In [ ]:
# Connessioni verso Purkinje (senza IO)
pc_conn = [
    "ascending_axon_to_purkinje",
    "basket_to_purkinje",
    "basket_to_basket",
    "parallel_fiber_to_purkinje",
    "stellate_to_purkinje",
    "stellate_to_stellate"
    
]

# Sotto-dataframe solo PC-target + colonne utili
A = df_all_str[
    df_all_str["connection"].isin(pc_conn)
][["effective_convergence", "connection", "SZ Factor (%)", "Resilience", "synType"]]

# Funzione di somma sulle unique (se necessario puoi togliere unique)
def sum_unique(x):
    return np.unique(x).sum()

# Somma ECCITATORIE per SZ × Resilience
exc_sum = (
    A[A["synType"] == "excitatory (glutamate)"]
    .groupby(["SZ Factor (%)", "Resilience"], as_index=False)
    .agg({"effective_convergence": sum_unique})
    .rename(columns={"effective_convergence": "exc_convergence"})
)

# Somma INIBITORIE per SZ × Resilience
inh_sum = (
    A[A["synType"] == "inhibitory (GABA)"]
    .groupby(["SZ Factor (%)", "Resilience"], as_index=False)
    .agg({"effective_convergence": sum_unique})
    .rename(columns={"effective_convergence": "inh_convergence"})
)

# Merge eccitatorie + inibitorie
df_EI = exc_sum.merge(
    inh_sum,
    on=["SZ Factor (%)", "Resilience"],
    how="outer"
).fillna(0)

df_EI

import matplotlib.pyplot as plt
import seaborn as sns

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Assicuriamoci SZ come float
df_EI["SZ Factor (%)"] = df_EI["SZ Factor (%)"].astype(float)

# Calcolo E/I ratio
df_EI["EI_ratio"] = df_EI["exc_convergence"] / df_EI["inh_convergence"].replace(0, np.nan)

# Mask per le due condizioni
df_with    = df_EI[df_EI["Resilience"] == "15"]
df_without = df_EI[df_EI["Resilience"] == "2"]

fig, axs = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# --------- PLOT 1: WITH RESILIENCE ---------
sns.scatterplot(
    data=df_with,
    x="SZ Factor (%)",
    y="EI_ratio",
    hue="exc_convergence",
    palette="Reds",
    s=120,
    edgecolor="black",
    linewidth=0.6,
    ax=axs[0]
)
axs[0].set_title("With Resilience")
axs[0].set_xlabel("SZ Factor (%)")
axs[0].set_ylabel("Effective Exc / Effective Inh (→ PC)")

# --------- PLOT 2: WITHOUT RESILIENCE ---------
sns.scatterplot(
    data=df_without,
    x="SZ Factor (%)",
    y="EI_ratio",
    hue="exc_convergence",
    palette="Blues",
    s=120,
    edgecolor="black",
    linewidth=0.6,
    ax=axs[1]
)
axs[1].set_title("Without Resilience")
axs[1].set_xlabel("SZ Factor (%)")
axs[1].set_ylabel("")

# Colorbar condiviso
sm = plt.cm.ScalarMappable(
    norm=plt.Normalize(df_EI["exc_convergence"].min(), df_EI["exc_convergence"].max()),
    cmap="viridis"
)
sm.set_array([])

cbar = fig.colorbar(sm, ax=axs, shrink=0.8)
cbar.set_label("Excitatory convergence (→ PC)")

plt.tight_layout()
plt.savefig(_prefix_output("./tempPlots/scatter_SZ_vs_EIratio_two_panels.png"), dpi=300)
plt.show()



In [ ]:
# ===============================
# 1. Usa TUTTE LE CONNESSIONI
# ===============================

A = df_all_str[[
    "effective_convergence",
    "connection",
    "SZ Factor (%)",
    "Resilience",
    "synType"
]].copy()

# Funzione di somma con unique (come prima)
def sum_unique(x):
    return np.unique(x).sum()

# ===============================
# 2. Somma ECCITATORIE per SZ × Resilience
# ===============================

exc_sum = (
    A[A["synType"] == "excitatory (glutamate)"]
    .groupby(["SZ Factor (%)", "Resilience"], as_index=False)
    .agg({"effective_convergence": sum_unique})
    .rename(columns={"effective_convergence": "exc_convergence"})
)

# ===============================
# 3. Somma INIBITORIE per SZ × Resilience
# ===============================

inh_sum = (
    A[A["synType"] == "inhibitory (GABA)"]
    .groupby(["SZ Factor (%)", "Resilience"], as_index=False)
    .agg({"effective_convergence": sum_unique})
    .rename(columns={"effective_convergence": "inh_convergence"})
)

# ===============================
# 4. Merge totale
# ===============================

df_EI = exc_sum.merge(
    inh_sum,
    on=["SZ Factor (%)", "Resilience"],
    how="outer"
).fillna(0)

df_EI["SZ Factor (%)"] = df_EI["SZ Factor (%)"].astype(float)

# rapporto E/I
df_EI["EI_ratio"] = df_EI["exc_convergence"] / df_EI["inh_convergence"].replace(0, np.nan)

print(df_EI.head())

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Split
df_with    = df_EI[df_EI["Resilience"] == "15"]
df_without = df_EI[df_EI["Resilience"] == "2"]

fig, axs = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# ---------------- WITH RESILIENCE -----------------
sns.scatterplot(
    data=df_with,
    x="SZ Factor (%)",
    y="EI_ratio",
    hue="exc_convergence",
    palette="Reds",
    s=140,
    edgecolor="black",
    linewidth=0.6,
    ax=axs[0]
)
axs[0].set_title("With Resilience")
axs[0].set_xlabel("SZ Factor (%)")
axs[0].set_ylabel("Exc / Inh convergence (E/I ratio)")

# ---------------- WITHOUT RESILIENCE -----------------
sns.scatterplot(
    data=df_without,
    x="SZ Factor (%)",
    y="EI_ratio",
    hue="exc_convergence",
    palette="Blues",
    s=140,
    edgecolor="black",
    linewidth=0.6,
    ax=axs[1]
)
axs[1].set_title("Without Resilience")
axs[1].set_xlabel("SZ Factor (%)")
axs[1].set_ylabel("")

# Colorbar condiviso
norm = plt.Normalize(df_EI["exc_convergence"].min(), df_EI["exc_convergence"].max())
sm = plt.cm.ScalarMappable(cmap="viridis", norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=axs, shrink=0.8)
cbar.set_label("Excitatory convergence (sum)")

plt.tight_layout()
plt.savefig(_prefix_output("./tempPlots/scatter_SZ_vs_EIratio_ALLconnections_two_panels.png"), dpi=300)
plt.show()


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def norm_res(val):
    s = str(val).strip()
    if s in ["15", "With Resilience"]:
        return "With Resilience"
    if s in ["2", "Without Resilience"]:
        return "Without Resilience"
    return s

def plot_SZ_vs_EIratio_allconns(
        df_all_str,
        measure_col="effective_convergence",
        outdir="./tempPlots",
        fname_prefix="SZ_vs_EIratio_allconns"
    ):
    """
    Usa TUTTE le connessioni di df_all_str (nessuna maschera su 'connection').

    Per ogni combinazione (SZ Factor (%), Resilience):
      - somma il valore eccitatorio di `measure_col`
      - somma il valore inibitorio di `measure_col`
      - calcola E/I ratio = exc_sum / inh_sum

    Poi fa un plot 1x2:
      - left: With Resilience
      - right: Without Resilience
    con SZ Factor (%) sull'asse X e E/I ratio sull'asse Y.
    Il colore è la somma eccitatoria (exc_sum) per ciascun punto.
    """

    os.makedirs(outdir, exist_ok=True)

    # copia e normalizza
    df = df_all_str.copy()
    df["Resilience"]    = df["Resilience"].apply(norm_res)
    df["SZ Factor (%)"] = df["SZ Factor (%)"].astype(float)

    # controlli minimi
    needed = [measure_col, "SZ Factor (%)", "Resilience", "synType"]
    for c in needed:
        if c not in df.columns:
            raise ValueError(f"Column '{c}' not found in df_all_str")

    # funzione di somma sulle unique (come già usavi)
    def sum_unique(x):
        return np.unique(x).sum()

    # --- somme eccitatorie per SZ×Resilience ---
    exc_sum = (
        df[df["synType"] == "excitatory (glutamate)"]
        .groupby(["SZ Factor (%)", "Resilience"], as_index=False)
        .agg({measure_col: sum_unique})
        .rename(columns={measure_col: "exc_val"})
    )

    # --- somme inibitorie per SZ×Resilience ---
    inh_sum = (
        df[df["synType"] == "inhibitory (GABA)"]
        .groupby(["SZ Factor (%)", "Resilience"], as_index=False)
        .agg({measure_col: sum_unique})
        .rename(columns={measure_col: "inh_val"})
    )

    # --- merge E + I ---
    df_EI = exc_sum.merge(
        inh_sum,
        on=["SZ Factor (%)", "Resilience"],
        how="outer"
    ).fillna(0)

    # --- E/I ratio ---
    df_EI["EI_ratio"] = df_EI["exc_val"] / df_EI["inh_val"].replace(0, np.nan)

    # split per Resilience
    df_with    = df_EI[df_EI["Resilience"] == "With Resilience"]
    df_without = df_EI[df_EI["Resilience"] == "Without Resilience"]

    # ===== PLOT 1x2: SZ vs E/I =====
    fig, axs = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

    # pannello With Resilience
    sns.scatterplot(
        data=df_with,
        x="SZ Factor (%)",
        y="EI_ratio",
        hue="exc_val",
        palette="Reds",
        s=120,
        edgecolor="black",
        linewidth=0.6,
        ax=axs[0]
    )
    axs[0].set_title("With Resilience")
    axs[0].set_xlabel("SZ Factor (%)")
    axs[0].set_ylabel(f"Exc/Inh ratio ({measure_col})")
    axs[0].axhline(1, color="gray", linestyle="--", alpha=0.5)

    # pannello Without Resilience
    sns.scatterplot(
        data=df_without,
        x="SZ Factor (%)",
        y="EI_ratio",
        hue="exc_val",
        palette="Blues",
        s=120,
        edgecolor="black",
        linewidth=0.6,
        ax=axs[1]
    )
    axs[1].set_title("Without Resilience")
    axs[1].set_xlabel("SZ Factor (%)")
    axs[1].set_ylabel("")
    axs[1].axhline(1, color="gray", linestyle="--", alpha=0.5)

    # Colorbar condivisa sulla somma eccitatoria
    vmin = df_EI["exc_val"].min()
    vmax = df_EI["exc_val"].max()
    norm = plt.Normalize(vmin, vmax)
    sm   = plt.cm.ScalarMappable(norm=norm, cmap="viridis")
    sm.set_array([])

    cbar = fig.colorbar(sm, ax=axs, shrink=0.8)
    cbar.set_label(f"Excitatory sum ({measure_col})")

    plt.tight_layout()
    fname   = f"{fname_prefix}_{measure_col}.png"
    outpath = os.path.join(outdir, fname)
    plt.savefig(_prefix_output(outpath), dpi=300)
    print("Saved:", outpath)
    plt.show()

    return df_EI


In [ ]:
df_EI_conv = plot_SZ_vs_EIratio_allconns(
    df_all_str,
    measure_col="effective_convergence",
    outdir="./tempPlots",
    fname_prefix="SZ_vs_EIratio_allconns"
)

df_EI_conv = plot_SZ_vs_EIratio_allconns(
    df_all_str,
    measure_col="effective_synapses",
    outdir="./tempPlots",
    fname_prefix="SZ_vs_EIratio_allconns"
)

df_EI_conv = plot_SZ_vs_EIratio_allconns(
    df_all_str,
    measure_col="effective_divergence",
    outdir="./tempPlots",
    fname_prefix="SZ_vs_EIratio_allconns"
)

df_EI_conv = plot_SZ_vs_EIratio_allconns(
    df_all_str,
    measure_col="synapses",
    outdir="./tempPlots",
    fname_prefix="SZ_vs_EIratio_allconns"
)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colormaps
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
from matplotlib.cm import ScalarMappable
from pathlib import Path

def norm_res(val):
    s = str(val).strip()
    if s in ["15", "With Resilience"]:
        return "With Resilience"
    if s in ["2", "Without Resilience"]:
        return "Without Resilience"
    return s

df_str = df_all_str.copy()
df_str["Resilience"] = df_str["Resilience"].apply(norm_res)
df_str["SZ Factor (%)"] = pd.to_numeric(df_str["SZ Factor (%)"]).round(2)

df_te_norm = df_te.copy()
df_te_norm["Resilience"] = df_te_norm["Resilience"].apply(norm_res)
df_te_norm["SZ Factor (%)"] = pd.to_numeric(df_te_norm["SZ Factor (%)"]).round(2)

df_mean_struct = (
    df_str.groupby(["SZ Factor (%)", "Resilience"], as_index=False)
    .agg(total_cells=("total_cells", "mean"), synapses=("synapses", "mean"))
)

df_mean_te = (
    df_te_norm.groupby(["SZ Factor (%)", "Resilience"], as_index=False)
    .agg(te_mean=("te_mean", "mean"), te_std=("te_std", "mean"))
)

df_plot = df_mean_struct.merge(
    df_mean_te,
    on=["SZ Factor (%)", "Resilience"],
    how="inner",
    validate="one_to_one"
)

print("df_plot shape:", df_plot.shape)
print(df_plot.sort_values(["Resilience", "SZ Factor (%)"]).to_string(index=False))

cmaps = {
    "With Resilience": colormaps["Reds"],
    "Without Resilience": colormaps["Blues"]
}

markers = {
    "With Resilience": "o",
    "Without Resilience": "s"
}

te_min = df_plot["te_mean"].min()
te_max = df_plot["te_mean"].max()

if te_min == te_max:
    te_min -= 0.5
    te_max += 0.5

norm = Normalize(vmin=te_min, vmax=te_max)

std_min = df_plot["te_std"].min()
std_max = df_plot["te_std"].max()

if std_max > std_min:
    df_plot["marker_size"] = 60 + 180 * (df_plot["te_std"] - std_min) / (std_max - std_min)
else:
    df_plot["marker_size"] = 120

fig, ax = plt.subplots(figsize=(8, 6))

for res in ["With Resilience", "Without Resilience"]:
    sub = df_plot[df_plot["Resilience"] == res]

    if sub.empty:
        continue

    colors = cmaps[res](0.2 + 0.8 * norm(sub["te_mean"].to_numpy()))

    ax.scatter(
        sub["total_cells"],
        sub["synapses"],
        c=colors,
        s=sub["marker_size"],
        marker=markers[res],
        edgecolor="black",
        linewidth=0.6,
        alpha=0.9,
        zorder=2
    )

    idx_max = sub["te_mean"].idxmax()
    row = df_plot.loc[idx_max]

    ax.scatter(
        row["total_cells"],
        row["synapses"],
        s=row["marker_size"] + 120,
        marker=markers[res],
        facecolors="none",
        edgecolors="black",
        linewidth=2,
        zorder=3
    )

    print(
        f"{res}: max TE = {row['te_mean']:.4f} bit, "
        f"SZ = {row['SZ Factor (%)']:.2f}%, "
        f"cells = {row['total_cells']:.0f}, "
        f"synapses = {row['synapses']:.0f}"
    )

ax.set_xlabel("Total cells (mean per SZ)")
ax.set_ylabel("Synapses (mean per SZ)")
ax.set_title("Cells vs Synapses | Color: TE mean | Size: TE std")

legend_handles = [
    Line2D(
        [0], [0],
        marker="o",
        linestyle="",
        markerfacecolor="firebrick",
        markeredgecolor="black",
        markersize=9,
        label="With Resilience"
    ),
    Line2D(
        [0], [0],
        marker="s",
        linestyle="",
        markerfacecolor="royalblue",
        markeredgecolor="black",
        markersize=9,
        label="Without Resilience"
    ),
    Line2D(
        [0], [0],
        marker="o",
        linestyle="",
        markerfacecolor="none",
        markeredgecolor="black",
        markeredgewidth=2,
        markersize=12,
        label="Maximum TE"
    )
]

ax.legend(handles=legend_handles, loc="best", title="Resilience")

fig.subplots_adjust(right=0.77)

cax_red = fig.add_axes([0.80, 0.55, 0.025, 0.30])
cax_blue = fig.add_axes([0.80, 0.14, 0.025, 0.30])

sm_red = ScalarMappable(
    norm=Normalize(vmin=te_min, vmax=te_max),
    cmap=cmaps["With Resilience"]
)

sm_blue = ScalarMappable(
    norm=Normalize(vmin=te_min, vmax=te_max),
    cmap=cmaps["Without Resilience"]
)

cbar_red = fig.colorbar(sm_red, cax=cax_red)
cbar_blue = fig.colorbar(sm_blue, cax=cax_blue)

cbar_red.set_label("TE mean (bit) - With")
cbar_blue.set_label("TE mean (bit) - Without")

Path("./tempPlots").mkdir(parents=True, exist_ok=True)

plt.savefig(
    _prefix_output("./tempPlots/cells_vs_synapses_colored_by_TEmean.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colormaps
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.cm import ScalarMappable
from pathlib import Path

def norm_res(val):
    s = str(val).strip()
    if s in ["15", "15.0", "With Resilience"]:
        return "With Resilience"
    if s in ["2", "2.0", "Without Resilience"]:
        return "Without Resilience"
    return s

def sum_unique_synapses(x):
    return np.unique(x.dropna()).sum()

keys = ["SZ Factor (%)", "Resilience"]
conditions = ["With Resilience", "Without Resilience"]

df_str = df_all_str.copy()
df_str["Resilience"] = df_str["Resilience"].apply(norm_res)
df_str["SZ Factor (%)"] = pd.to_numeric(df_str["SZ Factor (%)"]).round(2)
df_str["total_cells"] = pd.to_numeric(df_str["total_cells"], errors="coerce")
df_str["synapses"] = pd.to_numeric(df_str["synapses"], errors="coerce")

df_te_norm = df_te.copy()
df_te_norm["Resilience"] = df_te_norm["Resilience"].apply(norm_res)
df_te_norm["SZ Factor (%)"] = pd.to_numeric(df_te_norm["SZ Factor (%)"]).round(2)
df_te_norm["te_mean"] = pd.to_numeric(df_te_norm["te_mean"], errors="coerce")
df_te_norm["te_std"] = pd.to_numeric(df_te_norm["te_std"], errors="coerce")

df_struct = (
    df_str.groupby(keys, as_index=False)
    .agg(
        total_cells=("total_cells", "first"),
        synapses=("synapses", sum_unique_synapses)
    )
)

df_mean_te = (
    df_te_norm.groupby(keys, as_index=False)
    .agg(
        te_mean=("te_mean", "mean"),
        te_std=("te_std", "mean")
    )
)

df_plot = df_struct.merge(
    df_mean_te,
    on=keys,
    how="inner",
    validate="one_to_one"
)

df_plot = df_plot.dropna(
    subset=["total_cells", "synapses", "te_mean"]
).copy()

if df_plot.empty:
    raise ValueError("Nessuna combinazione SZ × Resilience disponibile.")

print("\n=== CONTROL SZ=0, WITH RESILIENCE ===")
print(df_struct.loc[
    (df_struct["SZ Factor (%)"] == 0) &
    (df_struct["Resilience"] == "With Resilience")
])

print("\n=== DATAFRAME COMPLETO ===")
print(df_plot.sort_values(keys).to_string(index=False))
print("\ndf_plot shape:", df_plot.shape)

cmaps = {
    "With Resilience": LinearSegmentedColormap.from_list(
        "Reds_trimmed",
        colormaps["Reds"](np.linspace(0.2, 1, 256))
    ),
    "Without Resilience": LinearSegmentedColormap.from_list(
        "Blues_trimmed",
        colormaps["Blues"](np.linspace(0.2, 1, 256))
    )
}

markers = {
    "With Resilience": "o",
    "Without Resilience": "s"
}

te_min = df_plot["te_mean"].min()
te_max = df_plot["te_mean"].max()

if te_min == te_max:
    te_min -= 0.5
    te_max += 0.5

norm = Normalize(vmin=te_min, vmax=te_max)

std_min = df_plot["te_std"].min()
std_max = df_plot["te_std"].max()

if pd.notna(std_min) and pd.notna(std_max) and std_max > std_min:
    df_plot["marker_size"] = 80 + 160 * (
        (df_plot["te_std"] - std_min) / (std_max - std_min)
    )
    df_plot["marker_size"] = df_plot["marker_size"].fillna(120)
else:
    df_plot["marker_size"] = 120

fig, ax = plt.subplots(figsize=(8, 6))

for res in conditions:
    sub = df_plot[df_plot["Resilience"] == res]

    if sub.empty:
        continue

    ax.scatter(
        sub["total_cells"],
        sub["synapses"],
        c=sub["te_mean"],
        cmap=cmaps[res],
        norm=norm,
        s=sub["marker_size"],
        marker=markers[res],
        edgecolor="black",
        linewidth=0.6,
        alpha=0.9,
        zorder=2
    )

    idx_max = sub["te_mean"].idxmax()
    row = df_plot.loc[idx_max]

    ax.scatter(
        row["total_cells"],
        row["synapses"],
        s=row["marker_size"] + 120,
        marker=markers[res],
        facecolors="none",
        edgecolors="black",
        linewidth=2.5,
        zorder=5
    )

    print(
        f"{res}: max TE={row['te_mean']:.4f} bit, "
        f"SZ={row['SZ Factor (%)']:.2f}%, "
        f"cells={row['total_cells']:.0f}, "
        f"synapses={row['synapses']:.0f}"
    )

ax.set_xlabel("Total cells")
ax.set_ylabel("Total synapses")
ax.set_title("Cells vs Total Synapses")
ax.grid(alpha=0.15)

legend_handles = [
    Line2D(
        [0], [0],
        marker="o",
        linestyle="",
        markerfacecolor="firebrick",
        markeredgecolor="black",
        markersize=9,
        label="With Resilience"
    ),
    Line2D(
        [0], [0],
        marker="s",
        linestyle="",
        markerfacecolor="royalblue",
        markeredgecolor="black",
        markersize=9,
        label="Without Resilience"
    ),
    Line2D(
        [0], [0],
        marker="o",
        linestyle="",
        markerfacecolor="none",
        markeredgecolor="black",
        markeredgewidth=2.5,
        markersize=12,
        label="Maximum TE"
    )
]

ax.legend(handles=legend_handles, title="Resilience", loc="best")

fig.subplots_adjust(
    left=0.14,
    right=0.77,
    top=0.91,
    bottom=0.13
)

cax_red = fig.add_axes([0.80, 0.55, 0.025, 0.30])
cax_blue = fig.add_axes([0.80, 0.14, 0.025, 0.30])

sm_red = ScalarMappable(
    norm=norm,
    cmap=cmaps["With Resilience"]
)

sm_blue = ScalarMappable(
    norm=norm,
    cmap=cmaps["Without Resilience"]
)

fig.colorbar(
    sm_red,
    cax=cax_red
).set_label("TE mean (bit) - With")

fig.colorbar(
    sm_blue,
    cax=cax_blue
).set_label("TE mean (bit) - Without")

idx_max_syn = df_plot["synapses"].idxmax()
idx_max_cells = df_plot["total_cells"].idxmax()

print("\n=== MAX TOTAL SYNAPSES ===")
print(df_plot.loc[idx_max_syn])

print("\n=== MAX TOTAL CELLS ===")
print(df_plot.loc[idx_max_cells])

Path("./tempPlots").mkdir(parents=True, exist_ok=True)

output = "./tempPlots/cells_vs_synapses_colored_by_TEmean.png"

fig.savefig(
    _prefix_output(output),
    dpi=300,
    bbox_inches="tight"
)

print(f"\nSaved: {output}")

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colormaps
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.cm import ScalarMappable
from pathlib import Path

def norm_res(val):
    s = str(val).strip()
    if s in ["15", "With Resilience"]:
        return "With Resilience"
    if s in ["2", "Without Resilience"]:
        return "Without Resilience"
    return s

def sum_unique_synapses(x):
    return np.unique(x.dropna()).sum()

df_str = df_all_str.copy()
df_str["Resilience"] = df_str["Resilience"].apply(norm_res)
df_str["SZ Factor (%)"] = pd.to_numeric(df_str["SZ Factor (%)"]).round(2)

df_te_norm = df_te.copy()
df_te_norm["Resilience"] = df_te_norm["Resilience"].apply(norm_res)
df_te_norm["SZ Factor (%)"] = pd.to_numeric(df_te_norm["SZ Factor (%)"]).round(2)

df_struct = (
    df_str.groupby(["SZ Factor (%)", "Resilience"], as_index=False)
    .agg(
        total_cells=("total_cells", "first"),
        synapses=("synapses", sum_unique_synapses)
    )
)

df_mean_te = (
    df_te_norm.groupby(["SZ Factor (%)", "Resilience"], as_index=False)
    .agg(
        te_mean=("te_mean", "mean"),
        te_std=("te_std", "mean")
    )
)

df_plot = df_struct.merge(
    df_mean_te,
    on=["SZ Factor (%)", "Resilience"],
    how="inner",
    validate="one_to_one"
)

baseline = (
    df_struct[df_struct["SZ Factor (%)"] == 0]
    .set_index("Resilience")[["total_cells", "synapses"]]
    .rename(columns={
        "total_cells": "baseline_cells",
        "synapses": "baseline_synapses"
    })
)

df_plot = df_plot.join(baseline, on="Resilience", validate="many_to_one")

if df_plot[["baseline_cells", "baseline_synapses"]].isna().any().any():
    raise ValueError("Baseline SZ=0 mancante per almeno una condizione.")

if (df_plot[["baseline_cells", "baseline_synapses"]] == 0).any().any():
    raise ValueError("Baseline uguale a zero: impossibile calcolare Δ%.")

df_plot["cells_pct"] = 100 * (
    df_plot["total_cells"] / df_plot["baseline_cells"] - 1
)

df_plot["syn_pct"] = 100 * (
    df_plot["synapses"] / df_plot["baseline_synapses"] - 1
)

print("df_plot shape:", df_plot.shape)
print(df_plot.sort_values(["Resilience", "SZ Factor (%)"]).to_string(index=False))

print("\n=== BASELINE CHECK ===")
print(df_plot.loc[
    df_plot["SZ Factor (%)"] == 0,
    ["Resilience", "cells_pct", "syn_pct"]
])

cmaps = {
    "With Resilience": LinearSegmentedColormap.from_list(
        "Reds_trimmed", colormaps["Reds"](np.linspace(0.2, 1, 256))
    ),
    "Without Resilience": LinearSegmentedColormap.from_list(
        "Blues_trimmed", colormaps["Blues"](np.linspace(0.2, 1, 256))
    )
}

markers = {
    "With Resilience": "o",
    "Without Resilience": "s"
}

te_min = df_plot["te_mean"].min()
te_max = df_plot["te_mean"].max()

if te_min == te_max:
    te_min -= 0.5
    te_max += 0.5

norm = Normalize(vmin=te_min, vmax=te_max)

std_min = df_plot["te_std"].min()
std_max = df_plot["te_std"].max()

if std_max > std_min:
    df_plot["marker_size"] = 60 + 180 * (
        (df_plot["te_std"] - std_min) / (std_max - std_min)
    )
else:
    df_plot["marker_size"] = 120

fig, ax = plt.subplots(figsize=(8, 6))

for res in ["With Resilience", "Without Resilience"]:
    sub = df_plot[df_plot["Resilience"] == res]

    if sub.empty:
        continue

    ax.scatter(
        sub["cells_pct"],
        sub["syn_pct"],
        c=sub["te_mean"],
        cmap=cmaps[res],
        norm=norm,
        s=sub["marker_size"],
        marker=markers[res],
        edgecolor="black",
        linewidth=0.6,
        alpha=0.9,
        zorder=2
    )

    idx_max = sub["te_mean"].idxmax()
    row = df_plot.loc[idx_max]

    ax.scatter(
        row["cells_pct"],
        row["syn_pct"],
        s=row["marker_size"] + 120,
        marker=markers[res],
        facecolors="none",
        edgecolors="black",
        linewidth=2,
        zorder=3
    )

    print(
        f"{res}: max TE={row['te_mean']:.4f} bit, "
        f"SZ={row['SZ Factor (%)']:.2f}%, "
        f"ΔCells={row['cells_pct']:.2f}%, "
        f"ΔSynapses={row['syn_pct']:.2f}%"
    )

ax.axhline(0, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
ax.axvline(0, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)

ax.set_xlabel("Δ Total cells (%) vs SZ=0")
ax.set_ylabel("Δ Total synapses (%) vs SZ=0")
ax.set_title("Structural changes and transfer entropy")

legend_handles = [
    Line2D([0], [0], marker="o", linestyle="",
           markerfacecolor="firebrick", markeredgecolor="black",
           markersize=9, label="With Resilience"),
    Line2D([0], [0], marker="s", linestyle="",
           markerfacecolor="royalblue", markeredgecolor="black",
           markersize=9, label="Without Resilience"),
    Line2D([0], [0], marker="o", linestyle="",
           markerfacecolor="none", markeredgecolor="black",
           markeredgewidth=2, markersize=12, label="Maximum TE")
]

ax.legend(handles=legend_handles, loc="best", title="Resilience")

fig.subplots_adjust(right=0.77)

cax_red = fig.add_axes([0.80, 0.55, 0.025, 0.30])
cax_blue = fig.add_axes([0.80, 0.14, 0.025, 0.30])

sm_red = ScalarMappable(norm=norm, cmap=cmaps["With Resilience"])
sm_blue = ScalarMappable(norm=norm, cmap=cmaps["Without Resilience"])

fig.colorbar(sm_red, cax=cax_red).set_label("TE mean (bit) - With")
fig.colorbar(sm_blue, cax=cax_blue).set_label("TE mean (bit) - Without")

Path("./tempPlots").mkdir(parents=True, exist_ok=True)

plt.savefig(
    _prefix_output("./tempPlots/delta_percent_cells_vs_synapses.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colormaps
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.cm import ScalarMappable
from pathlib import Path

def norm_res(val):
    s = str(val).strip()
    if s in ["15", "With Resilience"]:
        return "With Resilience"
    if s in ["2", "Without Resilience"]:
        return "Without Resilience"
    return s

def sum_unique_synapses(x):
    return np.unique(x.dropna()).sum()

df_str = df_all_str.copy()
df_str["Resilience"] = df_str["Resilience"].apply(norm_res)
df_str["SZ Factor (%)"] = pd.to_numeric(df_str["SZ Factor (%)"]).round(2)

df_te_norm = df_te.copy()
df_te_norm["Resilience"] = df_te_norm["Resilience"].apply(norm_res)
df_te_norm["SZ Factor (%)"] = pd.to_numeric(df_te_norm["SZ Factor (%)"]).round(2)

df_struct = (
    df_str.groupby(["SZ Factor (%)", "Resilience"], as_index=False)
    .agg(
        total_cells=("total_cells", "first"),
        synapses=("synapses", sum_unique_synapses)
    )
)

df_mean_te = (
    df_te_norm.groupby(["SZ Factor (%)", "Resilience"], as_index=False)
    .agg(
        te_mean=("te_mean", "mean"),
        te_std=("te_std", "mean")
    )
)

df_plot = df_struct.merge(
    df_mean_te,
    on=["SZ Factor (%)", "Resilience"],
    how="inner",
    validate="one_to_one"
)

baseline = (
    df_plot[df_plot["SZ Factor (%)"] == 0]
    .set_index("Resilience")[["total_cells", "synapses", "te_mean"]]
    .rename(columns={
        "total_cells": "baseline_cells",
        "synapses": "baseline_synapses",
        "te_mean": "baseline_te"
    })
)

df_plot = df_plot.join(baseline, on="Resilience", validate="many_to_one")

baseline_cols = ["baseline_cells", "baseline_synapses", "baseline_te"]

if df_plot[baseline_cols].isna().any().any():
    raise ValueError("Baseline SZ=0 mancante per almeno una condizione.")

if (df_plot[baseline_cols] == 0).any().any():
    raise ValueError("Baseline uguale a zero: impossibile calcolare Δ%.")

df_plot["cells_pct"] = 100 * (
    df_plot["total_cells"] / df_plot["baseline_cells"] - 1
)

df_plot["syn_pct"] = 100 * (
    df_plot["synapses"] / df_plot["baseline_synapses"] - 1
)

df_plot["te_pct"] = 100 * (
    df_plot["te_mean"] / df_plot["baseline_te"] - 1
)

print("\n=== BASELINE CHECK ===")
print(df_plot.loc[
    df_plot["SZ Factor (%)"] == 0,
    ["Resilience", "cells_pct", "syn_pct", "te_pct"]
])

print("\n=== DATA ===")
print(df_plot.sort_values(["Resilience", "SZ Factor (%)"]).to_string(index=False))

cmaps = {
    "With Resilience": LinearSegmentedColormap.from_list(
        "Reds_trimmed", colormaps["Reds"](np.linspace(0.2, 1, 256))
    ),
    "Without Resilience": LinearSegmentedColormap.from_list(
        "Blues_trimmed", colormaps["Blues"](np.linspace(0.2, 1, 256))
    )
}

markers = {
    "With Resilience": "o",
    "Without Resilience": "s"
}

te_min = df_plot["te_pct"].min()
te_max = df_plot["te_pct"].max()

if te_min == te_max:
    te_min -= 0.5
    te_max += 0.5

norm = Normalize(vmin=te_min, vmax=te_max)

std_min = df_plot["te_std"].min()
std_max = df_plot["te_std"].max()

if std_max > std_min:
    df_plot["marker_size"] = 60 + 180 * (
        (df_plot["te_std"] - std_min) / (std_max - std_min)
    )
else:
    df_plot["marker_size"] = 120

fig, ax = plt.subplots(figsize=(8, 6))

for res in ["With Resilience", "Without Resilience"]:
    sub = df_plot[df_plot["Resilience"] == res]

    if sub.empty:
        continue

    ax.scatter(
        sub["cells_pct"],
        sub["syn_pct"],
        c=sub["te_pct"],
        cmap=cmaps[res],
        norm=norm,
        s=sub["marker_size"],
        marker=markers[res],
        edgecolor="black",
        linewidth=0.6,
        alpha=0.9,
        zorder=2
    )

    idx_max = sub["te_mean"].idxmax()
    row = df_plot.loc[idx_max]

    ax.scatter(
        row["cells_pct"],
        row["syn_pct"],
        s=row["marker_size"] + 120,
        marker=markers[res],
        facecolors="none",
        edgecolors="black",
        linewidth=2,
        zorder=3
    )

    print(
        f"{res}: max TE={row['te_mean']:.4f} bit, "
        f"SZ={row['SZ Factor (%)']:.2f}%, "
        f"ΔCells={row['cells_pct']:.2f}%, "
        f"ΔSynapses={row['syn_pct']:.2f}%, "
        f"ΔTE={row['te_pct']:.2f}%"
    )

ax.axhline(0, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
ax.axvline(0, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)

ax.set_xlabel("Δ Total cells (%) vs SZ=0")
ax.set_ylabel("Δ Total synapses (%) vs SZ=0")
ax.set_title("Structural changes and relative transfer entropy")

legend_handles = [
    Line2D([0], [0], marker="o", linestyle="",
           markerfacecolor="firebrick", markeredgecolor="black",
           markersize=9, label="With Resilience"),
    Line2D([0], [0], marker="s", linestyle="",
           markerfacecolor="royalblue", markeredgecolor="black",
           markersize=9, label="Without Resilience"),
    Line2D([0], [0], marker="o", linestyle="",
           markerfacecolor="none", markeredgecolor="black",
           markeredgewidth=2, markersize=12, label="Maximum TE")
]

ax.legend(handles=legend_handles, loc="best", title="Resilience")

fig.subplots_adjust(right=0.77)

cax_red = fig.add_axes([0.80, 0.55, 0.025, 0.30])
cax_blue = fig.add_axes([0.80, 0.14, 0.025, 0.30])

sm_red = ScalarMappable(norm=norm, cmap=cmaps["With Resilience"])
sm_blue = ScalarMappable(norm=norm, cmap=cmaps["Without Resilience"])

fig.colorbar(sm_red, cax=cax_red).set_label("Δ TE (%) - With")
fig.colorbar(sm_blue, cax=cax_blue).set_label("Δ TE (%) - Without")

Path("./tempPlots").mkdir(parents=True, exist_ok=True)

plt.savefig(
    _prefix_output("./tempPlots/delta_percent_cells_vs_synapses_TE_norm.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colormaps
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.cm import ScalarMappable
from pathlib import Path

def norm_res(val):
    s = str(val).strip()
    if s in ["15", "With Resilience"]:
        return "With Resilience"
    if s in ["2", "Without Resilience"]:
        return "Without Resilience"
    return s

def sum_unique_synapses(x):
    return np.unique(x.dropna()).sum()

keys = ["SZ Factor (%)", "Resilience"]

df_str = df_all_str.copy()
df_str["Resilience"] = df_str["Resilience"].apply(norm_res)
df_str["SZ Factor (%)"] = pd.to_numeric(df_str["SZ Factor (%)"]).round(2)
df_str["synapses"] = pd.to_numeric(df_str["synapses"], errors="coerce")

df_te_norm = df_te.copy()
df_te_norm["Resilience"] = df_te_norm["Resilience"].apply(norm_res)
df_te_norm["SZ Factor (%)"] = pd.to_numeric(df_te_norm["SZ Factor (%)"]).round(2)

df_exc = df_str[df_str["synType"] == "excitatory (glutamate)"]
df_inh = df_str[df_str["synType"] == "inhibitory (GABA)"]

df_exc_struct = (
    df_exc.groupby(keys, as_index=False)
    .agg(syn_exc=("synapses", sum_unique_synapses))
)

df_inh_struct = (
    df_inh.groupby(keys, as_index=False)
    .agg(syn_inh=("synapses", sum_unique_synapses))
)

df_struct = df_exc_struct.merge(
    df_inh_struct,
    on=keys,
    how="inner",
    validate="one_to_one"
)

df_mean_te = (
    df_te_norm.groupby(keys, as_index=False)
    .agg(
        te_mean=("te_mean", "mean"),
        te_std=("te_std", "mean")
    )
)

df_plot = df_struct.merge(
    df_mean_te,
    on=keys,
    how="inner",
    validate="one_to_one"
)

baseline = (
    df_plot[df_plot["SZ Factor (%)"] == 0]
    .set_index("Resilience")[["syn_exc", "syn_inh", "te_mean"]]
    .rename(columns={
        "syn_exc": "syn_exc_0",
        "syn_inh": "syn_inh_0",
        "te_mean": "te_mean_0"
    })
)

df_plot = df_plot.join(
    baseline,
    on="Resilience",
    validate="many_to_one"
)

baseline_cols = ["syn_exc_0", "syn_inh_0", "te_mean_0"]

if df_plot[baseline_cols].isna().any().any():
    raise ValueError("Baseline SZ=0 mancante per almeno una condizione.")

if (df_plot[baseline_cols] == 0).any().any():
    raise ValueError("Baseline uguale a zero: Δ% non calcolabile.")

df_plot["exc_pct"] = 100 * (
    df_plot["syn_exc"] / df_plot["syn_exc_0"] - 1
)

df_plot["inh_pct"] = 100 * (
    df_plot["syn_inh"] / df_plot["syn_inh_0"] - 1
)

df_plot["te_pct"] = 100 * (
    df_plot["te_mean"] / df_plot["te_mean_0"] - 1
)

print("\n=== BASELINE SZ=0 ===")
print(df_plot.loc[
    df_plot["SZ Factor (%)"] == 0,
    ["Resilience", "exc_pct", "inh_pct", "te_pct"]
])

print("\n=== DATAFRAME COMPLETO ===")
print(df_plot.sort_values(keys).to_string(index=False))

cmaps = {
    "With Resilience": LinearSegmentedColormap.from_list(
        "Reds_trimmed",
        colormaps["Reds"](np.linspace(0.2, 1, 256))
    ),
    "Without Resilience": LinearSegmentedColormap.from_list(
        "Blues_trimmed",
        colormaps["Blues"](np.linspace(0.2, 1, 256))
    )
}

markers = {
    "With Resilience": "o",
    "Without Resilience": "s"
}

te_min = df_plot["te_pct"].min()
te_max = df_plot["te_pct"].max()

if te_min == te_max:
    te_min -= 0.5
    te_max += 0.5

norm = Normalize(vmin=te_min, vmax=te_max)

std_min = df_plot["te_std"].min()
std_max = df_plot["te_std"].max()

if std_max > std_min:
    df_plot["marker_size"] = 60 + 180 * (
        (df_plot["te_std"] - std_min) / (std_max - std_min)
    )
else:
    df_plot["marker_size"] = 120

fig, ax = plt.subplots(figsize=(8, 6))

for res in ["With Resilience", "Without Resilience"]:
    sub = df_plot[df_plot["Resilience"] == res]

    if sub.empty:
        continue

    ax.scatter(
        sub["exc_pct"],
        sub["inh_pct"],
        c=sub["te_pct"],
        cmap=cmaps[res],
        norm=norm,
        s=sub["marker_size"],
        marker=markers[res],
        edgecolor="black",
        linewidth=0.6,
        alpha=0.9,
        zorder=2
    )

    idx_max = sub["te_mean"].idxmax()
    row = df_plot.loc[idx_max]

    ax.scatter(
        row["exc_pct"],
        row["inh_pct"],
        s=row["marker_size"] + 120,
        marker=markers[res],
        facecolors="none",
        edgecolors="black",
        linewidth=2,
        zorder=3
    )

    print(
        f"{res}: max TE={row['te_mean']:.4f} bit, "
        f"SZ={row['SZ Factor (%)']:.2f}%, "
        f"ΔExc={row['exc_pct']:.2f}%, "
        f"ΔInh={row['inh_pct']:.2f}%, "
        f"ΔTE={row['te_pct']:.2f}%"
    )

ax.axhline(0, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
ax.axvline(0, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)

ax.set_xlabel("Δ Excitatory synapses (%) vs SZ=0")
ax.set_ylabel("Δ Inhibitory synapses (%) vs SZ=0")
ax.set_title("Excitatory/Inhibitory structural changes and transfer entropy")

legend_handles = [
    Line2D(
        [0], [0], marker="o", linestyle="",
        markerfacecolor="firebrick", markeredgecolor="black",
        markersize=9, label="With Resilience"
    ),
    Line2D(
        [0], [0], marker="s", linestyle="",
        markerfacecolor="royalblue", markeredgecolor="black",
        markersize=9, label="Without Resilience"
    ),
    Line2D(
        [0], [0], marker="o", linestyle="",
        markerfacecolor="none", markeredgecolor="black",
        markeredgewidth=2, markersize=12, label="Maximum TE"
    )
]

ax.legend(handles=legend_handles, loc="best", title="Resilience")

fig.subplots_adjust(right=0.77)

cax_red = fig.add_axes([0.80, 0.55, 0.025, 0.30])
cax_blue = fig.add_axes([0.80, 0.14, 0.025, 0.30])

sm_red = ScalarMappable(
    norm=norm,
    cmap=cmaps["With Resilience"]
)

sm_blue = ScalarMappable(
    norm=norm,
    cmap=cmaps["Without Resilience"]
)

fig.colorbar(sm_red, cax=cax_red).set_label("Δ TE (%) - With")
fig.colorbar(sm_blue, cax=cax_blue).set_label("Δ TE (%) - Without")

Path("./tempPlots").mkdir(parents=True, exist_ok=True)

plt.savefig(
    _prefix_output("./tempPlots/delta_percent_exc_vs_inh_synapses_TE_norm.png"),
    dpi=300,
    bbox_inches="tight"
)

print("\nSaved: ./tempPlots/delta_percent_exc_vs_inh_synapses_TE_norm.png")

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colormaps
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.cm import ScalarMappable
from pathlib import Path

def norm_res(val):
    s = str(val).strip()
    if s in ["15", "With Resilience"]:
        return "With Resilience"
    if s in ["2", "Without Resilience"]:
        return "Without Resilience"
    return s

def sum_unique_effective(x):
    return np.unique(x.dropna()).sum()

keys = ["SZ Factor (%)", "Resilience"]

df_eff = df_all_str.copy()
df_eff["Resilience"] = df_eff["Resilience"].apply(norm_res)
df_eff["SZ Factor (%)"] = pd.to_numeric(df_eff["SZ Factor (%)"]).round(2)
df_eff["effective_synapses"] = pd.to_numeric(df_eff["effective_synapses"], errors="coerce")

df_te_norm = df_te.copy()
df_te_norm["Resilience"] = df_te_norm["Resilience"].apply(norm_res)
df_te_norm["SZ Factor (%)"] = pd.to_numeric(df_te_norm["SZ Factor (%)"]).round(2)
df_te_norm["te_mean"] = pd.to_numeric(df_te_norm["te_mean"], errors="coerce")
df_te_norm["te_std"] = pd.to_numeric(df_te_norm["te_std"], errors="coerce")

df_exc = df_eff[df_eff["synType"] == "excitatory (glutamate)"]
df_inh = df_eff[df_eff["synType"] == "inhibitory (GABA)"]

df_exc_eff = (
    df_exc.groupby(keys, as_index=False)
    .agg(eff_exc=("effective_synapses", sum_unique_effective))
)

df_inh_eff = (
    df_inh.groupby(keys, as_index=False)
    .agg(eff_inh=("effective_synapses", sum_unique_effective))
)

df_eff_xy = df_exc_eff.merge(
    df_inh_eff,
    on=keys,
    how="inner",
    validate="one_to_one"
)

df_mean_te = (
    df_te_norm.groupby(keys, as_index=False)
    .agg(
        te_mean=("te_mean", "mean"),
        te_std=("te_std", "mean")
    )
)

df_plot = df_eff_xy.merge(
    df_mean_te,
    on=keys,
    how="inner",
    validate="one_to_one"
)

df_plot = df_plot.dropna(subset=["eff_exc", "eff_inh", "te_mean"])

if df_plot.empty:
    raise ValueError("Nessuna combinazione SZ × Resilience disponibile per il plot.")

print("\n=== DATAFRAME COMPLETO ===")
print(df_plot.sort_values(keys).to_string(index=False))
print("\ndf_plot shape:", df_plot.shape)

cmaps = {
    "With Resilience": LinearSegmentedColormap.from_list(
        "Reds_trimmed",
        colormaps["Reds"](np.linspace(0.2, 1, 256))
    ),
    "Without Resilience": LinearSegmentedColormap.from_list(
        "Blues_trimmed",
        colormaps["Blues"](np.linspace(0.2, 1, 256))
    )
}

te_min = df_plot["te_mean"].min()
te_max = df_plot["te_mean"].max()

if te_min == te_max:
    te_min -= 0.5
    te_max += 0.5

norm = Normalize(vmin=te_min, vmax=te_max)

std_min = df_plot["te_std"].min()
std_max = df_plot["te_std"].max()

if pd.notna(std_min) and pd.notna(std_max) and std_max > std_min:
    df_plot["marker_size"] = 80 + 160 * (
        (df_plot["te_std"] - std_min) / (std_max - std_min)
    )
    df_plot["marker_size"] = df_plot["marker_size"].fillna(120)
else:
    df_plot["marker_size"] = 120

fig, ax = plt.subplots(figsize=(8, 6))

for res in ["With Resilience", "Without Resilience"]:
    sub = df_plot[df_plot["Resilience"] == res]

    if sub.empty:
        continue

    ax.scatter(
        sub["eff_exc"],
        sub["eff_inh"],
        c=sub["te_mean"],
        cmap=cmaps[res],
        norm=norm,
        s=sub["marker_size"],
        edgecolor="black",
        linewidth=0.6,
        alpha=0.9,
        zorder=2
    )

    idx_max = sub["te_mean"].idxmax()
    row = df_plot.loc[idx_max]

    ax.scatter(
        row["eff_exc"],
        row["eff_inh"],
        s=row["marker_size"] + 120,
        facecolors="none",
        edgecolors="black",
        linewidth=2.5,
        zorder=5
    )

    print(
        f"{res}: max TE={row['te_mean']:.4f} bit, "
        f"SZ={row['SZ Factor (%)']:.2f}%, "
        f"Exc={row['eff_exc']:.0f}, "
        f"Inh={row['eff_inh']:.0f}"
    )

ax.set_xlabel("Total effective synapses (excitatory)")
ax.set_ylabel("Total effective synapses (inhibitory)")
ax.set_title("Excitatory vs Inhibitory effective synapses")

legend_handles = [
    Line2D(
        [0], [0], marker="o", linestyle="",
        markerfacecolor="firebrick",
        markeredgecolor="black",
        markersize=9,
        label="With Resilience"
    ),
    Line2D(
        [0], [0], marker="o", linestyle="",
        markerfacecolor="royalblue",
        markeredgecolor="black",
        markersize=9,
        label="Without Resilience"
    ),
    Line2D(
        [0], [0], marker="o", linestyle="",
        markerfacecolor="none",
        markeredgecolor="black",
        markeredgewidth=2.5,
        markersize=12,
        label="Maximum TE"
    )
]

ax.legend(handles=legend_handles, title="Resilience", loc="best")

fig.subplots_adjust(right=0.77)

cax_red = fig.add_axes([0.80, 0.55, 0.025, 0.30])
cax_blue = fig.add_axes([0.80, 0.14, 0.025, 0.30])

sm_red = ScalarMappable(norm=norm, cmap=cmaps["With Resilience"])
sm_blue = ScalarMappable(norm=norm, cmap=cmaps["Without Resilience"])

fig.colorbar(sm_red, cax=cax_red).set_label("TE mean (bit) - With")
fig.colorbar(sm_blue, cax=cax_blue).set_label("TE mean (bit) - Without")

Path("./tempPlots").mkdir(parents=True, exist_ok=True)

plt.savefig(
    _prefix_output("./tempPlots/eff_exc_vs_eff_inh_colored_by_TEmean.png"),
    dpi=300,
    bbox_inches="tight"
)

print("\nSaved: ./tempPlots/eff_exc_vs_eff_inh_colored_by_TEmean.png")

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colormaps
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.cm import ScalarMappable
from pathlib import Path

keys = ["SZ Factor (%)", "Resilience"]

df_mean_te = (
    df_te_norm.groupby(keys, as_index=False)
    .agg(
        te_mean=("te_mean", "mean"),
        te_std=("te_std", "mean")
    )
)

df_plot = df_eff_xy.merge(
    df_mean_te,
    on=keys,
    how="inner",
    validate="one_to_one"
)

baseline = (
    df_eff_xy[df_eff_xy["SZ Factor (%)"] == 0]
    .set_index("Resilience")[["eff_exc", "eff_inh"]]
    .rename(columns={
        "eff_exc": "eff_exc_0",
        "eff_inh": "eff_inh_0"
    })
)

df_plot = df_plot.join(baseline, on="Resilience", validate="many_to_one")

baseline_cols = ["eff_exc_0", "eff_inh_0"]

if df_plot[baseline_cols].isna().any().any():
    raise ValueError("Baseline SZ=0 mancante per almeno una condizione.")

if (df_plot[baseline_cols] == 0).any().any():
    raise ValueError("Baseline SZ=0 uguale a zero.")

df_plot["eff_exc_pct"] = 100 * (
    df_plot["eff_exc"] / df_plot["eff_exc_0"] - 1
)

df_plot["eff_inh_pct"] = 100 * (
    df_plot["eff_inh"] / df_plot["eff_inh_0"] - 1
)

df_plot = df_plot.dropna(
    subset=["eff_exc_pct", "eff_inh_pct", "te_mean"]
).copy()

if df_plot.empty:
    raise ValueError("Nessun dato valido per il plot.")

print("\n=== BASELINE CHECK ===")
print(df_plot.loc[
    df_plot["SZ Factor (%)"] == 0,
    ["Resilience", "eff_exc_pct", "eff_inh_pct"]
])

print("\n=== DATAFRAME COMPLETO ===")
print(df_plot.sort_values(keys).to_string(index=False))

cmaps = {
    "With Resilience": LinearSegmentedColormap.from_list(
        "Reds_trimmed",
        colormaps["Reds"](np.linspace(0.2, 1, 256))
    ),
    "Without Resilience": LinearSegmentedColormap.from_list(
        "Blues_trimmed",
        colormaps["Blues"](np.linspace(0.2, 1, 256))
    )
}

te_min = df_plot["te_mean"].min()
te_max = df_plot["te_mean"].max()

if te_min == te_max:
    te_min -= 0.5
    te_max += 0.5

norm = Normalize(vmin=te_min, vmax=te_max)

std_min = df_plot["te_std"].min()
std_max = df_plot["te_std"].max()

if pd.notna(std_min) and pd.notna(std_max) and std_max > std_min:
    df_plot["marker_size"] = 80 + 160 * (
        (df_plot["te_std"] - std_min) / (std_max - std_min)
    )
    df_plot["marker_size"] = df_plot["marker_size"].fillna(120)
else:
    df_plot["marker_size"] = 120

fig, ax = plt.subplots(figsize=(8, 6))

for res in ["With Resilience", "Without Resilience"]:
    sub = df_plot[df_plot["Resilience"] == res]

    if sub.empty:
        continue

    ax.scatter(
        sub["eff_exc_pct"],
        sub["eff_inh_pct"],
        c=sub["te_mean"],
        cmap=cmaps[res],
        norm=norm,
        s=sub["marker_size"],
        edgecolor="black",
        linewidth=0.6,
        alpha=0.9,
        zorder=2
    )

    idx_max = sub["te_mean"].idxmax()
    row = df_plot.loc[idx_max]

    ax.scatter(
        row["eff_exc_pct"],
        row["eff_inh_pct"],
        s=row["marker_size"] + 120,
        facecolors="none",
        edgecolors="black",
        linewidth=2.5,
        zorder=5
    )

    print(
        f"{res}: max TE={row['te_mean']:.4f} bit, "
        f"SZ={row['SZ Factor (%)']:.2f}%, "
        f"ΔExc={row['eff_exc_pct']:.2f}%, "
        f"ΔInh={row['eff_inh_pct']:.2f}%"
    )

ax.axhline(0, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
ax.axvline(0, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)

ax.set_xlabel("Δ Effective excitatory synapses (%) vs SZ=0")
ax.set_ylabel("Δ Effective inhibitory synapses (%) vs SZ=0")
ax.set_title("Effective synaptic changes and transfer entropy")

legend_handles = [
    Line2D(
        [0], [0], marker="o", linestyle="",
        markerfacecolor="firebrick", markeredgecolor="black",
        markersize=9, label="With Resilience"
    ),
    Line2D(
        [0], [0], marker="o", linestyle="",
        markerfacecolor="royalblue", markeredgecolor="black",
        markersize=9, label="Without Resilience"
    ),
    Line2D(
        [0], [0], marker="o", linestyle="",
        markerfacecolor="none", markeredgecolor="black",
        markeredgewidth=2.5, markersize=12,
        label="Maximum TE"
    )
]

ax.legend(handles=legend_handles, title="Resilience", loc="best")

fig.subplots_adjust(right=0.77)

cax_red = fig.add_axes([0.80, 0.55, 0.025, 0.30])
cax_blue = fig.add_axes([0.80, 0.14, 0.025, 0.30])

sm_red = ScalarMappable(norm=norm, cmap=cmaps["With Resilience"])
sm_blue = ScalarMappable(norm=norm, cmap=cmaps["Without Resilience"])

fig.colorbar(sm_red, cax=cax_red).set_label("TE mean (bit) - With")
fig.colorbar(sm_blue, cax=cax_blue).set_label("TE mean (bit) - Without")

Path("./tempPlots").mkdir(parents=True, exist_ok=True)

plt.savefig(
    _prefix_output("./tempPlots/delta_eff_exc_vs_inh_colored_by_TEmean.png"),
    dpi=300,
    bbox_inches="tight"
)

print("\nSaved: ./tempPlots/delta_eff_exc_vs_inh_colored_by_TEmean.png")

plt.show()

In [ ]:
# =========================================================
# 6. FIGURA: SZ Factor vs effective_exc / effective_inh
# =========================================================

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 8), sharex=True)

colors_res = {
    "With Resilience": "red",
    "Without Resilience": "blue",
}

for res in ["With Resilience", "Without Resilience"]:
    sub = df_plot[df_plot["Resilience"] == res].sort_values("SZ Factor (%)")
    if sub.empty:
        continue

    # --------- eccitatorie ----------
    ax1.plot(
        sub["SZ Factor (%)"],
        sub["eff_exc"],
        marker="o",
        linestyle="-",
        color=colors_res[res],
        label=res,
    )

    # --------- inibitorie ----------
    ax2.plot(
        sub["SZ Factor (%)"],
        sub["eff_inh"],
        marker="o",
        linestyle="-",
        color=colors_res[res],
        label=res,
    )

ax1.set_ylabel("Total effective synapses (excitatory)")
ax1.set_title("SZ Factor vs effective synapses")

ax2.set_xlabel("SZ Factor (%)")
ax2.set_ylabel("Total effective synapses (inhibitory)")

ax1.legend(title="Resilience", loc="upper right")
ax2.legend(title="Resilience", loc="upper right")

plt.tight_layout()
plt.savefig(_prefix_output("./tempPlots/SZ_vs_effective_synapses_exc_inh.png"), dpi=300)
print("Saved: ./tempPlots/SZ_vs_effective_synapses_exc_inh.png")

plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colormaps
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.cm import ScalarMappable
from pathlib import Path

def norm_res(val):
    s = str(val).strip()
    if s in ["15", "15.0", "With Resilience"]:
        return "With Resilience"
    if s in ["2", "2.0", "Without Resilience"]:
        return "Without Resilience"
    return s

def sum_unique_synapses(x):
    return np.unique(x.dropna()).sum()

keys = ["SZ Factor (%)", "Resilience"]

df_str = df_all_str.copy()
df_str["Resilience"] = df_str["Resilience"].apply(norm_res)
df_str["SZ Factor (%)"] = pd.to_numeric(df_str["SZ Factor (%)"]).round(2)
df_str["synapses"] = pd.to_numeric(df_str["synapses"], errors="coerce")
df_str["total_cells"] = pd.to_numeric(df_str["total_cells"], errors="coerce")

df_te_norm = df_te.copy()
df_te_norm["Resilience"] = df_te_norm["Resilience"].apply(norm_res)
df_te_norm["SZ Factor (%)"] = pd.to_numeric(df_te_norm["SZ Factor (%)"]).round(2)
df_te_norm["te_mean"] = pd.to_numeric(df_te_norm["te_mean"], errors="coerce")
df_te_norm["te_std"] = pd.to_numeric(df_te_norm["te_std"], errors="coerce")

df_total = (
    df_str.groupby(keys, as_index=False)
    .agg(
        total_cells=("total_cells", "first"),
        syn_total=("synapses", sum_unique_synapses)
    )
)

df_exc = df_str[df_str["synType"] == "excitatory (glutamate)"]
df_inh = df_str[df_str["synType"] == "inhibitory (GABA)"]

df_exc_struct = (
    df_exc.groupby(keys, as_index=False)
    .agg(syn_exc=("synapses", sum_unique_synapses))
)

df_inh_struct = (
    df_inh.groupby(keys, as_index=False)
    .agg(syn_inh=("synapses", sum_unique_synapses))
)

df_struct = (
    df_total
    .merge(df_exc_struct, on=keys, how="inner", validate="one_to_one")
    .merge(df_inh_struct, on=keys, how="inner", validate="one_to_one")
)

df_mean_te = (
    df_te_norm.groupby(keys, as_index=False)
    .agg(
        te_mean=("te_mean", "mean"),
        te_std=("te_std", "mean")
    )
)

df_plot = df_struct.merge(
    df_mean_te,
    on=keys,
    how="inner",
    validate="one_to_one"
)

if df_plot.empty:
    raise ValueError("Nessuna combinazione SZ × Resilience comune ai DataFrame.")

baseline = (
    df_plot[df_plot["SZ Factor (%)"] == 0]
    .set_index("Resilience")[
        ["total_cells", "syn_total", "syn_exc", "syn_inh", "te_mean"]
    ]
    .rename(columns={
        "total_cells": "cells_0",
        "syn_total": "syn_total_0",
        "syn_exc": "syn_exc_0",
        "syn_inh": "syn_inh_0",
        "te_mean": "te_mean_0"
    })
)

df_plot = df_plot.join(
    baseline,
    on="Resilience",
    validate="many_to_one"
)

baseline_cols = [
    "cells_0", "syn_total_0", "syn_exc_0",
    "syn_inh_0", "te_mean_0"
]

if df_plot[baseline_cols].isna().any().any():
    raise ValueError("Baseline SZ=0 mancante per almeno una condizione.")

if (df_plot[baseline_cols] == 0).any().any():
    raise ValueError("Baseline SZ=0 uguale a zero: Δ% non calcolabile.")

pct_vars = {
    "cells_pct": ("total_cells", "cells_0"),
    "syn_pct": ("syn_total", "syn_total_0"),
    "exc_pct": ("syn_exc", "syn_exc_0"),
    "inh_pct": ("syn_inh", "syn_inh_0"),
    "te_pct": ("te_mean", "te_mean_0")
}

for output, (value, reference) in pct_vars.items():
    df_plot[output] = 100 * (
        df_plot[value] / df_plot[reference] - 1
    )

print("\n=== BASELINE SZ=0 ===")
print(df_plot.loc[
    df_plot["SZ Factor (%)"] == 0,
    ["Resilience", "cells_pct", "syn_pct", "exc_pct", "inh_pct", "te_pct"]
])

print("\n=== DATAFRAME COMPLETO ===")
print(df_plot.sort_values(keys).to_string(index=False))

df_plot = df_plot.replace([np.inf, -np.inf], np.nan)
df_plot = df_plot.dropna(
    subset=["cells_pct", "syn_pct", "exc_pct", "inh_pct", "te_pct"]
).copy()

if df_plot.empty:
    raise ValueError("Nessun dato valido per la figura.")

cmaps = {
    "With Resilience": LinearSegmentedColormap.from_list(
        "Reds_trimmed",
        colormaps["Reds"](np.linspace(0.2, 1, 256))
    ),
    "Without Resilience": LinearSegmentedColormap.from_list(
        "Blues_trimmed",
        colormaps["Blues"](np.linspace(0.2, 1, 256))
    )
}

markers = {
    "With Resilience": "o",
    "Without Resilience": "s"
}

te_min = df_plot["te_pct"].min()
te_max = df_plot["te_pct"].max()

if te_min == te_max:
    te_min -= 0.5
    te_max += 0.5

norm = Normalize(vmin=te_min, vmax=te_max)

std_min = df_plot["te_std"].min()
std_max = df_plot["te_std"].max()

if pd.notna(std_min) and pd.notna(std_max) and std_max > std_min:
    df_plot["marker_size"] = 80 + 160 * (
        (df_plot["te_std"] - std_min) / (std_max - std_min)
    )
    df_plot["marker_size"] = df_plot["marker_size"].fillna(120)
else:
    df_plot["marker_size"] = 120

fig, axes = plt.subplots(2, 1, figsize=(9, 12))
ax1, ax2 = axes

panels = [
    (
        ax1,
        "exc_pct",
        "inh_pct",
        "A",
        "Δ Excitatory synapses (%) vs SZ=0",
        "Δ Inhibitory synapses (%) vs SZ=0"
    ),
    (
        ax2,
        "cells_pct",
        "syn_pct",
        "B",
        "Δ Total cells (%) vs SZ=0",
        "Δ Total synapses (%) vs SZ=0"
    )
]

for ax, xcol, ycol, panel, xlabel, ylabel in panels:

    for res in ["With Resilience", "Without Resilience"]:
        sub = df_plot[df_plot["Resilience"] == res]

        if sub.empty:
            continue

        ax.scatter(
            sub[xcol],
            sub[ycol],
            c=sub["te_pct"],
            cmap=cmaps[res],
            norm=norm,
            s=sub["marker_size"],
            marker=markers[res],
            edgecolor="black",
            linewidth=0.6,
            alpha=0.9,
            zorder=2
        )

        idx_max = sub["te_mean"].idxmax()
        row = df_plot.loc[idx_max]

        ax.scatter(
            row[xcol],
            row[ycol],
            s=row["marker_size"] + 120,
            marker=markers[res],
            facecolors="none",
            edgecolors="black",
            linewidth=2.5,
            zorder=5
        )

    ax.axhline(0, color="gray", linestyle=":", linewidth=0.8, alpha=0.6)
    ax.axvline(0, color="gray", linestyle=":", linewidth=0.8, alpha=0.6)

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(panel, loc="left", fontweight="bold", fontsize=14)
    ax.grid(alpha=0.15)

xlims = ax1.get_xlim()
ylims = ax1.get_ylim()

min_lim = min(xlims[0], ylims[0])
max_lim = max(xlims[1], ylims[1])

ax1.plot(
    [min_lim, max_lim],
    [min_lim, max_lim],
    linestyle="--",
    color="gray",
    linewidth=1.2,
    alpha=0.8,
    zorder=1
)

ax1.set_xlim(min_lim, max_lim)
ax1.set_ylim(min_lim, max_lim)
ax1.set_aspect("equal", adjustable="box")

legend_handles = [
    Line2D(
        [0], [0], marker="o", linestyle="",
        markerfacecolor="firebrick",
        markeredgecolor="black",
        markersize=9,
        label="With Resilience"
    ),
    Line2D(
        [0], [0], marker="s", linestyle="",
        markerfacecolor="royalblue",
        markeredgecolor="black",
        markersize=9,
        label="Without Resilience"
    ),
    Line2D(
        [0], [0], marker="o", linestyle="",
        markerfacecolor="none",
        markeredgecolor="black",
        markeredgewidth=2.5,
        markersize=12,
        label="Maximum TE"
    ),
    Line2D(
        [0], [0], linestyle="--",
        color="gray",
        linewidth=1.2,
        label="Equal relative change (y=x)"
    )
]

ax1.legend(
    handles=legend_handles,
    title="Resilience",
    loc="best",
    fontsize=9
)

ax2.legend(
    handles=legend_handles[:3],
    title="Resilience",
    loc="best",
    fontsize=9
)

fig.subplots_adjust(
    left=0.14,
    right=0.78,
    top=0.96,
    bottom=0.07,
    hspace=0.30
)

cax_red = fig.add_axes([0.81, 0.56, 0.025, 0.30])
cax_blue = fig.add_axes([0.81, 0.15, 0.025, 0.30])

sm_red = ScalarMappable(
    norm=norm,
    cmap=cmaps["With Resilience"]
)

sm_blue = ScalarMappable(
    norm=norm,
    cmap=cmaps["Without Resilience"]
)

fig.colorbar(
    sm_red,
    cax=cax_red
).set_label("Δ TE (%) - With Resilience")

fig.colorbar(
    sm_blue,
    cax=cax_blue
).set_label("Δ TE (%) - Without Resilience")

Path("./tempPlots").mkdir(parents=True, exist_ok=True)

output = "./tempPlots/combined_EI_and_cells_synapses_TE.png"

fig.savefig(
    _prefix_output(output),
    dpi=300,
    bbox_inches="tight"
)

print(f"\nSaved: {output}")

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colormaps
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.cm import ScalarMappable
from pathlib import Path

def norm_res(val):
    s = str(val).strip()
    if s in ["15", "15.0", "With Resilience"]:
        return "With Resilience"
    if s in ["2", "2.0", "Without Resilience"]:
        return "Without Resilience"
    return s

keys = ["SZ Factor (%)", "Resilience"]

df_str = df_all_str.copy()
df_str["Resilience"] = df_str["Resilience"].apply(norm_res)
df_str["SZ Factor (%)"] = pd.to_numeric(df_str["SZ Factor (%)"]).round(2)
df_str["total_cells"] = pd.to_numeric(df_str["total_cells"], errors="coerce")
df_str["synapses"] = pd.to_numeric(df_str["synapses"], errors="coerce")

df_te_norm = df_te.copy()
df_te_norm["Resilience"] = df_te_norm["Resilience"].apply(norm_res)
df_te_norm["SZ Factor (%)"] = pd.to_numeric(df_te_norm["SZ Factor (%)"]).round(2)
df_te_norm["te_mean"] = pd.to_numeric(df_te_norm["te_mean"], errors="coerce")
df_te_norm["te_std"] = pd.to_numeric(df_te_norm["te_std"], errors="coerce")

df_mean_struct = (
    df_str.groupby(keys, as_index=False)
    .agg(
        total_cells=("total_cells", "mean"),
        synapses=("synapses", "mean")
    )
)

df_mean_te = (
    df_te_norm.groupby(keys, as_index=False)
    .agg(
        te_mean=("te_mean", "mean"),
        te_std=("te_std", "mean")
    )
)

df_plot = df_mean_struct.merge(
    df_mean_te,
    on=keys,
    how="inner",
    validate="one_to_one"
)

if df_plot.empty:
    raise ValueError("Nessuna combinazione SZ × Resilience comune.")

baseline = (
    df_plot[df_plot["SZ Factor (%)"] == 0]
    .set_index("Resilience")[["total_cells", "synapses", "te_mean"]]
    .rename(columns={
        "total_cells": "cells_0",
        "synapses": "synapses_0",
        "te_mean": "te_mean_0"
    })
)

df_plot = df_plot.join(
    baseline,
    on="Resilience",
    validate="many_to_one"
)

baseline_cols = ["cells_0", "synapses_0", "te_mean_0"]

if df_plot[baseline_cols].isna().any().any():
    raise ValueError("Baseline SZ=0 mancante per almeno una condizione.")

if (df_plot[baseline_cols] == 0).any().any():
    raise ValueError("Baseline SZ=0 uguale a zero: Δ% non calcolabile.")

df_plot["cells_delta_pct"] = 100 * (
    df_plot["total_cells"] / df_plot["cells_0"] - 1
)

df_plot["syn_delta_pct"] = 100 * (
    df_plot["synapses"] / df_plot["synapses_0"] - 1
)

df_plot["te_delta_pct"] = 100 * (
    df_plot["te_mean"] / df_plot["te_mean_0"] - 1
)

df_plot["te_std_pct"] = 100 * (
    df_plot["te_std"] / df_plot["te_mean_0"].abs()
)

df_plot = df_plot.replace([np.inf, -np.inf], np.nan)
df_plot = df_plot.dropna(
    subset=["cells_delta_pct", "syn_delta_pct", "te_delta_pct"]
).copy()

if df_plot.empty:
    raise ValueError("Nessun dato valido per il plot.")

print("\n=== BASELINE CHECK ===")
print(df_plot.loc[
    df_plot["SZ Factor (%)"] == 0,
    ["Resilience", "cells_delta_pct", "syn_delta_pct", "te_delta_pct"]
])

print("\n=== DATAFRAME COMPLETO ===")
print(df_plot.sort_values(keys).to_string(index=False))

cmaps = {
    "With Resilience": LinearSegmentedColormap.from_list(
        "Reds_trimmed",
        colormaps["Reds"](np.linspace(0.2, 1, 256))
    ),
    "Without Resilience": LinearSegmentedColormap.from_list(
        "Blues_trimmed",
        colormaps["Blues"](np.linspace(0.2, 1, 256))
    )
}

markers = {
    "With Resilience": "o",
    "Without Resilience": "s"
}

te_min = df_plot["te_delta_pct"].min()
te_max = df_plot["te_delta_pct"].max()

if te_min == te_max:
    te_min -= 0.5
    te_max += 0.5

norm = Normalize(vmin=te_min, vmax=te_max)

std_min = df_plot["te_std_pct"].min()
std_max = df_plot["te_std_pct"].max()

if pd.notna(std_min) and pd.notna(std_max) and std_max > std_min:
    df_plot["marker_size"] = 80 + 160 * (
        (df_plot["te_std_pct"] - std_min) / (std_max - std_min)
    )
    df_plot["marker_size"] = df_plot["marker_size"].fillna(120)
else:
    df_plot["marker_size"] = 120

fig, ax = plt.subplots(figsize=(8, 6))

for res in ["With Resilience", "Without Resilience"]:
    sub = df_plot[df_plot["Resilience"] == res]

    if sub.empty:
        continue

    ax.scatter(
        sub["cells_delta_pct"],
        sub["syn_delta_pct"],
        c=sub["te_delta_pct"],
        cmap=cmaps[res],
        norm=norm,
        s=sub["marker_size"],
        marker=markers[res],
        edgecolor="black",
        linewidth=0.6,
        alpha=0.9,
        zorder=2
    )

    idx_max = sub["te_delta_pct"].idxmax()
    row = df_plot.loc[idx_max]

    ax.scatter(
        row["cells_delta_pct"],
        row["syn_delta_pct"],
        s=row["marker_size"] + 120,
        marker=markers[res],
        facecolors="none",
        edgecolors="black",
        linewidth=2.5,
        zorder=5
    )

    print(
        f"{res}: "
        f"max ΔTE={row['te_delta_pct']:.2f}%, "
        f"SZ={row['SZ Factor (%)']:.2f}%, "
        f"ΔCells={row['cells_delta_pct']:.2f}%, "
        f"ΔSynapses={row['syn_delta_pct']:.2f}%, "
        f"TE std={row['te_std_pct']:.2f}%"
    )

ax.axhline(0, color="gray", linestyle=":", linewidth=0.8, alpha=0.6)
ax.axvline(0, color="gray", linestyle=":", linewidth=0.8, alpha=0.6)

ax.set_xlabel("Δ Total cells (%) vs SZ=0")
ax.set_ylabel("Δ Synapses (%) vs SZ=0")
ax.set_title("Structural changes and relative transfer entropy")
ax.grid(alpha=0.15)

legend_handles = [
    Line2D(
        [0], [0], marker="o", linestyle="",
        markerfacecolor="firebrick",
        markeredgecolor="black",
        markersize=9,
        label="With Resilience"
    ),
    Line2D(
        [0], [0], marker="s", linestyle="",
        markerfacecolor="royalblue",
        markeredgecolor="black",
        markersize=9,
        label="Without Resilience"
    ),
    Line2D(
        [0], [0], marker="o", linestyle="",
        markerfacecolor="none",
        markeredgecolor="black",
        markeredgewidth=2.5,
        markersize=12,
        label="Maximum ΔTE"
    )
]

ax.legend(
    handles=legend_handles,
    title="Resilience",
    loc="best"
)

fig.subplots_adjust(
    left=0.14,
    right=0.77,
    top=0.91,
    bottom=0.13
)

cax_red = fig.add_axes([0.80, 0.55, 0.025, 0.30])
cax_blue = fig.add_axes([0.80, 0.14, 0.025, 0.30])

sm_red = ScalarMappable(
    norm=norm,
    cmap=cmaps["With Resilience"]
)

sm_blue = ScalarMappable(
    norm=norm,
    cmap=cmaps["Without Resilience"]
)

fig.colorbar(
    sm_red,
    cax=cax_red
).set_label("Δ TE mean (%) - With")

fig.colorbar(
    sm_blue,
    cax=cax_blue
).set_label("Δ TE mean (%) - Without")

Path("./tempPlots").mkdir(parents=True, exist_ok=True)

output = "./tempPlots/delta_cells_vs_delta_synapses_colored_by_delta_TEmean.png"

fig.savefig(
    _prefix_output(output),
    dpi=300,
    bbox_inches="tight"
)

print(f"\nSaved: {output}")

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D

# ==========================
# 1. Normalizza Resilience / SZ
# ==========================

def norm_res(val):
    s = str(val).strip()
    if s in ["15", "With Resilience"]:
        return "With Resilience"
    if s in ["2", "Without Resilience"]:
        return "Without Resilience"
    return s

df_str = df_all_str.copy()
df_str["Resilience"]    = df_str["Resilience"].apply(norm_res)
df_str["SZ Factor (%)"] = df_str["SZ Factor (%)"].astype(float).round(2)

df_te_norm = df_te.copy()
df_te_norm["Resilience"]    = df_te_norm["Resilience"].apply(norm_res)
df_te_norm["SZ Factor (%)"] = df_te_norm["SZ Factor (%)"].astype(float).round(2)

# ==========================
# 2. Media sinapsi per SZ×Resilience
# ==========================

df_mean_struct = (
    df_str
    .groupby(["SZ Factor (%)", "Resilience"], as_index=False)
    .agg({"synapses": "mean"})
)

# ==========================
# 3. Merge con TE media
# ==========================

df_plot = df_mean_struct.merge(
    df_te_norm[["SZ Factor (%)", "Resilience", "te_mean"]],
    on=["SZ Factor (%)", "Resilience"],
    how="inner"
)

# ==========================
# 4. Calcola seconda derivata di TE_mean vs SZ
# ==========================

df_plot["te_second"] = np.nan

for res in ["With Resilience", "Without Resilience"]:
    sub = df_plot[df_plot["Resilience"] == res].copy()
    if sub.empty:
        continue

    sub = sub.sort_values("SZ Factor (%)")
    x = sub["SZ Factor (%)"].values.astype(float)
    y = sub["te_mean"].values.astype(float)

    if len(x) >= 3:
        d1 = np.gradient(y, x)
        d2 = np.gradient(d1, x)
        df_plot.loc[sub.index, "te_second"] = d2

# togli righe senza seconda derivata
df_plot = df_plot.dropna(subset=["te_second", "synapses", "SZ Factor (%)"])

# ==========================
# 5. Plot: synapses vs seconda derivata TE, colore = TE_mean
# ==========================

fig, ax = plt.subplots(figsize=(7, 6))

# colormap TE
te_min = df_plot["te_mean"].min()
te_max = df_plot["te_mean"].max()
norm = Normalize(vmin=te_min, vmax=te_max)
cmap = plt.cm.plasma   # o viridis/coolwarm se preferisci

marker_map = {
    "With Resilience": "o",
    "Without Resilience": "s"
}

for res in ["With Resilience", "Without Resilience"]:
    sub = df_plot[df_plot["Resilience"] == res].copy()
    if sub.empty:
        continue

    sc = ax.scatter(
        sub["synapses"],
        sub["te_second"],
        c=sub["te_mean"],
        cmap=cmap,
        norm=norm,
        s=70,
        marker=marker_map[res],
        edgecolor="k",
        linewidth=0.6,
        alpha=0.9,
        label=res
    )

    # cerchio nero sul massimo TE_mean (puoi cambiare in max te_second se vuoi)
    idx_max = sub["te_mean"].idxmax()
    ax.scatter(
        df_plot.loc[idx_max, "synapses"],
        df_plot.loc[idx_max, "te_second"],
        s=130,
        facecolors="none",
        edgecolors="black",
        linewidth=2.0,
        zorder=5
    )

# assi e stile
ax.set_xlabel("Mean synapses per SZ×Resilience")
ax.set_ylabel("Second derivative of TE_mean vs SZ")
ax.set_title("Synapses vs d²(TE_mean)/dSZ²\n(colore = TE_mean)")
ax.grid(False)

# niente notazione scientifica in y
ax.ticklabel_format(style='plain', axis='y')

# legenda Resilience (marker diversi)
legend_handles = [
    Line2D([0], [0], marker='o', linestyle='', markerfacecolor='grey',
           markeredgecolor='k', label='With Resilience'),
    Line2D([0], [0], marker='s', linestyle='', markerfacecolor='grey',
           markeredgecolor='k', label='Without Resilience')
]
ax.legend(handles=legend_handles, title="Resilience", loc="best")

# colorbar TE_mean
sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label("TE_mean (bit)")

plt.tight_layout()
plt.savefig(_prefix_output("./tempPlots/synapses_vs_TE_second_colored_by_TEmean.png"), dpi=300)
print("Saved: ./tempPlots/synapses_vs_TE_second_colored_by_TEmean.png")

plt.show()


In [ ]:
# ==========================
# 5. Plot: synapses vs seconda derivata TE
#    colore = TE_mean (Reds/Blues per Resilience)
# ==========================

fig, ax = plt.subplots(figsize=(7, 6))

# normalizzazione globale di TE_mean
te_min = df_plot["te_mean"].min()
te_max = df_plot["te_mean"].max()
norm = Normalize(vmin=te_min, vmax=te_max)

# colormap separate per le due condizioni
cmaps = {
    "With Resilience": plt.cm.Reds,
    "Without Resilience": plt.cm.Blues
}

marker_map = {
    "With Resilience": "o",
    "Without Resilience": "s"
}

for res in ["With Resilience", "Without Resilience"]:
    sub = df_plot[df_plot["Resilience"] == res].copy()
    if sub.empty:
        continue

    # colori RGBA: Reds/Blues con intensità proporzionale a TE_mean
    colors = cmaps[res](norm(sub["te_mean"].values))

    ax.scatter(
        sub["synapses"],
        sub["te_second"],
        c=colors,
        s=70,
        marker=marker_map[res],
        edgecolor="k",
        linewidth=0.6,
        alpha=0.9,
        label=res
    )

    # cerchio nero sul massimo TE_mean in quella condizione
    idx_max = sub["te_mean"].idxmax()
    ax.scatter(
        df_plot.loc[idx_max, "synapses"],
        df_plot.loc[idx_max, "te_second"],
        s=130,
        facecolors="none",
        edgecolors="black",
        linewidth=2.0,
        zorder=5
    )

# assi e stile
ax.set_xlabel("Mean synapses per SZ×Resilience")
ax.set_ylabel("Second derivative of TE_mean vs SZ")
ax.set_title("Synapses vs d²(TE_mean)/dSZ²\n(colore = TE_mean, Reds/Blues)")
ax.grid(False)
ax.ticklabel_format(style='plain', axis='y')

# legenda per le due Resilience
legend_handles = [
    Line2D([0], [0], marker='o', linestyle='',
           markerfacecolor=cmaps["With Resilience"](norm((te_min+te_max)/2)),
           markeredgecolor='k', label='With Resilience'),
    Line2D([0], [0], marker='s', linestyle='',
           markerfacecolor=cmaps["Without Resilience"](norm((te_min+te_max)/2)),
           markeredgecolor='k', label='Without Resilience')
]
ax.legend(handles=legend_handles, title="Resilience", loc="best")

# colorbar per TE_mean (scala comune, indipendente dal fatto che sia rosso o blu)
sm = plt.cm.ScalarMappable(norm=norm, cmap=plt.cm.coolwarm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label("TE_mean (bit)")

plt.tight_layout()
plt.savefig(_prefix_output("./tempPlots/synapses_vs_TE_second_colored_by_TEmean_RB.png"), dpi=300)
print("Saved: ./tempPlots/synapses_vs_TE_second_colored_by_TEmean_RB.png")

plt.show()


In [ ]:
# ==========================
# 5. Plot: synapses vs seconda derivata TE
#    colore = TE_mean (Reds/Blues per Resilience)
# ==========================

fig, ax = plt.subplots(figsize=(7, 6))

# normalizzazione globale di TE_mean
te_min = df_plot["te_mean"].min()
te_max = df_plot["te_mean"].max()
norm = Normalize(vmin=te_min, vmax=te_max)

# colormap separate per le due condizioni
cmaps = {
    "With Resilience": plt.cm.Reds,
    "Without Resilience": plt.cm.Blues
}

marker_map = {
    "With Resilience": "o",
    "Without Resilience": "s"
}

line_colors = {
    "With Resilience": "darkred",
    "Without Resilience": "darkblue"
}

for res in ["With Resilience", "Without Resilience"]:
    sub = df_plot[df_plot["Resilience"] == res].copy()
    if sub.empty:
        continue

    # ordina per SZ (fondamentale per le linee!)
    sub = sub.sort_values("SZ Factor (%)")

    # colori RGBA: Reds/Blues con intensità proporzionale a TE_mean
    colors = cmaps[res](norm(sub["te_mean"].values))

    # === LINEA tra i punti ===
    ax.plot(
        sub["synapses"],
        sub["te_second"],
        color=line_colors[res],
        linewidth=1.8,
        alpha=0.7,
        zorder=1
    )

    # === SCATTER colorato da TE ===
    ax.scatter(
        sub["synapses"],
        sub["te_second"],
        c=colors,
        s=80,
        marker=marker_map[res],
        edgecolor="k",
        linewidth=0.6,
        alpha=0.95,
        label=res
    )

    # cerchio nero sul massimo TE_mean
    idx_max = sub["te_mean"].idxmax()
    ax.scatter(
        df_plot.loc[idx_max, "synapses"],
        df_plot.loc[idx_max, "te_second"],
        s=150,
        facecolors="none",
        edgecolors="black",
        linewidth=2.2,
        zorder=6
    )

# assi e stile
ax.set_xlabel("Mean synapses per SZ×Resilience")
ax.set_ylabel("Second derivative of TE_mean vs SZ")
ax.set_title("Synapses vs d²(TE_mean)/dSZ²\n(colore = TE_mean, Reds/Blues)")
ax.grid(False)
ax.ticklabel_format(style='plain', axis='y')

# legenda Resilience
legend_handles = [
    Line2D([0], [0], marker='o', linestyle='', 
           markerfacecolor=cmaps["With Resilience"](norm((te_min+te_max)/2)),
           markeredgecolor='k', label='With Resilience'),
    Line2D([0], [0], marker='s', linestyle='',
           markerfacecolor=cmaps["Without Resilience"](norm((te_min+te_max)/2)),
           markeredgecolor='k', label='Without Resilience')
]
ax.legend(handles=legend_handles, title="Resilience", loc="best")

# colorbar TE_mean
sm = plt.cm.ScalarMappable(norm=norm, cmap=plt.cm.coolwarm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label("TE_mean (bit)")

plt.tight_layout()
plt.savefig(_prefix_output("./tempPlots/synapses_vs_TE_second_colored_by_TEmean_RB_lines.png"), dpi=300)
print("Saved: ./tempPlots/synapses_vs_TE_second_colored_by_TEmean_RB_lines.png")

plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D

# ==========================
# 1. Normalizza Resilience / SZ
# ==========================

def norm_res(val):
    s = str(val).strip()
    if s in ["15", "With Resilience"]:
        return "With Resilience"
    if s in ["2", "Without Resilience"]:
        return "Without Resilience"
    return s

df_str = df_all_str.copy()
df_str["Resilience"]    = df_str["Resilience"].apply(norm_res)
df_str["SZ Factor (%)"] = df_str["SZ Factor (%)"].astype(float).round(2)

df_te_norm = df_te.copy()
df_te_norm["Resilience"]    = df_te_norm["Resilience"].apply(norm_res)
df_te_norm["SZ Factor (%)"] = df_te_norm["SZ Factor (%)"].astype(float).round(2)

# ==========================
# 2. Media sinapsi per SZ×Resilience
# ==========================

df_mean_struct = (
    df_str
    .groupby(["SZ Factor (%)", "Resilience"], as_index=False)
    .agg({"synapses": "mean"})
)

# ==========================
# 3. Merge con TE media
# ==========================

df_plot = df_mean_struct.merge(
    df_te_norm[["SZ Factor (%)", "Resilience", "te_mean"]],
    on=["SZ Factor (%)", "Resilience"],
    how="inner"
)

# ==========================
# 4. Calcola seconda derivata TE_mean per ciascuna Resilience
# ==========================

df_plot["te_second"] = np.nan
eps = 1e-9

for res in ["With Resilience", "Without Resilience"]:
    sub = df_plot[df_plot["Resilience"] == res].copy()
    if sub.empty:
        continue

    sub = sub.sort_values("SZ Factor (%)")
    x = sub["SZ Factor (%)"].values
    y = sub["te_mean"].values

    if len(x) >= 3:
        d1 = np.gradient(y, x)
        d2 = np.gradient(d1, x)
        df_plot.loc[sub.index, "te_second"] = d2

df_plot = df_plot.dropna(subset=["te_second"])

# ==========================
# 5. Calcolo dei Δ% rispetto a SZ = 0
# ==========================

df_plot["syn_delta_pct"] = np.nan
df_plot["d2te_delta_pct"] = np.nan

for res in ["With Resilience", "Without Resilience"]:
    sub = df_plot[df_plot["Resilience"] == res].sort_values("SZ Factor (%)")
    if sub.empty:
        continue

    base_syn = sub.iloc[0]["synapses"]
    base_d2te = sub.iloc[0]["te_second"]

    for idx, row in sub.iterrows():
        syn = row["synapses"]
        d2 = row["te_second"]

        df_plot.loc[idx, "syn_delta_pct"] = (syn - base_syn) / base_syn * 100

        df_plot.loc[idx, "d2te_delta_pct"] = (d2 - base_d2te) / (abs(base_d2te) + eps) * 100

# ==========================
# 6. PLOT: Δ% synapsi vs Δ% seconda derivata TE
# ==========================

fig, ax = plt.subplots(figsize=(7, 6))

# normalizzazione globale TE_mean
te_min = df_plot["te_mean"].min()
te_max = df_plot["te_mean"].max()
norm = Normalize(vmin=te_min, vmax=te_max)

cmaps = {
    "With Resilience": plt.cm.Reds,
    "Without Resilience": plt.cm.Blues
}

marker_map = {"With Resilience": "o", "Without Resilience": "s"}
line_colors = {"With Resilience": "darkred", "Without Resilience": "darkblue"}

for res in ["With Resilience", "Without Resilience"]:
    sub = df_plot[df_plot["Resilience"] == res].sort_values("SZ Factor (%)")
    if sub.empty:
        continue

    colors = cmaps[res](norm(sub["te_mean"].values))

    # line
    ax.plot(
        sub["syn_delta_pct"],
        sub["d2te_delta_pct"],
        color=line_colors[res],
        linewidth=2,
        alpha=0.7
    )

    # scatter
    ax.scatter(
        sub["syn_delta_pct"],
        sub["d2te_delta_pct"],
        c=colors,
        s=90,
        marker=marker_map[res],
        edgecolor="k",
        linewidth=0.8
    )

    # cerchia massimo TE_mean
    idx_max = sub["te_mean"].idxmax()
    ax.scatter(
        df_plot.loc[idx_max, "syn_delta_pct"],
        df_plot.loc[idx_max, "d2te_delta_pct"],
        s=160,
        facecolors="none",
        edgecolors="black",
        linewidth=2.4
    )

# Assi
ax.set_xlabel("Δ% Synapses vs SZ=0")
ax.set_ylabel("Δ% d²(TE_mean)/dSZ² vs SZ=0")
ax.set_title("ΔSynapses vs ΔSecond Derivative of TE\n(colore = TE_mean Reds/Blues)")

# Legenda
legend_handles = [
    Line2D([0], [0], marker='o', linestyle='', markerfacecolor='lightcoral',
           markeredgecolor='k', label='With Resilience'),
    Line2D([0], [0], marker='s', linestyle='', markerfacecolor='lightblue',
           markeredgecolor='k', label='Without Resilience')
]
ax.legend(handles=legend_handles, title="Resilience")

# Colorbar TE_mean
sm = plt.cm.ScalarMappable(norm=norm, cmap=plt.cm.coolwarm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label("TE_mean (bit)")

plt.tight_layout()
plt.savefig(_prefix_output("./tempPlots/delta_syn_vs_delta_te_second_resilience.png"), dpi=300)
print("Saved: ./tempPlots/delta_syn_vs_delta_te_second_resilience.png")

plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colormaps
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.cm import ScalarMappable
from pathlib import Path

def norm_res(val):
    s = str(val).strip()
    if s in ["15", "15.0", "With Resilience"]:
        return "With Resilience"
    if s in ["2", "2.0", "Without Resilience"]:
        return "Without Resilience"
    return s

keys = ["SZ Factor (%)", "Resilience"]

df_str = df_all_str.copy()
df_str["Resilience"] = df_str["Resilience"].apply(norm_res)
df_str["SZ Factor (%)"] = pd.to_numeric(df_str["SZ Factor (%)"]).round(2)
df_str["total_cells"] = pd.to_numeric(df_str["total_cells"], errors="coerce")
df_str["synapses"] = pd.to_numeric(df_str["synapses"], errors="coerce")

df_te_norm = df_te.copy()
df_te_norm["Resilience"] = df_te_norm["Resilience"].apply(norm_res)
df_te_norm["SZ Factor (%)"] = pd.to_numeric(df_te_norm["SZ Factor (%)"]).round(2)
df_te_norm["te_mean"] = pd.to_numeric(df_te_norm["te_mean"], errors="coerce")
df_te_norm["te_std"] = pd.to_numeric(df_te_norm["te_std"], errors="coerce")

df_mean_struct = (
    df_str.groupby(keys, as_index=False)
    .agg(
        total_cells=("total_cells", "mean"),
        synapses=("synapses", "mean")
    )
)

df_mean_te = (
    df_te_norm.groupby(keys, as_index=False)
    .agg(
        te_mean=("te_mean", "mean"),
        te_std=("te_std", "mean")
    )
)

df_plot = df_mean_struct.merge(
    df_mean_te,
    on=keys,
    how="inner",
    validate="one_to_one"
)

if df_plot.empty:
    raise ValueError("Nessuna combinazione SZ × Resilience comune.")

baseline = (
    df_plot[df_plot["SZ Factor (%)"] == 0]
    .set_index("Resilience")[["total_cells", "synapses", "te_mean"]]
    .rename(columns={
        "total_cells": "cells_0",
        "synapses": "synapses_0",
        "te_mean": "te_mean_0"
    })
)

df_plot = df_plot.join(
    baseline,
    on="Resilience",
    validate="many_to_one"
)

baseline_cols = ["cells_0", "synapses_0", "te_mean_0"]

if df_plot[baseline_cols].isna().any().any():
    raise ValueError("Baseline SZ=0 mancante per almeno una condizione.")

if (df_plot[baseline_cols] == 0).any().any():
    raise ValueError("Baseline SZ=0 uguale a zero: Δ% non calcolabile.")

df_plot["cells_delta_pct"] = 100 * (
    df_plot["total_cells"] / df_plot["cells_0"] - 1
)

df_plot["syn_delta_pct"] = 100 * (
    df_plot["synapses"] / df_plot["synapses_0"] - 1
)

df_plot["te_delta_pct"] = 100 * (
    df_plot["te_mean"] / df_plot["te_mean_0"] - 1
)

df_plot["te_std_pct"] = 100 * (
    df_plot["te_std"] / df_plot["te_mean_0"].abs()
)

df_plot["te_second"] = np.nan

for res in ["With Resilience", "Without Resilience"]:
    sub = (
        df_plot[df_plot["Resilience"] == res]
        .dropna(subset=["SZ Factor (%)", "te_mean"])
        .sort_values("SZ Factor (%)")
    )

    x = sub["SZ Factor (%)"].to_numpy(dtype=float)
    y = sub["te_mean"].to_numpy(dtype=float)

    if len(x) < 3:
        print(f"{res}: insufficient points for second derivative.")
        continue

    if np.any(np.diff(x) <= 0):
        raise ValueError(f"Valori SZ duplicati o non crescenti: {res}")

    d1 = np.gradient(y, x, edge_order=2)
    d2 = np.gradient(d1, x, edge_order=2)

    df_plot.loc[sub.index, "te_second"] = d2

df_plot = df_plot.replace([np.inf, -np.inf], np.nan)
df_plot = df_plot.dropna(
    subset=["cells_delta_pct", "syn_delta_pct", "te_second"]
).copy()

if df_plot.empty:
    raise ValueError("Nessun dato valido per il plot.")

print("\n=== BASELINE CHECK ===")
print(df_plot.loc[
    df_plot["SZ Factor (%)"] == 0,
    ["Resilience", "cells_delta_pct", "syn_delta_pct", "te_delta_pct"]
])

print("\n=== DATAFRAME COMPLETO ===")
print(
    df_plot[
        [
            "SZ Factor (%)",
            "Resilience",
            "cells_delta_pct",
            "syn_delta_pct",
            "te_mean",
            "te_delta_pct",
            "te_std_pct",
            "te_second"
        ]
    ].sort_values(keys).to_string(index=False)
)

cmaps = {
    "With Resilience": LinearSegmentedColormap.from_list(
        "Reds_trimmed",
        colormaps["Reds"](np.linspace(0.2, 1, 256))
    ),
    "Without Resilience": LinearSegmentedColormap.from_list(
        "Blues_trimmed",
        colormaps["Blues"](np.linspace(0.2, 1, 256))
    )
}

markers = {
    "With Resilience": "o",
    "Without Resilience": "s"
}

sec_min = df_plot["te_second"].min()
sec_max = df_plot["te_second"].max()

if sec_min == sec_max:
    sec_min -= 0.5
    sec_max += 0.5

norm = Normalize(vmin=sec_min, vmax=sec_max)

std_min = df_plot["te_std_pct"].min()
std_max = df_plot["te_std_pct"].max()

if pd.notna(std_min) and pd.notna(std_max) and std_max > std_min:
    df_plot["marker_size"] = 80 + 160 * (
        (df_plot["te_std_pct"] - std_min) / (std_max - std_min)
    )
    df_plot["marker_size"] = df_plot["marker_size"].fillna(120)
else:
    df_plot["marker_size"] = 120

fig, ax = plt.subplots(figsize=(8, 6))

for res in ["With Resilience", "Without Resilience"]:
    sub = df_plot[df_plot["Resilience"] == res]

    if sub.empty:
        continue

    ax.scatter(
        sub["cells_delta_pct"],
        sub["syn_delta_pct"],
        c=sub["te_second"],
        cmap=cmaps[res],
        norm=norm,
        s=sub["marker_size"],
        marker=markers[res],
        edgecolor="black",
        linewidth=0.6,
        alpha=0.9,
        zorder=2
    )

    idx_max = sub["te_second"].idxmax()
    row = df_plot.loc[idx_max]

    ax.scatter(
        row["cells_delta_pct"],
        row["syn_delta_pct"],
        s=row["marker_size"] + 120,
        marker=markers[res],
        facecolors="none",
        edgecolors="black",
        linewidth=2.5,
        zorder=5
    )

    print(
        f"{res}: "
        f"max d²TE/dSZ²={row['te_second']:.6f}, "
        f"SZ={row['SZ Factor (%)']:.2f}%, "
        f"ΔCells={row['cells_delta_pct']:.2f}%, "
        f"ΔSynapses={row['syn_delta_pct']:.2f}%"
    )

ax.axhline(0, color="gray", linestyle=":", linewidth=0.8, alpha=0.6)
ax.axvline(0, color="gray", linestyle=":", linewidth=0.8, alpha=0.6)

ax.set_xlabel("Δ Total cells (%) vs SZ=0")
ax.set_ylabel("Δ Synapses (%) vs SZ=0")
ax.set_title("Structural changes and TE curvature")
ax.grid(alpha=0.15)

legend_handles = [
    Line2D(
        [0], [0], marker="o", linestyle="",
        markerfacecolor="firebrick",
        markeredgecolor="black",
        markersize=9,
        label="With Resilience"
    ),
    Line2D(
        [0], [0], marker="s", linestyle="",
        markerfacecolor="royalblue",
        markeredgecolor="black",
        markersize=9,
        label="Without Resilience"
    ),
    Line2D(
        [0], [0], marker="o", linestyle="",
        markerfacecolor="none",
        markeredgecolor="black",
        markeredgewidth=2.5,
        markersize=12,
        label="Maximum TE curvature"
    )
]

ax.legend(
    handles=legend_handles,
    title="Resilience",
    loc="best"
)

fig.subplots_adjust(
    left=0.14,
    right=0.77,
    top=0.91,
    bottom=0.13
)

cax_red = fig.add_axes([0.80, 0.55, 0.025, 0.30])
cax_blue = fig.add_axes([0.80, 0.14, 0.025, 0.30])

sm_red = ScalarMappable(
    norm=norm,
    cmap=cmaps["With Resilience"]
)

sm_blue = ScalarMappable(
    norm=norm,
    cmap=cmaps["Without Resilience"]
)

fig.colorbar(
    sm_red,
    cax=cax_red
).set_label("d²TE/dSZ² - With")

fig.colorbar(
    sm_blue,
    cax=cax_blue
).set_label("d²TE/dSZ² - Without")

Path("./tempPlots").mkdir(parents=True, exist_ok=True)

output = "./tempPlots/delta_cells_vs_delta_synapses_colored_by_TE_second.png"

fig.savefig(
    _prefix_output(output),
    dpi=300,
    bbox_inches="tight"
)

print(f"\nSaved: {output}")

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colormaps
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.cm import ScalarMappable
from pathlib import Path

def norm_res(val):
    s = str(val).strip()
    if s in ["15", "15.0", "With Resilience"]:
        return "With Resilience"
    if s in ["2", "2.0", "Without Resilience"]:
        return "Without Resilience"
    return s

keys = ["SZ Factor (%)", "Resilience"]
conditions = ["With Resilience", "Without Resilience"]

df_str = df_all_str.copy()
df_str["Resilience"] = df_str["Resilience"].apply(norm_res)
df_str["SZ Factor (%)"] = pd.to_numeric(df_str["SZ Factor (%)"]).round(2)
df_str["total_cells"] = pd.to_numeric(df_str["total_cells"], errors="coerce")
df_str["synapses"] = pd.to_numeric(df_str["synapses"], errors="coerce")

df_te_norm = df_te.copy()
df_te_norm["Resilience"] = df_te_norm["Resilience"].apply(norm_res)
df_te_norm["SZ Factor (%)"] = pd.to_numeric(df_te_norm["SZ Factor (%)"]).round(2)
df_te_norm["te_mean"] = pd.to_numeric(df_te_norm["te_mean"], errors="coerce")
df_te_norm["te_std"] = pd.to_numeric(df_te_norm["te_std"], errors="coerce")

df_mean_struct = (
    df_str.groupby(keys, as_index=False)
    .agg(
        total_cells=("total_cells", "mean"),
        synapses=("synapses", "mean")
    )
)

df_mean_te = (
    df_te_norm.groupby(keys, as_index=False)
    .agg(
        te_mean=("te_mean", "mean"),
        te_std=("te_std", "mean")
    )
)

df_plot = df_mean_struct.merge(
    df_mean_te,
    on=keys,
    how="inner",
    validate="one_to_one"
)

if df_plot.empty:
    raise ValueError("Nessuna combinazione SZ × Resilience comune.")

df_plot["te_second"] = np.nan

for res in conditions:
    sub = (
        df_plot[df_plot["Resilience"] == res]
        .dropna(subset=["SZ Factor (%)", "te_mean"])
        .sort_values("SZ Factor (%)")
    )

    x = sub["SZ Factor (%)"].to_numpy(dtype=float)
    y = sub["te_mean"].to_numpy(dtype=float)

    if len(x) < 3:
        raise ValueError(f"Almeno tre livelli SZ necessari per {res}.")

    if np.any(np.diff(x) <= 0):
        raise ValueError(f"Valori SZ duplicati per {res}.")

    d1 = np.gradient(y, x, edge_order=2)
    d2 = np.gradient(d1, x, edge_order=2)

    df_plot.loc[sub.index, "te_second"] = d2

baseline = (
    df_plot[df_plot["SZ Factor (%)"] == 0]
    .set_index("Resilience")[
        ["total_cells", "synapses", "te_mean", "te_second"]
    ]
    .rename(columns={
        "total_cells": "cells_0",
        "synapses": "synapses_0",
        "te_mean": "te_mean_0",
        "te_second": "te_second_0"
    })
)

df_plot = df_plot.join(
    baseline,
    on="Resilience",
    validate="many_to_one"
)

baseline_cols = ["cells_0", "synapses_0", "te_mean_0", "te_second_0"]

if df_plot[baseline_cols].isna().any().any():
    raise ValueError("Baseline SZ=0 mancante per almeno una condizione.")

if (df_plot[["cells_0", "synapses_0", "te_mean_0"]] == 0).any().any():
    raise ValueError("Baseline nulla: impossibile calcolare Δ%.")

df_plot["cells_delta_pct"] = 100 * (
    df_plot["total_cells"] / df_plot["cells_0"] - 1
)

df_plot["syn_delta_pct"] = 100 * (
    df_plot["synapses"] / df_plot["synapses_0"] - 1
)

df_plot["te_delta_pct"] = 100 * (
    df_plot["te_mean"] / df_plot["te_mean_0"] - 1
)

df_plot["te_std_pct"] = 100 * (
    df_plot["te_std"] / df_plot["te_mean_0"].abs()
)

eps = 1e-9

df_plot["d2te_delta_pct"] = 100 * (
    (df_plot["te_second"] - df_plot["te_second_0"]) /
    (df_plot["te_second_0"].abs() + eps)
)

df_plot = df_plot.replace([np.inf, -np.inf], np.nan)

df_plot = df_plot.dropna(
    subset=[
        "cells_delta_pct",
        "syn_delta_pct",
        "te_delta_pct",
        "te_second",
        "d2te_delta_pct"
    ]
).copy()

if df_plot.empty:
    raise ValueError("Nessun dato valido per il plot.")

print("\n=== BASELINE SZ=0 ===")
print(
    df_plot.loc[
        df_plot["SZ Factor (%)"] == 0,
        [
            "Resilience",
            "cells_delta_pct",
            "syn_delta_pct",
            "te_delta_pct",
            "d2te_delta_pct"
        ]
    ]
)

print("\n=== DATAFRAME COMPLETO ===")
print(df_plot.sort_values(keys).to_string(index=False))

cmaps = {
    "With Resilience": LinearSegmentedColormap.from_list(
        "Reds_trimmed",
        colormaps["Reds"](np.linspace(0.2, 1, 256))
    ),
    "Without Resilience": LinearSegmentedColormap.from_list(
        "Blues_trimmed",
        colormaps["Blues"](np.linspace(0.2, 1, 256))
    )
}

markers = {
    "With Resilience": "o",
    "Without Resilience": "s"
}

line_colors = {
    "With Resilience": "darkred",
    "Without Resilience": "darkblue"
}

def make_norm(values):
    vmin = values.min()
    vmax = values.max()
    if vmin == vmax:
        vmin -= 0.5
        vmax += 0.5
    return Normalize(vmin=vmin, vmax=vmax)

norm_L = make_norm(df_plot["te_delta_pct"])
norm_R = make_norm(df_plot["te_mean"])

std_min = df_plot["te_std_pct"].min()
std_max = df_plot["te_std_pct"].max()

if pd.notna(std_min) and pd.notna(std_max) and std_max > std_min:
    df_plot["marker_size"] = 80 + 160 * (
        (df_plot["te_std_pct"] - std_min) / (std_max - std_min)
    )
    df_plot["marker_size"] = df_plot["marker_size"].fillna(120)
else:
    df_plot["marker_size"] = 120

fig, (axL, axR) = plt.subplots(1, 2, figsize=(15, 6.5))

for res in conditions:
    sub = df_plot[df_plot["Resilience"] == res].sort_values("SZ Factor (%)")

    if sub.empty:
        continue

    axL.scatter(
        sub["cells_delta_pct"],
        sub["syn_delta_pct"],
        c=sub["te_delta_pct"],
        cmap=cmaps[res],
        norm=norm_L,
        s=sub["marker_size"],
        marker=markers[res],
        edgecolor="black",
        linewidth=0.6,
        alpha=0.9,
        zorder=2
    )

    idx_max_L = sub["te_delta_pct"].idxmax()
    row_L = df_plot.loc[idx_max_L]

    axL.scatter(
        row_L["cells_delta_pct"],
        row_L["syn_delta_pct"],
        s=row_L["marker_size"] + 120,
        marker=markers[res],
        facecolors="none",
        edgecolors="black",
        linewidth=2.5,
        zorder=5
    )

    axR.plot(
        sub["syn_delta_pct"],
        sub["d2te_delta_pct"],
        color=line_colors[res],
        linewidth=1.8,
        alpha=0.65,
        zorder=1
    )

    axR.scatter(
        sub["syn_delta_pct"],
        sub["d2te_delta_pct"],
        c=sub["te_mean"],
        cmap=cmaps[res],
        norm=norm_R,
        s=100,
        marker=markers[res],
        edgecolor="black",
        linewidth=0.7,
        zorder=2
    )

    idx_max_R = sub["te_mean"].idxmax()
    row_R = df_plot.loc[idx_max_R]

    axR.scatter(
        row_R["syn_delta_pct"],
        row_R["d2te_delta_pct"],
        s=220,
        marker=markers[res],
        facecolors="none",
        edgecolors="black",
        linewidth=2.5,
        zorder=5
    )

    print(
        f"{res}: "
        f"max ΔTE={row_L['te_delta_pct']:.2f}% "
        f"at SZ={row_L['SZ Factor (%)']:.2f}%; "
        f"max TE={row_R['te_mean']:.6f} bit "
        f"at SZ={row_R['SZ Factor (%)']:.2f}%"
    )

axL.axhline(0, color="gray", linestyle=":", linewidth=0.8, alpha=0.6)
axL.axvline(0, color="gray", linestyle=":", linewidth=0.8, alpha=0.6)
axR.axhline(0, color="gray", linestyle=":", linewidth=0.8, alpha=0.6)
axR.axvline(0, color="gray", linestyle=":", linewidth=0.8, alpha=0.6)

axL.set_xlabel("Δ Total cells (%) vs SZ=0")
axL.set_ylabel("Δ Synapses (%) vs SZ=0")
axL.set_title("A. Structural changes and relative TE")
axL.grid(alpha=0.15)

axR.set_xlabel("Δ Synapses (%) vs SZ=0")
axR.set_ylabel("Δ TE second derivative (%) vs SZ=0")
axR.set_title("B. Synaptic changes and TE curvature")
axR.grid(alpha=0.15)

legend_handles = [
    Line2D(
        [0], [0], marker="o", linestyle="",
        markerfacecolor="firebrick",
        markeredgecolor="black",
        markersize=9,
        label="With Resilience"
    ),
    Line2D(
        [0], [0], marker="s", linestyle="",
        markerfacecolor="royalblue",
        markeredgecolor="black",
        markersize=9,
        label="Without Resilience"
    ),
    Line2D(
        [0], [0], marker="o", linestyle="",
        markerfacecolor="none",
        markeredgecolor="black",
        markeredgewidth=2.5,
        markersize=12,
        label="Maximum TE"
    )
]

axL.legend(handles=legend_handles, loc="best", fontsize=9)
axR.legend(handles=legend_handles, loc="best", fontsize=9)

fig.subplots_adjust(
    left=0.08,
    right=0.80,
    top=0.90,
    bottom=0.15,
    wspace=0.35
)

cbar_positions = [
    [0.83, 0.56, 0.012, 0.28],
    [0.87, 0.56, 0.012, 0.28],
    [0.83, 0.16, 0.012, 0.28],
    [0.87, 0.16, 0.012, 0.28]
]

for pos, res, norm, label in [
    (cbar_positions[0], "With Resilience", norm_L, "ΔTE (%)"),
    (cbar_positions[1], "Without Resilience", norm_L, "ΔTE (%)"),
    (cbar_positions[2], "With Resilience", norm_R, "TE (bit)"),
    (cbar_positions[3], "Without Resilience", norm_R, "TE (bit)")
]:
    cax = fig.add_axes(pos)
    sm = ScalarMappable(norm=norm, cmap=cmaps[res])
    cbar = fig.colorbar(sm, cax=cax)
    cbar.ax.tick_params(labelsize=7)
    cbar.set_label(label, fontsize=8)

Path("./tempPlots").mkdir(parents=True, exist_ok=True)

output = "./tempPlots/two_panel_deltas_cells_syn_d2TE.png"

fig.savefig(_prefix_output(output), dpi=300, bbox_inches="tight")

print(f"\nSaved: {output}")

plt.show()